# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.

Khôi phục checkpoint PubMed 6.000/110.000: giải nén ZIP đầy đủ
`ids_5979207_7061937_synced_vi_en_zh_efetch.zip`, đặt `RESUME_RUN_PATH` tới thư mục chứa manifest,
`ID_FROM=5979207`, `ID_TO=7061937`, `POSTPROCESS_ONLY=False`, `PUBMED_USE_EFETCH=True`, `MAX_NEW=None`.
Giữ `PUBMED_UPGRADE_EFETCH_SNAPSHOT=True`: chỉ nâng cấp transport EFetch; XML parser/schema, core/runner/probe
và dữ liệu cũ giữ nguyên. Log sẽ ghi completed/pending thực tế. Không dùng ZIP diagnostics để resume.
Checkpoint full ZIP được tạo mỗi 10.000 URL mới hoặc 15 phút, và cuối lượt. Nếu có Secret HF_TOKEN quyền ghi,
upload ZIP/checksum vào `checkpoints/<run>/latest.zip` và `latest.json` trong cùng commit. Đây là backup;
run còn pending không upload corpus như hoàn tất. Có thể tắt upload, ZIP local vẫn được tạo.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
REQUESTS_TRUST_ENV = False  # False như lượt thử Kaggle; True nếu môi trường bắt buộc proxy
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
VOV_USE_CURL = True  # vov.vn và các subdomain dùng curl/Chrome
LONGCHAU_USE_CURL = True
LONGCHAU_WORKERS = 1
LONGCHAU_DELAY = 5.0
LONGCHAU_ATTEMPTS = 3
LONGCHAU_CONNECT_TIMEOUT = 10.0
LONGCHAU_READ_TIMEOUT = 45.0
LONGCHAU_REQUEST_BUDGET = 180.0
LONGCHAU_RETRY_FAILED_ON_RESUME = True

CNKANG_REQUEST_GAP = 1.5  # 1 worker, robots luôn bật, không chạy Selenium
CNKANG_ATTEMPTS = 2
CNKANG_CONNECT_TIMEOUT = 10.0
CNKANG_READ_TIMEOUT = 30.0
CNKANG_USER_AGENT = 'ViBioMIR-CnkangCheck/1.0'

BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 5
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 90.0
PUBMED_EFETCH_BUDGET = 300.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

PUBMED_UPGRADE_EFETCH_SNAPSHOT = True  # Upgrade transport compatible with existing XML; keep saved runner/core/data
PUBMED_MIN_BATCH_SIZE = 1
PUBMED_MAX_SPLIT_REQUESTS = 50
PUBMED_FALLBACK_BUDGET = 900.0
PUBMED_PENDING_RETRY_PASSES = 1
PUBMED_MAX_FAILED_BATCHES = 3
CHECKPOINT_ENABLED = True
CHECKPOINT_UPLOAD_ENABLED = True
CHECKPOINT_EVERY_URLS = 10000
CHECKPOINT_EVERY_SECONDS = 900.0
CHECKPOINT_HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
CHECKPOINT_HF_SECRET_NAME = 'HF_TOKEN'

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(REQUESTS_TRUST_ENV) is not bool:
    raise ValueError('REQUESTS_TRUST_ENV phải là bool.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
previous_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
default_tag = previous_tag + ('_longchau_v2' if LONGCHAU_USE_CURL else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = previous_tag if (WORKDIR / 'runs' / previous_tag).exists() else (source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced')
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'huggingface_hub', 'curl_cffi']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    pinned = {line.split('==',1)[0].strip().replace('_','-').lower() for line in requirements_path.read_text(encoding='utf-8').splitlines() if line.strip() and not line.startswith('#')}
    install_args = ['-r', str(requirements_path)] + [p for p in package_names if p.replace('_','-').lower() not in pinned]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import importlib.util
import hashlib
import zlib

EMBEDDED_FILES = {'ask120_snapshot.py': 'c-pO7dz0M8asPim1$0UUfVjK&pyHA*I?=R5N32k!OdeIKT5cdLcJ>xJECAxcyEE^qNL5%i70Pxcie;O0EIVSt*k!U(v15v)&F^Ay?~Z@*E2O(;1{l0{S)y1a9d<C&)6<XNboUGv78c$*^NtqUQOqLE_J>+9fTv6Lz+;`a*WWp%efLdm6hzwc%90&#Y8O)$CvM=whz%BZTdnmm)0|)u+D-z0qcDxNO~%4l3w&=!d;k4}Z71))-_^ABam4JQ7AH~aBx%IjS{SiTV!K`^Vw2#K?P=Jzg*?EVPLTQui((jH?`W>CB`{IQqRz>;G-u45%`kBNM2i?qI@AU%c884mMD~`3jCWfL3k$7L6ihVB8l^a;Wohmt459=U>IaF93yE92j}wGD^6d~7fWKSf_t=ido;#2~BgTiMkq50^lARn1ZE@(iiJ<CjPdxc*oJ_n{tM%TAv*%8pIjyZ}I+tmO3%_$|MQ@$>hxN0^-&{X)Hh*nhw5?$<vGDcVXU?skgIAZuyDmsxZ)^J2)>c<(*=v3G{E2hxFpKr>@%6W1K*aDx=z7eE^hK$}#okiqoBbmfy1<%lw$_iIJXP%5(%@gDUx;6M^6>VP`ycK9^xpK|7rnn+xwY`+l^b7P`C;EU_U`M6n|SPv-ar23$}gb#<1erL=z`zByxd+{TG}&vc`I~YJ$(GLC-;A29((%f-KW3)>A}zcY5MTN;oZBsn5p;d(T(<(S3Z39=;q&Ei2JWh@88(Jbra^?|M1TKt?SeKcc&lyH;^g!JGlJ|=rO%|d-~vy<}rAC^(A=Q#aDl&x6Ib5<KI4UiX=x9>BGQDg2=M{c#B2)s-{nGJUsa1m(!d7e(<|b^fvDlf`KqUu||RC1zTx|{q{fnkHah14j=ys79jfxVrn-|?(kQ)r`N6(iKhOhA8h%>-ar1~!NcD@dGPuEAFe{bz1I1Y);q_~{ykU<u*|ifPp|#olZQV!_@9SQ9zEjHL(L!Dyg$A6zxzM={=siQ%YPq!d~f=%SMj%4*R4AT4<1jieSG-)FQ)%;=kVUI@!jFQKR)@><AYD`O#giM$>U$`U;pis`&Z!qlGXi>{_XJIb?W(t&z@en%4t3Q<UjX6{gmeV@Qdm9AMXG1_Q9=xo__vc)0=;Wk!I`ciFeMtcj7H;{rKN;TaBH_4HFy@Cp#X)2X_?N6M6z{Arl^?NfP+<&JR=S?FY)pOB-O;lJuLWj-NYsLeZxa3|Sb&E?PBqPl7Z~iL3=qF$lJ;LBfeoHt^5I2uJXz4u9l?251xZaG1E9J^9-g?E0>=DW9<qpwlEIowZs+Hqv}pj_tYMV+MkmxytRD0s8ppSaCWq(4UUrcn$uCy)=MVT{pV`jc#a|f|3B$=b%$0BZj-NCEeTtP%_{@L51LE2(pt{Wx&Mr-c)BXc)xB8Xggtuz=l3fk`QOazp-x8ej)pfgE&FS$kl*#v&b-UAi);Vmj4LkRgCJ6QqP;%i8Ge;!28B9ItGy@+UdY&a%Gm|*>TKb!w0<;B@?s6pyuudJn7UC*p%|6#hMu%fOSrbhO{;0zDSSO0}CilDaQ{PT8=RYhC6Lg7i?Dm7RgS?)?hYHXAD~&?5hN-F|#9BUZgu?TwwyJx(5IC7B1-hV@B`7R;Pc&)Qglj>unr<l51ze61zxmFpnTI*jrux#?jYa?W?4Kf6tD{6gZ#VjVMS%V;RIglB_ciHaDC!d)XE1*6rAFU2>`oFngxcFyfB}=^WG@(6u|X6BfffJvk?gN|RCNH5|S%SblBk=#t*2{@?_}66%ukiQz{^1=W$0T+OVC`M{LxZ@|dxwY0&^vt(!BGwV-=y)jrhy1c@f*~CBF9eglX3s$Slkuj~(L`%*30MqImw5%dIRJ7}-ENi4bT|gT{jx?2bJH^q$L{Yg?BXSfWZR9$MZnm|&@jEuCB^TJX!@@)}P6fec8lH%vAZlyp0pEbqTSSlA%o#k_u#jstVxea{3|Wv$&oBXGgC5=1)*~P#+$o9JY-i$A4AmuV*^$o`((H|9t0F(aI7Vb$jMGqB{)QnYFl%OLsUmyN_EIJ!Ml=}m-hG$iI4(8Kj<LHkrR{=G<Rc9c!cnX$VUC~=LvQdUiXVZ56b}W-%<==Eqe7qKfh0iyGH4YTl{wDX^@eDGo*O3yb?&+lSPxn3Fn?(KNeqy<S~g=hc752}e1{oCp_|ws43LvK$A$$`qDCZW5`lvA<uui8q7XPB>w?r@Yo&d^zybaXFm>56<hG8)E}^6|>L>|nA^6~Q&GJ%+r(qO?+!PDDAqh^Z0wrwWmMb=nSK68k=hn!{1Z9#Xq$QwTH1~@ANrV&{tWmKfBgy9^Ue+-4y2K)Pgm{^-3Ah`;V+^4v$ZY7^Ua*le=?S>*p$pkkA}k^#)*yq+2?FaLI@7>*Hnk+s;HP#83<2zs2N+vH3L|644E)5yq|EVBSe}fD7BL`NJbyB$3HEGnM0PkP?B^2~jFK<i9nG+}in8+vqH0BJfXt!TNYs5ii^?GA>V5#3qmeg-U0j?c1O?7OEQZKl&cFFW9;JjDG+GHPJy(A&J9J2`AVsn$<b#`*kYSaBqTJk?ct>|4`+GZ<&$eVJmxoU7P>GuoJ4)g$HyIm*>N~pGZ_Se~5Gsnbz_Y=Z11&vbzi;MYnGbOd1g;`Evm84bs(1%FL3>`)B(uChEO-lq$7CNRcx+M!?<w^G^HudBR>^y+rp?*l4;4nlsM88;K-$NK*1!v#O|S!4Rx#MD(Jb_tqgVvfs$yv|D0vb;*JML4VS;8QJa5(m$`yb0v#lxc3R_aGv6XqMqG2&9my{6LYyAhhQ=+DuMYklonB|bA`LaxW?u;@u{K_*{DhjIptEGX`){nI^N(Uj|!;*`bfU+hqL@eLPq=1sfma?DoPRO)KE2%-m^g59PBhX%2%kxfU3WZBPe;!II&AnAll<>kZ``t*y-NiUBOb39RV{iom6N@T+211>fjW%k+x_$<_+2&1=);1=GhQ7E+o3PnKi?U$@F}iJS;mLysP;!}m{_%p@SIvx*?ZOC9kpaa+!%rsz771jcXPf~Nq>&3s^gPJb3|=BLu>>aR>a;N#M9(ZH&3umDQXl0m)`89hJEnmN4R11Oqgv)^l8O){8i(qV7A7#NRA{>fp%ex1O1^e@fT6~Mw$RQYxf6pjnei<$Wl#jXrT<f1#2zebBqSao;}*a6RnVMR6M~S|s3yF|vDaTBIhw}G5<OgA+1p+IrhGJeoyG3-i{JM_+ROc8yeqct-}(Wzb&kqLQR=9gcEwIftjh0^ei>D+n*B1k=+djhy-wxL3cgvXdvjF0k-iDcMCdI%?SLXmQ}P1E=o7rJh9M8z#;0;o{ruTe8t$_}3MFc7gZU8NIUQ2W2<#C9vo2P(3xRr5ou@wQI2|BqF>ziAe%&3ufn{K1LD~Ztnwkmm?8s75u7-?F3`hL1nymI)?r?2kh(`GK@`AKLu_R?=zLG|Ag^lY)JtA?@Ar=?M%Yvm$wsRITiQf$T@<NJ(^R>((`n4_x3ZLZQUn@}@s=@{;rhprYssITq(KQH1Ri;`i${b-$O|pvO0+K|dBt%K3#p4$WSz!5%;(akG5Pd0Kvu@5FzD1D4zkreI>)Mn?*i0$-sKlp0K!hJM$i)K8HiaLUK4Jcc!AD;d-=bk@C#~Tpw@m6Ux5!F7D(5MLIWwX?5l*)n6S(JDu;*6Ro5fq|f`t}5FN>dw=~B(Q=P$0F_2w0eXqrnq4${bBIXf0YyCt^SIGq4=2A)x=CS;`q44Ta1PFq->sm?M1JBEO21CKO;{1Pu%O~PcS{*|31wu3>KDlPsh_3>17!v_~AzDAxiiJ<PDZvm)j>X}1CO{t@d6S?dg1wcJa!84DYF%`*I!e<#$<eaxAbP_GxLa8uELz#3b9>LQ!1f**EmUSWtXFYrL#Ep)g(7O1`qUBIyO=^_aFY1(gWR1c}WzL+D8igO>daF|xjTRP`aav0^3$?O$bIO1sw~`wiWwLb}T=&IcIPt#bJuVKHc@$tP*u`^|n)STM60!n}vX5kw;hu_Gs^YN93l;#LyKDoK@{_bIs^XV2@RSx(9<~jq(P&%3Lrm?6w%o44K#@8_@7$$TjW=xj1)~rfb}f0nKv>KwUu(8FJnbY>TtZ5LC!+`FvJmgbFtpd-D=pNTfqF`>dN9d0(4tS5iba9POSxDQ;r>;u*Tm$Vm>^Nlwc4^r9=>nM{i5u)3e1W$0O?kWL-<ZoS*?h=;<mgT&1@f!Sp<42m1(L#NOH0YLF&N8U|p}(U@CHX?P}S@<f>EX@)~-n2&W=BU>YW7qOo?-LLk*OH(LTwU2f`LGzmGsRAHrZQ_GSN>~pGtPuvsM-0#|9h(Xz99cM+yfIn#LATjurDNUDR#4h4-uK?s2a3Rqir_`@8Lb7v8o9W_Jfss;mv|f&lLN1K02(?bwHA0D{F!)ctIyk?eNjk^^B+zsr%+}Hv&(rws4gsWcy-kBFCwNv&J0l#N*dqpz0ssxi*>G0b;s;CN*!Bx(F|oIerS`1gu2Ulvkt?AHF+$)a2^2C1;7rW%EYbH{S+KzsmC@>PJLnfj(`=Y-&DM55EB{y_ZW%-rT^p>4-!yvm=;s%F9d-kmboH|=jyrF!v13c(m?TZ7Ci!Ifxk^P8s|DC?kV(-R<8(A~w_%pif_7Q<<U3n`Z=vjzvV+*o`V^@aghiJF<|A0ur3E8LkdtSResH-w)!ws5*Ngmk_JDc7IlrpbQ`zW)KqWy<2N@T4iCmb{FE}me%ax&8h*{ZS=8w!ErC+ENLLcbGON2;7W@(h8Ddq%{#s_%|Bd1~EUClXka?`0LlNXbzjEsBLjQX-TK-yqd7zG1XU+pYan9;~(p;_D2ur^fqiS;kp01=kiTYYtDX-TBtW5`#S5AzLK0%xzZnNk#<mq+uUPq!UNXW{IY`EaDW`#~CGscsTXAcEQjNP$0C8;RJ(6i#x7yL1f}OL04ZMcN<=wgB389k|co0-R28IINNR2zJ9=U73iEjdySyVd6NQ*#0i>sC{5xviZF`e#Xkl?LPg@jl+9a4?erPJH2~-dhHJXy?c1~F?jy;N1s0Z*^iZ2E|;;ma3pRVr6JX~nMKu9goDd_Un>A?GK`zEGmJ*V2V;V+RlhBB2wF;@&TFS$V!(hl1eK=r6{7?&?E-c*FS!Tt8^3!@>&k!O8GU(veqzAOEEt++y-n>6ZJC?dMJKxnNg)^Hk|Bl&dcN6Ncu&Q`JK2hsbTz#cg%Oc9!swdZQyD4>-{58HekAe-waHWl)w(8(DS7dX5kq%0Vo(QiaDB#UaeJVQ0^-62-<v87!3JF#U{+-!!Cj=gVqwMlOY<lZ4)7ZnRwx6q=Ac0#RSh0sWVKNlQCqOd8LzMBDC5gf4vZ-`6=o|2yF5&zd6ckCHI=+5Q8BV=!9p3B1^wc33y=OODO9onWkBvd=!wFGgzCzkrD`vAfrV%jjKcy(gtB_PiF{N%f`ckov+=fL){cYvoaCOUJ=rlYxQXJcaA2kOP3Nu48fEt4*h4+XqVW=J;I`!^oZ46u7Xnm&!@_`MGH^Fixan+(^ILVAVtG8D3ExS0lzkvOe_dN%`f77aB6{giE_b-Obp1fJj+{_k_3Ndz8i&TamK8&qzZ}IPVQF;Cmrq`UDOs%nH^%b&I>zR*P=>cE8}-gRmvB8KWFOz;^vjMumvU)<%BkZ1&%s~~isZy<(n=6;Gk(sUFPn$7RU_z&POlCJ`tw&J2U&3%hYWAsN}8o3Q!LC~fH;0M)Fouhr7lQRNLNG4W>O4~e$i{QLGsLs0#1mpW5g+b9wCZ4^43BPR=fdFvXoc#3l`sMIO)L>v`UWm7qw^pBP%{<ZVeNyEVX8w9f^e&u3#u}5QoK8O(wGf*m;pjMle*y%)Yy-4Lxie)o?ij(?M~)*^+1FxUb1$b7gZX7z6YvHng)KB7<(a*DtoOO#aoNLtfAl!PKk(OpL&sC8I7?BUCa=BQwGzb@s+6Nk!adcJZU264jaEPx)4rTvVm(QMrCqEtja=meTx2vgLBXY^nvAtVB&;A?Oi6k?q5)S@ET<)ogW0O8r@-B(g8`6#kUcx#%Eyr)G>u=Y@OWV{<1f9@{H_^9|@LzO3x>219n}lKW&|6PRXq%(jOvd~pg%c9m|#F1c*W5@<l8qNkj11?bS4@?TYARba*PYNfF{kblN$L2ey$HHa@gt4S03cFa76W<v)R`pe=fjJBwC<wX>@g8~b9S==qj-rzZQH@k>Ydwx<b!FSag3*G!+T{Ga@-wu2~3$9M5o5EW4ERf}woa}Vy7Opt)uWn~tbvnyBVy*N&F6d{7Ih&{`Oz<e5KGwx&4<sr-K*pjilxnj}PT*RwG0nMTB}#}c^XO35C~p7~`3soeE&+Ta{LaDKb{(j;oA7Za7FL0;T?cOM8f0q3mu5hjfglaI(XJpw0T^b0Lc0+H+D%SerKb9DyUFXG%b(lI#Q_-z@=H5pM1<e`2>@K!6+gbDgGg$jjEeh#(wJp>Ioap<1PJ7@sC?5d$3g)IE3s`^n9OO4+7i`$fUBOOg?>XUB&BSlh!q@?UyuL{;dPCw{8STAdt<!ZPz35%spKJhi9}7cqh@$bg#rgxnrntOmMbXWuYXxX=AvWCvKkKL&lyhsKVq;@G5',
 'audit_snapshot.py': 'c-qBW-H#l{aeo(}|DknCFtgU)97)MSy!W9{Bo#9jC4!^`0ms8=Znt-Le6us7`QY98ct8R_<S9S|2QZAlfMg>O9K*1K1PDSt<lzMWAD6$xzpCz@>6x9`+mqykf^K(vx~saY>#M7(XJcdI!|5~%V=2P8kh3f-PPfJhzQ%=^guyIM@**7PB8VsWUzBka9Lgw?6OqYhp*$A(T*^XBlS~9+oCU{`$jaE;*w}dWm2i<JSs{XKmIhfaUwvhoB?~c5FnnBuNt_Ga<4#fnV^)or{Ul7QAM+reN8w@pyI4db$obQk`8Y^r^-O)Bw^tTnR5kxNPhyQd4GMy0Fh9i4s#}IdCX4!K4L3i{tM6qTj+2R;1VvD_oh%|x7MYaZy+|%VyC?Wglf3st<DHiAxJZ(yT62_@{x}%V4Oz-8qIol=Odqu6X%rT(zM?xtKqM?WR1-)~gDgvqY4~R)_5-NX&uG}IuT10=O9>_}EqyS0^%a4CnJmgIX2N(;5=?S_>QO(xkSB#JV=OX^XS+^WOgI0^84PM@<7ANr<HDtmEr5=2ydNiF>{?Xv;v`!HQTP+-Iv>1u#~BJ9=<(8A8*n}kTpG~=syV>J<6GZ<TlMhf@?;XuWDW?{I)h`M`GWSsJxnI2Lt$28fKDV&OzaAJ?x~;EdWXWvWEury=?ql=a4O=YFvGmO$imb$z^qw*ki-(B<svOkeVWK58JENhj_O)ur^|zAvL0<MfsJRb#xfW(IMrRAjAdE~m-{L%dq`Z8-NTk=L-BDCm6E^r&#H`>v4X=$w&zCw{dL8S?jktJu~oZo4Qi4?(kCF0Y_Jef4@4wcWg;%pOy;>u5~cbnZaO?IWX`*6%}VVIZ8v=zTseufJY2|$5AOFfpvu6nJkan-f(_b*u-T;4ko6W)7q19r;<a7jJY;aUGibI*K9+@J(=7-gfHh5B>iq!xVu&EZoEhTYd$JhM#U~Hm_nc~VJU2|`>|C{wUR15nuRIA0Rg{ba(u3&l00U{^J7K|14$qkf@~n5;vv!1xFi+>*Bssd<WHkDhE<)mP4PhF_6PD@<;~T=+didb}T?nINW^b*)%Vhz&2W3>a1ZpVu4+dV4rZS$ms!wx6^|)FRgK`oUe!Xvc)KJ*_1}pKg%);rZ$dfVyX^XIkz^?L{%uYpch@FI<5xFuKY=hYRK#9f^Nt3~<qf?PgMKPDk{K!l&t7pmvS;R2fR8M8!L4$*3vF1}^?QB_6uZcI4h3n{r5F}?nJVf7Cam{o?vkg&s>!U~Si$$0(0&vVooS6v#*unXR#>fvPF;5@2^Hi$sST)Ze4gDOPROXm1@eYvZ5DK*bYu*@c@K&6vAQ8V{aI5vR5{ex9tpPk3lY){Bz^jR(r^Cdl1?wsb*>(nuduH#7uR%H^@gytd`L(DsDZv+z49Sl=XcWoV)pQz&9dYZ=M3!nE-w|Qzb*VMANYhvqc=~30SvGy2k)Vc>*k3C^0=BjY@CY~s@J=AG23X&Mr1GL53wnP0HxAU3u~v|Hq&u;8bC(q8fL3U_tWu#(Fb#dGXA`LYKxwe+&IZuhxXgI%5s<d?V;FpZ)<~L=pHL4o4en01Lc3V?a4&@S%nE+MQ?UT+y&^Y2TMTU_;>6G4#Sy?LSR76Q_R2=se=jc&-OPEKzrogFD6aP9Y*$HvRkLHZs+t<=g;&PbXRsC%@qjH@+bg9ZlT(?I+zgH7Af>v`01ilX>Rr|5g&(>xrgB@iU_Z%j$7HhYZ3&{twK}!oc%TH?M~3ww3{utwu)#tMvDI}@dk!#U;HV+QGMt5KItH0VGxSo6IH=ng9q)gvp)9Z932XS4a-!|Dc&yD`&CYC$hk`WBEXl|l<o>+D@6$82!NzAET8}@77I*ZpW~U~GNeGi2ocfH3S(QGqIJZIyrmMb_bNA5)w;w;g|2H4pyK8(Trv=n-g>h)kl(8G};xQRUf}eVMP8=<Kg96Rw>#zxR>7S#v4I|VSh8)kzU?$O~cSX}mH!~&hr#Ma9eV$MgKR<6RhXfuI?>Lz>IK-^hZP3)qwC6Jdlfo+;33ff3bT?!~2Nw5+ZDio1r73BiB~935@Q(QzvXJl7OP1wfm7trYDNL1+IGC?Hb%c9n5S$Nqbrn8qVMKtIovy%Z!RD3fPI-}J(sHqR!UnN#ILv8!YGiQLKvs<WQ}<*~D~M0s<{T>B38?TfEZ0n_&W3)KCn=yw=tJ&t&{~g1U_m;PrwpMQh=v2!9r!}3!_~6T7J^~L+%{7i#%-$!4h}KaVTFpI<K(ExQETcotwH;2kgsJK)G5O^I__!wG}d}9Rr;0>X=dA*xs6nHSU3yX3OiTt)%6xe1J_y<q9cAaYL~Y-HymJQ&#R+TQ%&baaH9c(j#=Nxxm~d;-CG=o#XKpCYMK2uoV{6##Fkr22cyazRZcA^Y@GWg85}hQZI#+KKx}$oy#4dMKI2m0%#V_IHl7Eii)EQDrV}N$f<qzHEVdvex}TrNL7LAIjYv<O#wPGNRGx%+kvFus4bAtbAzVA3POYRUZOApi95RLx&1k4ctv&@v6bus!fZF1-x&Z@j<?xL@l_rytz0MXe)vmx1V3+<3YV;gc#35Xs0IBNis*mOltF{(RgZ${`ts7TSRQ3Aq>8ap;OKK`0|G%_s?rS3gGt@y6D-!caZbHmh0~q0=;2(=DscoDj!$OS9jOetPlKDJD+<p>dq0EOAfQeOPIFpEWN0D+$svu13C|)TOa@jQAmNu~uw3YPCiz2P!k$w_Krz=nV)i||WQX9EC9lo|};zS?cd-VAJhaa>Lzj{(hhR(!JWDc9EqlT_e2?(n${AV}WD4Iv`q-2nHJ}Tq9Oo<cdsIDFlnsJBkGRE+}oCV`k@jkGNJ3ue=0zr<p7Ty*mMh7BBeh1l{$|7)Bu1N5`Vij<R*BgeYRvIyGzsA_yG(@9CwH>;Xj^5i8zzHUZe2UsxaVZnVtF|A&jQOKY?v81xPJ9Mk?xhahs|{OC#){XsNLu==^|nnkI>;&F8lrN}%sTl&`l5;Qz_blKgoZit$CN1}COX2lx7&2jIl}G~>udIHwRRc7DoV(3a|@(u@Kpk)!Nc0D`fwQY9;j7cl`+IekSpJ~esq{XiDo!*AXb%!Gwd@z4@$TTnZrpL8Y?NE;3#7sg-AWZR`3$b6wwLwt0h9%Y2H?*=%BUNs&Nh3sdk(9SQY^VL#oZj34<Y>5BmI>YsoUp8*^wM=?>Oq&dTwc$UMb4UJIE{-d^E-S(4O{W=aVm3snj8k~3CMt{w!p3Zclzu#z>z>`)g^-x#a}rkQ#LxVA9#0c>)<1!YMlnh>;rAbFl8K0sy0->d-Cq7tNuJ_^_n)KE&E$?Ta~!G3GlfjXI}z0%6fV(wP&fjZc7?r(Xn1!n;jL9B&#Jf+22{_Y~cNvX=YS$RQ^vvxi#V33&z&T<l?vzKDW-Kt2yyPeHA_^MKzNQmZR9ASvfc6~})!dJ70dcg!zugAE~<|e#Dhs?%jWr&02F7e~)0ZJ4$Dc?xpOdMc$Lkha-{MYWy8#h)1P=v$Sxs1|X=VRkM>73R>os!Qa`-phJseBYuBoVwioXB$Jl9*ugEU%3re-O6ZP>fDb+IgjBkGx@%K#IdD65<?`K)#=P!ksnE9wqy&^cH1bO`kkiqzH0S4;a^G;+9mVfw8Z98cVQJ4>^yyFQ@M-RcR)`US$vZ1i_r;0tBjQz!6La3k#_lNx@I6l!5X9>d31LTl6>%!Jx&QiVqrrnX^o3L6x)RQ4IXNj@9t*m5zaPw@TsYsolSun>AOl59Eh6GQ``p1G9v-5wft)EWGWSxiH4s8^ws0FI4TlE#Z3a>~*M03Lt2eT3C)9Z<sY3wM$NR*t0XnlVQC{%c3Z<t5OXk-JKcDy?Nu-TjG0yJ`npnbeygbrkbIr?7YiBn_)=Z!72-BF>Yh=2UppA=W$bdJDFH-0;KgbT{=U`)3vBHgeMna;rZ^qOB+5UeUHCzUWzJSHy9ubUBCvMa&s$Q$iONdtj5sRujC>gcW#IG))Cf8B2~snIW|olwp{11uPTcMb@(jj^Gu~#bXQ3AEJ_Yt=X+WWRoN4c3tk}+8c2P_)1$1l5jluHy6ZB%RxbWgN!b-Ty+*QNsiGP<n7bUwuL=c3d0QL@p)PlW+Yfs-K1vZlEGi)%<FIFEV>C06jj;<uS_e}+sSAE|V58&7I6ev}i@*G&O)G1;G$X?7a0*zHnSACfa%yU$q2-n0S+}=}uG1)(QIGwD^M({E--PGEUMv}ztcpgmPM6xI=;pO0a=JkYtEM21Dz<6l!pYlig&B4mVFsQXak=hB&J2mGv&LEAfv#PklYnu6uAE0N0)5KGW6*&tw_R=pEve-Q&?^uJgXIK}0ifure|3?yXHk!^m~|-IT$PNG0+5EfQ>usRO7GYru&<#Wkb?yYi%2Mzi-mNZ&@wDyW@803fL1RVVS;X5>{HeG%u&i32MZQ&N;pQzF)Y$Row6Zt5IYYlfOPcn@v&#M48+s!ps@;!coYI)sv&K@1x1;YySlZ1WAg_GZ#?x-w?|3z<q6r<bS(lmRREIsB<Z7`Nwm~OJA~SaAX^S{fp;39ynCNKd35{ElMf&DAg;k&D>k)WoJx_u6{K1tKILr;y3uuVA*F%bCPt+|w_8Cj6|-r?YX(j?qMEPIOpN-9_*avC^S+I?1g5c1EIw{Ok+qKL4x&}+KdZXv`fpz)^R&2JkEj}wo+s;h5V0?j`)3mB?kPL*9CBAC<0@|g5%-?OM(+C__eg5nkgt~2WwXDWMEb@Gw2t(fAb$(p{|)I$_LZZ;%99@VsLyLah3_!Q=?vmJo&~&TQ&wC5es$LLuU2nm9T706Gwt$fAj^5F9;7N)RRO9~RxR+YgcT9PCrY`7+CVhu$MlWTSPHT;w5PHHN83NWMxv{4%3aB?2AaWD#;IlF8iTL2ajT7#g7qC5DO((y6IDunT~&TXX;R-pd-Xr(H|PSwQgO$cmViybp~Nwzx148%<s-7}8^P*G0TLpzryZJ$q>C?lKpCKXhgHa-^Oq0rzPH(_Jpi<xHSRS;$Y)M`6OGd3P$8-1!E`x8%Zg7&B<;|5TuJHn%Ic9T387A32d3<)9(8Tkhy4ER^5xIt+1Ang;y=O5>H^Z&pa1a>moI-k7I!~<uq{sH!aC}TXBS_vOEkND`T4lh4O+34pg^b&$dV7O9G~M$EgT?nCgC$xPixq46R~~>?`8_az$Z&rmk?rW^1Sj-%ypuJ9(6!b_Uoa=wVN#c7APsuO?ArVx>UAQ5wv2e*0}0UmGoGO^aOQP4=+TmZZKHIG#!}=;sFYhaNEkk?IZ<_RP&^gR$eG*r&_AqL$6|1{YW9IZ#uosa8~HyPUhEB0|=jvVW-w4`z?cF!WF3Ml5f?usz1MyzIOh^(AN|ZpOE>41GBZoy?QxddD9jsx405$xR|X3iP-lMi$I__agI_qU78%&wa!W+0woWPzE=_<C<-KbC^JHKNpaYI$tX%fp{BsuP~0&JGCndt_1ttaeU9YL$ve+aqWt8Yf&0$rFP`te^ZLPnj?R?N@$Rp}u7jn_L3Bgo=J#(ffAv#wwAw`W@ESu<#~GDj%Kc=V>GbR>@9TAOzrM1hgW`3iA!me0r9$1JN}BFJiJ6RLR_BZOf)fiG*VXi6Vx<t-`&n~{%B4N0Z^daO=}S1x;6u_kT=nJ0oo2DA-MvYcq{^bLa^L#QPE~eUGzt`pHWR9%b1rah)nQ)4;H)-)KT=k2eDNPh#L>NE%;Vk57hfXw{U8Xp#Zh?q@~5DF7Stw>_977`9@j8{y0mmj)wim8YTkbkj-k1eX>p%!K*zGUhe|6rN!jy7o@}aJqHl%zy7Su8eeh5?eZIpDd(U?eFWb}iFtnVs2+_^T$#fc$`;_^MAUi5k#3+Izr!#oD6y14kMRcc$O;=900I8>38$D>osjHD7qRu4XDx%wX)E-k+w$l$2g(o2=U8k>>yHn<{F`F5V&~udy|EvM)h<&*LNlrO_qj-X|@>T|LNM3@<6__U`?@-swEHEo79Oy5OFaZhTBq%GTS11ZA^N#w~`Zm6AqVFhz0Bv}w9c8VKx#XMsd{$&biBVXet8@Nsvp#6t7iv;$T?~y<hO2PV+OChsHnn*ZOHUPp*b_I_z@twXs=$Re)ZPdSO`~I1CCpH~hZ-iGhF*<yxl*DBX;=C4_3<nfNUz4$b*JIRI_O&ELaP?GMrF}8*11N>Q5WvM-q&63s@kMgc$fRHiCG6(uO@S&<F_QThD9Yo&|NuVbXQ>Q?s>f>PVwrH<hV?)K-JOhdU%p7od<CZrxVLkr|aP;<D)n^uE+aXvZTybcU==-K2G>DnRB}=k_9`I;D9Z7n)VA`wpr~{s+-=wg32T-s3o-B_BskYoq35G3sawHTg4kXxnU{+R6tQ1S6v0*9pi;F%At0-fM-%J(%jXZC{U7@8S0<%aTxA0Zw~qDe_ZU|8q}}$sA`0|Fe(?p)D~F?%^gPzan^;BZ->EUt#j;})_wS^qRYyMakHfa{o2_x;6ZWiuD+z3RT=!M7DH8rCR|>iUSZ=s;t%(&TZmDx%i<bvLu7bkRYIgkQsLWBohWsmf$^}^yM3y_wjO|jI8DyRbqns?b>9L6U+}T>>Iwu^dR-MB=t}INsFUBL<&1iF)iH<+N+YYRu3PH&pcn>?QFP}cF_rkYtcP^Gq5`p{XWiA(T5-qa;*gY@?e}Pix@tMATR<t6a>qHhS)$rQ>x`=i`kgQ=Z$Ur9rt0qHzO(ZhJS__Q@{4KqcKBUObiZ$R<5{?|w?mcBf@4Rr*xg_lXyt9%yG_yt$2&{Qd}m9wwH1b=Pc0M%J)0tqHM;qBdP1*P!UQ+Va2sNel7i8#x6+es+KJ7%3}<t6^WHw!>8Z0rh>y%hH)&uJ;ng0mkH_Qf6lIxM+34mi4Ao%hJM<6|?O}GNue6Tvj;qNw{n-Rgt~lSsE2+vwoR4l!GlBoNXF<wSILDTr9nV3%O}^PT!tc%FEJ(L0ia&)i=3gV+xjV+iQ@O2{^p?G(bC10BGoVPa(d$2W`_1zmb@=We&iv-y!+Ci5;(to}+*w?F5h8GM`SKs?dhVSq^xaAK9$dcoZ7JfJ`k03o|3+8*pnBfA<%v6&FMolHgxSTH7>}53F%Kw?s}K~IFMmshU_>#8t-pPY3ou`Qe(?fl;X}eVPcC2l*O=aJBh($fZ82tW>HPA=uN6GC*Pgjja`CHDOfLSLPt)}Gvl5p@E?@k9EDm{IF~)~@#|0ieyL|aCq0T8mz3Irag3QL}>(9}7ETYR7ze_#w^*>$w3dxrFAO8@TUPo9&0TlRw2bVAYbwTs{3h9;^MXrAt(rUhp=VBIKeCh3MrF+i7Ql(TSb(Xe5y)o56)-M<H%sa;Fq~EN%aXj7F(QLW5qpj9nl>rj}`*;5=&M@4~mS1{6Y@i2VA5|}t1U;wu?jR&BIN-eN5Cc2Fpj>ds<(qqCzj5#O^5xGEo-RQAm^W^y$%H_~&lW<*um@n5H-RRT3XvTS;hEJc`4hRQ108CbGz^n9xa~>YVGcleT;ZYn;nhsqNSDLk%doewX8Nm1x?$<3DsZvGd8oa%vA2wq%2PEoXMNEZp!yZzV!~?hb@8KG{1W|sJjLz$l^D~FO`+fGb2JAtr)DpZ0JOsa=UZo1+~C~XXvn;?DJTI$`4kQuRaqBH`XB%B_9hCsTD}+Izo&^lL+9VK#C(O?;gYuQ7pY*irL>kf)10<{)q(v0^Ih4OxSI6?1<drWw-!ZHd)IV1gnUA@F%o}&_ypRdrc)oH=IQ?6yaT?`lI-DCB(KDIaNemjx3z50#40tRlm~FozBV@J&c&BwsGE!5r1pNq7hjx;6M7V5<nQo$jSR*h4k(OE8;Jp5qDql4<3=W{%p9@=Rrd1d6i?LgMlj89@Cj#ZPFwflt5dy$mX-wZ{v98z5L4&@ux{=-%M*Q0n*NvKxp+*7?3F*K-p|G6=BD7kBeWbCRyV=9DSDj2#XLu!GY*;Hn`2%BFjTQLtF%`IoILg^+{#z1;q8-vup;?%$pinCSNT(><o&zb;?tJcn&<TQAIFt~dsbN+X!#eU)={NXZlfF4`S-D{>#1+jfU6KI!X31<#_f;p-hbjhy7%zIM^C)Pq)%%sCM%pLylrlc(+gij(Qmqg$Lw?IhVQel@2k_XZ(^x%9uV)IV2&Osl6V@<0LblAT6`_%)mQ!m^VwG7',
 'checkpoint_snapshot.py': 'c-qxj>vP<;5&y2g0#!edlqjBLx0&P;)}zL9?P<pG#BMtiiRIu(+#Mpm3Xpn^{`c<UL4rJ<Wv8EdA0!f3U>A$s-#&PFc=&+{Ugta`>#itBwq@CMQ}e2o-;<_W6<lt~ukSt(TIGbdQWMCoB)lSZ#mL)kv*Fc-yrCH*SzVUAjSdeF4|v(sq9t1@w*_C>*WYAa+2@kBTl-wggSDtj(!dyy6Epn5ArZ_zwY=<~A9%Co1v@x+eeve}A3uHiNMiEnv5Kw=mU~MOtB@=tUe^>gbq54%>XhS%cV1<jRqemv7;@phq`$m4f0uszbpE%CmuW7&@PNS2V>e9Pr=2JytUf!1UuXv<%jzn}@$XJgqn>X{+m@A0ixV$lbWM5D36=`hjuxH0fJK@~TVJy(Wz`Kz`1Vcu>F2*+y!T)&71@^Gur%iarryIHJRKYy<ZMlDXu)&ZvJ|#k^Np|n1xvM|enIUV9a&w4<mgC`KPj4ZEyy*y$E)>w9S+JfJK>M7NqgTg!{QPaB>~}*RCP;MbzLlppj<NYPg->BLWo-UYwyS2w#Xx_cp_ydft;xokr&L!Flc&4mtQCk@`FvCDe?n3o0afgrF7_cftK_$!r9B)d#ht9l|YhLvZYlvk~bti@C(;)#x@90Vx&bUiMU+zino*FP0tjQ(}5NKe#Zfggn~tg2EaVq`og<BKlzoOJe;0<mqsVa@nZ50XE%cMUX?hf-Vwt!1hb$wh5EK)VgXi6C2OCTDoyY&3X&W-Vhqj$g3JhD)u8AI&~Z^$CAdAQXvrM5c0<?c5^GrKd&@l}F_EqCXXo_hU?iQW%xdGna!XT0Gaq5lXz-S|TZMyqFsd6?`JPyL0ihC^Fj_9{ZU${5+yt0ym~4Gb0_F+3zxeYvUJz~BotX<PW4dN3Q@UYyS*Z151Zde^YmNhRK?FBe0M`482DZHvET>t^@&rUeKh!z}l<n8J`@kj2ZUxtOwc4)J!IF~4Z^Ngnl{eloRLkZ!@L%!ZuZ#*NuH^C6M>IoEzmqTBVcWKie6?IGF8_TcUnR%E)hm2bCJhpvz1p*pQ8x};E&;Sp6HsJUP$`KGr-O}aHKLl+6x}#YeaVV7SgWA71rs5uyVlyS<2OhY^)%{D5$bUZ=RJY45g-V{eSu)SGawL9pi$xNn%&8Qcmp=125}W>gt6P3*<%nQQq)N2Pgp3+n2bg2j>}g1fi{-Mr}5tPN-}Z7a@gK4z~&(nX?@=cx&WxjYJvC~ZHjv3dq-NpAQ*UA_p_la(VR{vnt*$2Q%N<i5$_o~Og?}N|3zgDFi=-K^XGOnaSDhPru*Iruhsx9y*Xrc2Wgf%l5_1Pt@s+8)5@|(w@L?=)*Rgxf}z#I9+u?Nvx8nT(9yCf^lik$u6k7MS$J;n>hZ~07yKc1Sm<$sG|BSw*srM-Xpxdjh3k=X3I%-*D~)}u%D}&>P3(2;`sCXY4vxYfyKB^W<41AnVN6=Wf5;`k3^0xh6~*z8O0^OUtYU$h!Y7>7yW+`l1WZ@(#t(ugS0`wimJ{g0D*4Nft}I~o?2;k4(kx>Q9sz=(AB3m<G)k#l7tip#PfYTHc#mPJ`y7$N0gdvMK0qi;JM4L*4Txpt;p{@aD-};TKAQ%Br?wQbB9a5-@C+bg(N&fWdG}hnx-o)z2)`c6B~*1{gP*oOluHpLk+K*D7w1O4UR)%y7W=hgtAMiCe(gO*G|ziAjb`or9$R7l(*bn1_X=@4xW_p5EY0AFCi*-M9=)_rp$8=xSe6<@(?tG3=z-&)&kTeo&ScMFX`h}3kXm)T$fp?P<TC&<D)Ni~cbxC+;PE(B(y4;6;i!QCUfF+h45RWI{`#6g#-Bt7<(vv7X??9;1J^&@m4eB-xB*Lmj2g1^g0?u**1ZP}6u~p8v}*-e9{2fc*Gir<(ry_MT}6N`=(`s8u3;FNKy|Jn7lS89n1&vE=@5z|D0H12FtXHwiu+-Y2;nl7-FnUMd=KfPwshC%aMOns9ejrAM${WPzElPDHLp}@tT4h619L2EVNNk^$c72M`(<x;NgSVsBaLF^i8N|81Bb`L4pxq!scRv$VzJ&;DXPTITNRDB^km(MjHNanS+8C?foosr#Qb;^yk-}TB4hA0o2o^(0!p=Mq;`@jt7y$Rw6D(Piu<wksX%Bt9nr76d4p2*CJ=hJUWmHKU3gu^J>|O(DU|v@oqxJ`9nzwxZ(;2>f6_jJQS)IJ?%cp4p~1p($2m+;@q!tgH15{udNq~zCB)2YKX@h(qL1|8Pl;JJMbSwCZJDUsQH8J?(YCLPb7lLX2f;17_7sVirp<ju;UdyhMGdtEV5F7vr!ifVmRB8{C3w(Rt-@wVCceAyjunGBm<b{()25c-s6p@*H@)xDKrIhZ3&#9_`N_CEeHOX;hMs9(Om*LL?<+q8I0G4UC&0WWMmwzQDrE`~BBAlI*+psafO-wE165P&TXzV1Y+T*L(PE2o7#RV;(YyHr8>qC~#=X;8A;bB*waZlMyLQ}DE*?iW`LuxhL!1}`@IH^92V;XV>Y%|ij+-T+*255ab@63t#W9iNf4ba`p@SNkK8DI9M)B5ZhlHLNO#4x4+FHsl0LoC?7+ZN?6We=K(2#cmODU&+^gRI=qlGdhFg$t#R)i$38L8IpRyT^*U_t4&>s!2U*XIon$&adY{By$ucuQTK=R@s98xH4Fbh_yf>-NMb8Meh1W`nfaF#jwZsbJbWOpebcAhv_L`1>R2MS`;li0KRyqLwfKlAh^`Wwetz=)Z-dl^qtj65mU5y9IMJmkH;O4t6mHkB))7;gODosUGu9cPt(qrSGwY>IwUvv1dAF-n_Zj{UPGZT}>+hgVscf`^Cz@h8zv&fR!Sth4QU>^;?Xs9qVHId<=fiG5aFV({XYykD1?8T2<xcUD-%q9WP{+#7-@h8RxOeVuGPeNalIzJ4s6bC%R$rxSu_id`TX)8V~z){4jZ1*g>0w!<ZLVU75C9xb|GeXS-p~fCTt7SQ~Z}QHap1!2m+p4@|-cG~FPNCl~4;=zTV180(GW&pV0%hMgT$2SgT(HGu4u|KjM#tO;M922Piwr1vYP+A9XijzUtg+df;&Au~i5&X~D~oqqXzFQ;#MVw~I4Iy#QWta`GlzWrPiC-2y_6?8If75yci2sV7HNObr9_GHpFnqOi+6eMn7z^&9xVEb$4f`A9syhBUP)ZRnG&XkWB)!pK8ClJCeV^J-tO{JaxypimL6JK2>QPD#EbB`0uUz?UPUD52iYpm*>bEz0Jb1dBc>vdoR)pDW&#_ad2<cBZP$Y(5b(ij9T`?LOW$hS(-_4u^OFfz?>w_RyKeP|9O;a=n4>F(OR3~-aU7ZH1k9n^#)l|wM%(tbPC^yVnzs4?y5%v$*_a?Qr+vCL7|VEfyjh^1dqECc0`ENbvGsg~8a&arl}cnYhSj>FjR6vtR)sJ^?zlTR>Tk<JA(SGTVg9lGz~@IGGO2ncAt6EPnVr=av-LN%|yZHyWMZpUfH%!byJne1BvyHVB~Sqn6)9CHaLe!U#zzno0X?W{EGF49<`hl8iNgaAIpc5*wZRh*DK>~-)}2C{1yW2gH6vkRxWp%ohL>ig0RCI6$YF(*GK2B|SLbHH!|#3Zkou6+T4GAeTYVXAL!MF;-{2G21<',
 'cnkang_snapshot.py': 'c-pmG|8v~Nb^o1z1wyw4Ac4nklH#0AE0lDRnMkBcQBIuW8H2k6c@Xb_3xFd<>ojA>Q6g2bouqEkCeFmk#Hl;3>yDF7WywiqqW>fnN&Bb#3;n#eyTAf>q@+6aj74Cv`}XbIx9|Jj<m6;+J81i1%b)7Ue&Wa5LAYtO!YvPf`d*9{qaVeo-!?YlsAr@d-$(}iem6kN_9-KbQok8RThmG3Zv`7c%S(eOtV~R-#KC3|dR?OxwS6N<3@>hVf^Cc%glzzAN3CQ!_BZ^PCQqitl+W-4vvV^>?CoIwO+PVq(1qs7=B-Rzjs`I>{?@?j2I;QR@dt5`q(O^j@r~Hu4*VS>2{ti*;uxX7jR|{x8`yP?tAqBY-}A#XsZ2~xPEPQ$Tz6xT4r1SRji5*CHM|g0@?w$+(We!4yMBvXRHw7iAWZ#uLcX@5{;qoNcuA)lG_%*V*EPI^|GG)qR_(DbfDhs>dRLg@3Od>nEI5_(cdr422F@}3H1_?8i5F_CmzP$SjRnILRGivAHAVX|HET}PzOlBtcy?`NHUI2xpK|H5+m3qX#Kc6~-@pdBohV5y%xPCA4E#s07-K0c_)~>iL$Bu>QEZr|UFk;H9?Rwtu@4f4a$q)FrG;6jJGEE*u$}A#X~!}v+Gks0f#<&0O08|LJMbkQ5^C&Ml0nmo&GqEC>0oZh;9e3D2_Wfs;|9yksNnP5$y1_-5zwlGFiAZyplEZ9=5FdIw&8cN4cr8biGwY3kZw#pZQ3A#f2|#WF;h#;+V-i7%VVu()ZTTl5->)Xy6JA;UjTjr9q`+#7j79z8V9XZj!TR|LWiz5=%zux>tk#j5BkJtIS*@xa{zSX)C<CNw?Yg|fB~uN#e!Y2X&}}{5VpN;7X)i{h^f-`<k$lKuTQR<jnh_reP^n1+%}7tIy7?09tC;)$zT!<>ebm(^G}~>RKzMR(@R=GKwB64{jS&YO?z}Yh=+y3%=MXu;xXE=0#dw5%Teev0EhyQ)_&dGY|hQk&Y4vMPqR<Y%+JuIo6W6K3pVs=Y7!fZhHF!$PB#jexnUIdAV3zxRFzFXwbVB!>m^px>O=vg8v1r2m@G)1v_gk?%im3`x^HZNYr%n`VU?ge8a^fL@uwMVo-~;de#4eCd0{(fd*D1r@)MD;-f)cT0)3u}c$jzxGfLD1L(+d~yqmvpgMW=qK5e4fJOuza2;MdAY>0oY<@Zy=x)KJ|w-(1y>=+lI6b5~Mu}ScuS8<jc@ru{)LoQg?O*IWRVPr-Z$1!N*%U5t9z^SHBu<?%17NFb2a5Ie?c5xPPcMuNztnn%1qVFXT^~N?%e-p=zmivscc=_zolF{>GV2z^)5ymQK$LsZl(&Lj@$>6Bb3%O)^w*Jy{jbqPA=|o_2T$12r>f7_KS+kDO^+QVnu=75xt6S)j40@KiKA1l-Gh-UhE*LHP2Mu4^*w`?6H(RugYPd1Yurnb2ftLgEXG%aeo_;JK@`RF>1TfhPk{a5Mzyg!DIWnjhT@9Mv7A=$3SCALh>FR^mD#Hig#{a+i^SA$f_~6%nx;uREn?JwukHZJ={^^6^XK(#t|Hr@Hzw_Vr^*QGT3HK55B#?^){Dg6gNi}86&)|_3JY~$z(6f&Blre`Eg;WNG&W<__q@=`Yt4ipQbMuv%!#bDFsy1l0axx#3`aPCIkHyVvF{vGsA>lO!joTK#ImRj5DKInrXJY9tOTer*TDWf}S!w-wpS*1hG*B<=iD$r26m#Q2n4nd?-<UZ3<kFJ!jgn%ht`^+@HRv>O2VOUEC>xcQ)ylashf9QS)r+KRV1W>y3tg|*Y<otl$~~BD4N@eq@e+T5$SE)Z2s*($xT+oh+BH}+tsr%bjjp%3Nro+WwCT0B7!w!-ePQJpQP|ym2KI$CWLdpqq8?N)iv(X<&K2$4u(_d2&Rm>pYVUesBh17n!xDU@af}!@rS0C7!jphytf&pilu;47>fz?}MQ2@a<Xtk`LUN|q^rp|IAd?abncK3StSO?dMV;znIIuw$n)p~`E$u_jg$?65qcSrym+c{oN}AZlmsyzQZ8=76J<hqnYqpWMJ!@3vPFD3}tawt4&4|`$7++Y(KIVY54n?h{FPjLYTbYN>p>O!u%`T64><#`uZ*KN{L)x$~kzGJf5-Y?a;55O=FB;Ko_LIp%9oYIYc>pwlbOwZ<kQwvz(WTCbMUKiinABdm*7wLEn4UgXCv^>^5(^J-C9O$g(h<TIng%`%Sj+k{1GXJ}qkl}}xIpbV9w+Ta?o4!jFXZ_dV>vNBV(-XR!x8oS^dP=x))M^GZktxh3zgVarl*ru9Q4y6J*B%{pB};}ZwxkKpuw%)wzBwzF$j2-b{zFxTCen$o-$S-Utkx(;bHrJJBS7eTHqNE==?x3kppaOfKUNBr<G9**}7yjA%bI2x&?9U!MC++yTJdJK!SHIXy;p|Y->`5SGW3I;i+Z3z@xxJUJUQL>$`9|;z(bc1fM~N0}Is2SWj5|l6hVoQ9I))!p3AbgO++iL-$QS^;mz;OIsaS6EDH%IQZG!y_>(?|MB+@-hZe5mrs6LZICY`213`k(yb}b;jf9TIuxM2<TrYBX>Og};Gmu9M$j#Z-zUc<iWB9_`Np}Giy$|mA<tkhyFut<Kj9s+{SCKt7vYTdRj_=RW{#d>-x6=<mVQ@QKZ<*aEvzCd$SxUun=qmai_SX)<n5U2A)PU+&@8>~!nWyvNlswrNRS2F^d}KY)y$cR9=q%irZN57ndN`y_}_W{_XImVM89th5NIkhUN)g{SVj<~<`SZR)%B%q#=dhXUTT8$$O$C+ykAA$tw?nWqdQ(_+TmZl^_s`-p<x!7l6qCHMgbe)!Z)cK>uKY-=<5)h`4HDIulnGA;v42ZvTHEEkmJ;n2(OzSRc|F-iBpBn*4bSmK<)=AZWYh*%4bqc>J{14^blTwSD`sc3P9;>*>)a!5dt`h$DpAURUAdBIJ)B0L!OIMBNas_iV}oxV70NL(t<ToOTpNquPld@)%OgAM`4lq-CW&LuOK%PMIh26-$|WcvN@rYHB(nO>vdsOZH3WJ$mhGPJ7EwI-?SjQkLzkPHw8$4rm+_$UBI=z9JHOu#WQEm)xL86`K7OZ?ZU<7l}mrWdU@^23t#`ni{G4dCcS10tKaMdSGT&oFzSCRPSU~l&b8f_C+&qv=n<m#WJ9u+!fZ@DgFi_}Eu#sR74jyBw8>($G|9bi*Yfh#oD-N0C>ocLoCKRRbSqGSW<yZZu2HWE{v@A*qN0_WIAvBQX7W?a=x{hX%n*HaP=d&8R?Ccr9pNCDSv`_ZM_sNFe=dUO0zGe5XDcib!_2=E?p^t$IVbIrAcr7<!g-8wW0!ptHHOw9A6?!L(<zvbb#w9JnWghrR<2xj&#at#(Os>*uvGgx*{F(y$|WH3!V&Bf|2a(Pgym#5wUsfkb0mjrRI^OUh!jyN8H@if%wCuq0*emvcy!fAHKeFziP5=x_2_t%WNz+S;_OUbsC_+TE400Zpp%*;QM*!NYb;Y|>w*B)nEEj3`zV&y*cm~QnG!=Aqa2Mn!Oyv*5QKPK@;qf+QpOZm5xl)A(lcRhTaibf`d-_JkRAvCEF~Ko)?A;WI$kcZ#NbK+J}J@15-nx%BDP+Rf2?z<olP;y(GHV-BBctUz1GR3I+H`2@n&@#cpCX6z*8i`x-zOa$`aJo*4DvExs6>8nJKl#;k!YTQY~q~u(S_4*$CLED>Guj3sVVZ7@aaU#SXVJ?J*X8`4DOm+9RcT=_m_Mr(hP1%?uZ&S7~F3*;#uQWqHwTl$Het562wOqT1m}4?m0gL6l)a*+2(w{JG>eK97%3iV9P^yfI9O=MbZv5k4C)8_}3~e$6O`9)$qS7#>kmjgo0}ngZM}&XuMfohE0|@v}gxD4{@Me@<(MX(nBEyf_x`oM%Slkxd&|i9t+ywHL8@L2j6s2_|7hKdRCI5>986(e)H7H_Jwl2zjZ`Uiou*aU8>RghAzp2qeU$AT`1Y)4NIEwbpt7vvRiDus@Ib`5ZC^RLOUxh=U54QThZfP2Q~#4-Tt_b%f3`rs$&xmyDR1_INR@CVrB26bW{AW%=Ub(y}1emF3H|ON*<EYqfJ@H20H}tvJ<{<*zNTe0|xyu=wK2m9^po(kx*6iE=WQN#2Z^#aQ_}*pcHOdhdx5)XYIR&PvpAo0vF<HHyR-?FMu%XMmpl!g||AB53wY!?_uIy?uS|Mty3!viI@t8hDv?#Owasx76$YPu@L^{z&-ChAIz2lX$RC+o!W=Uk}D3knyB-CbIAvAw^)(k5zR8)nTOYa!J~n(pxSXnEZ8`S;(fmzjT1ba2OQKf$Y*=17R?KCu%b=k^eJsV%-grtL8*mMN`;CS-Vv&Zz2N*A<b!N?o)>Sh;RrBk=x9OCZm+M(=G<Vd4fZuU2;_k=isTU7X0kYQe+d+8GGCiX>^JMr@+7h6$mP@oUk{|5ra4n--vzC&u&fxiPDI5a2O4mX6abxTO>NT5-}Hy?5Z&nS-7XLTcZY*Myz~$j-;22_#*t*@|kFgcqHK!|Eh8&5gdS~*y&LHlT0~Dqo$xu6e7jF%x8hI)nO`j2ug-Fjm&hA3U2U&R5hJ4vo#*M^Vy_o>vJs1d*2WImW+NBEJ4zysgoImT;jxt?0DTRE2Fd{IwlJivV@?D>Dc4VL|P)P3Q3mO_qGnTB-x78Ly#Q2M8<)Q$u#J}k{lt?_{jUH!66o#$!d+wd9;u*b{=9o>TnceFMTFjpCcNAqC|BY#Srtz{pY03d0%p}i>=|1m#aAMFxFQN_*m92`{eM%7bZKiEIYH<k>(PQXI2d$=T0TSjC-op&MfkB8o19Tcbfi&C?#NffM=v_pn?PHpR=(@Cxe(L8WT@0uw+iUZHf78-I>iy6PT!|U6`@Q?35(DtC9evol@b$F%daZz*+T$L|7R)3D-`_x)q<QGDUsH{?FgtzxRXT!FzwiADgPWIIDsOtNzk@IBY-#1JfH>&6p~0k@mrLGuNr`DDoFNQBu%mAC>wvOD&vz%10Ix*|gwdo>Qeb#TS7Tr9YKYHWkHOK~*Ye;D-`qK@*qKh~!xDTV)eYQQw-)<K*Oc2;yaxXC*lO0XQh>Ae82%3O#V6+IW8FT-$68;zkz9ZV#w3ftJE)HcPC5DN!gPWG#Xab3{=Oqw6(&xGHTN#R26^@s#W$!j(YqQKcoifDOqBX&ZE=5o`?+@s72WlFYBd(CtS_po$2!u(Wh4$n%74%`p|txnNcp34UdrD?uU>w<W=t&zr?Z)R3ay&XCH=P3#ilnhlpXicU<P61VTab9e8LukFA2gM(k+6I_1UbTn`*JR5~F5(ZN(o9kkSB#~GT(Rrk<ROo7D5R_BGdnwe2wt5~Ys$z>B9RX9Lo0#KxF-##k%W}X$T3YO%UqO}CZ-(oY>+@V#r7e@8Rf`J+fTNd*g;B&qAVS@>rDlw`FglGsf}gEocwdbri8wcNG!9Q0i)31C1lK64=V5zL)0bjc&UJ|>;)Biy&v;<^4dPp}q2xKC3Lh%>1A~hCQ9c#4G>J&PO77<Tj8jC<R!3sMgwXbpT`K{`Au?rj6@8yIW*&oqTyQ&&rU$^!n7=OgN|8=Gt99BEl)yE}g_k0kghN}_UDY$dRSCDD3vNEY^=j6Mz;Ki73<gmm5U6G=xNdqP)>fs`CYsMxa>mUinIfAgfMTq;CXJso>1hhO;hQ3$$->GNuWq!lfqG+CvOKymO>Z=$@D!mKp;V>VWH(UNzQ`jaE;Kk&j5~V7O-Q?JF9;|(wHE0F3<v|TQ7UpCo3YpLEU@@DSV=6H0<a!ywB+RpY(MuAr3|8*5dH!QI}1Tkam++H7MQ1@7z!57e?cC?4zB_~GsBDO_|!8QumW+?YT~J0JvozA$;zXb2@*CF`21<C%49GvKfkiN=C0v7fr>5YC9YbbWPM!0QKJm5wtRNw+|u%SL`BZlR@W#<QUI%qU?`d<4s@|peY}QJY3!nMDkQWWV^-UyCn}faw7DjW<A^QLtRz<T;lehmi7S~{N0z|aB%g=tr>E<SQ{VKaUTz$JPG#!tf*CF6#T*u4kC^A2Dv-%0D}Wi+AH4S9+l>s`Xo!y@RjaF4E|oW!`a4EYs)N9+<3L=xvgm$gW%c6X+8S-_;Xo+jgh14w+f@ZUWMskuy?tl@?R)#bc=zD#JH!3k@9)3)=EHm6Ke+$9gLhsX9=v~N@AcpA{ph_<Km65j@8;{D{_bNa@XvnoAH#=t|KZ`?R}Mb>9-i*Vr-P6Fu=o3a+kfM|PhbDRaR2RB_J91JqEu-AjURpb%kQJ<mU?>SzIY;&SCMGk@>XjU%43#Ya+g<@F4vxDiJdg<C#S2^)9aUykDsQ%-Fo^tKiL6U%=papp>fx|QQDaT3p7De4*_1%heNmpJ4lBm!~DvXz23}F=X)5R3mre)B-7W$B@c|EkonQrwJWscz0Kj)=CIctZsBUyaC7>?5YhAQu(e)UZx8$H6=LFs{Z~Yj<wDSMp;9*e(ARN%3U?NEjI&?;8d8an?N9-@GQ>iJgEmonK<;%I<t5?77<nj0h$&%x>6L0&X8onVd~(ZLZ#S;bIP-9!>h#*M>flyz_`|!m_kQ-XhxgvvyZOPxyRYrteE;FcKi_-(RotaP$J=&e_>yqPFy^$yvCw(I_@7r^t^DPaH@Ju5sDcsB_>`#CAt5L-Oqd)s2aX90%<3He-o0&B=N%KcnAH>1{+3xiiT9fynAKDC{WY`tMS6eVtUitRkAH4fzl8VKUj>|*V$NurvwZMlEP;mfrsixrM=fg;`cUG=HHw7}v-U^=h`{S%Or;|f15nAZqZERjm&bMe4JdI`bayy@^D>N@OuNX3dNB(^d`sQQjeC~I<7u3HI9I!TwzkYVZTZT@Gqu&C5Tt=~WLn8`GXZq2A+bSpLwVCMS5wk%@^WFOo%?nk>lt6BrKxg+fLJrk<ZIW=0=~&1LKHk3g+F>psZLOIqLh0k0WZ}!dqG)@^Q()Op3m2p0nd{PMcoJg;}rB8ToJn9LBjTLlm$_?4KA%*zPxni!i(<ODy#!NUm|UTDcA<eZQ*DO+dzmJfm~s^<mA%o%Egs6+>czm;GVs(xO6cGMxNlu`ybsO?tT9z{n`J?JA1c3eE9pf58irjc<?WO*t_-Jhaca;!w`Xwzm-brqx<{cxdjDsd;gzqAuphpi(JA*OrEFq4LmNv#;NI(J-qkPr}y4Jxc}zf8{Zx7-}>RfPrsvH-u>j^-TT9X-~Vv$%{%63Jqe4Gf?p>WTfSxlx^v&EZoD$hC^rpPhQBH&V(Qe4T@@|t!yU&m8<|Zj?9*app}e&LQ(W4&lxNTyrA=$+rgh0`llD3=BDEwhOG1^EZ%h)-(-a9(Hnl9SceD#RkGgj$t1ICMr~(cOkdZ4BZdwwaomJI7BJGw%o+;j6E*P9BL={VA1A<M}9g~Ip#$q|ps#0s(DHOMTL92}b-3GCjAC(-S_PL@6n~@8&U10hKVXHeJ#Hb!IS&+BrR4b*eIiAk#1%^^HRr3|9Cb5iggGw7p3AhoGW(l~g<`r4$mI^kFM!&>l4uh}DIAOy+(v8D>(ldCZKbU7wDVibx>R}KuYcX!wsFrfBYfEbvYVPy3MV9&(mms!snP2d*1h{elvtj4rbzGL6!nk9>jW}$TQ*_sDv~E^Lp6!y1%_qCCxO%<@xma6V9>Lukq~v;wLcVgSpSD}Cz++MKd1~;)Y?F=YFUvD*7UEJmGFm)LglaR=U0H*h3ZHWEg~g={i)Sv>Ml<b)de%zdInuR00@@KJaJk2AF2humm(3*ld`JvRH*xsl8PyD79tX_<$D~B<FfN6MDx6YLR_as6^MWC#ISJMEukkM$Q1wgK>ytr5SJvPRW{aGnQ|h->sY>=k2^Dz=;~woSN$$dR&APaMIwfwMHb`R+l|Om?1cwpF9uV8k1sj1Tw{#4@vSBhK!=R!bI8|1Ss3H|_0AU}x*W&PTWyRu!rSr=dYs+h&XSa`n_gMR#hb>fI!{sHIRjDAwg5+KbG0e$X(oAV2^-`J(sRd{RzEW-4IaRcBc?Mz1cqFb0;g%wE5MW+6Whz94#l3NfM#{{<AEcn0xEaK6rRuLdHOn<cSw5D_^$+#sg)9EqFaL|2a{BYxKi~ZH!~1(T--91MJh=a_2lqb^P15fki)bDp&X-pfmzNgLFD);UfmhspA>0>q>x7$ZWzZz}y#;YkSLjTRK4w3c;bgy-A(R6JHxm)=l)ty(7{`v~_j->VqZ3ie2gyy4$tX_d35jefWvZIRFH=yM4!>Z*$D{;`tN!xQ<Al7b4RkryUFsx`UKuD|BjO4Op@@kS#FL|fl`a#_tSUQ5<;qm^IODH43H5Z<Ii$PLnYK0yrXdlE{t`*}a8Lt3IODF%@^+QWBeRtLA-B6^DmbfnmnD7X$i4})h4)TbFkCcs=sn9)A~0ju;()xJDpXnc6U{Z<Cewul-FsNNQU3l7eoqF*Um^nWSGi=(RBBT=iZX#I7B6%G_Yh}9G<z-j83znKWRfx;5UaWDBT7zV4!fZaY<}L1zrz_DU!LY1Qc~y~ZCc&X^C)zTVOm9^s6q0qx+&clb4^zak&cKrXa@y11gU?-$sX;FL@j`Rgd3B4s0hEwD`Vm+LqQfe<1Pg|8lO8FM>=Cp?U;N0Wm7XhRAU&qe5y;~52gq!2rR?q3)4|6doPukdjXi~1^VS2{_PJ-N>Tnjr`Vs86yxMukVs#Z;vob|8aZd<C`qQ&HCQ8a3?#eKPds^T`kj*|Y@N+boMCyQnBkG(>-+H(U!4`~P|0{QmsQ%Lih_<Z3NTToz_aJBT)MDC@dx+Z%GoQrJ#a%&p<W!U1%?hKgv<+xwI~{xrBVv1dO$RYsGMe5hAq3E|G&(BAK?',
 'cnkang_transport_snapshot.py': 'c-rkeU2ojjb?^EWcfBY`$`mz{?agM^<G}HHTLb%H$l7gzsX|c0I~v*?a>5TwT1gLi2#^3N8Xym8ffQ&96bJ%z)0d)sO8-oquKz{P$Gzm`*UV@)$x{Sd;#~5cd+xdC`y9@lJ9l2lAC@vHq9ArfmY$|Xz89PanG{EOBQJ7MWI>Xz(yS2q0&axj^is?ga`s9DNhoG%wkmV+GCj?`J9qBv%(HYUe1BdRWhQ-JL`#|)BuQEXMU*D_4nGqHg)E|_)KBQouE5`4ritX|<T8k2uPmZC*N?7(ESLDaU!=aCx1(;JeHpFvHUEjDQ<|?wU*>6|ua`lw;B{7T1yAZVp5qO5P2lOFGK&!!A)udxv3wOrg^O2ZQeVGJqogs7=u2}mJ^2WSK8j-boQ8Mwl9{6Z&dyFK=ORvn(C3-9Gu{#KgZPRG;b{ZC0fg^6-g#6k{3KXP+nOb>g5=E4lVFuE(!yI^S`N*O<^m3b0C@5uf>i+Kw(x2&`AW`ULX*ZGj{3N11`($-I(6If6+XNR-x7C4USu{d>$tQ!gI%U!8OttU9%s<Fh>gE$Ogto!%wR*XK0j;oAATm2z>tZ84pe>q_~^ycBUq+E+MN%de)RbBkB?u#jVtRcSXpCn;5~4Kg*=d!1$>?!j67G|8+j)%92A9Iu8JJS4_#VF|5zn%GdKg`f}7SC(f6bDlV?YJPlyapP=NLiype^lvobGynY@MvKMLYpwjeEo3qMNYD3Sho5FxmGhysnDUKSEpKNyK!_;q++{kc(`Qx-GLF^U9QV7w$fx3#PRjc1M*<>@@lKqi0^JpDY2K+I5L=%ifnI*3cz!-Dkz5?=v9VdLKNRT$wJ<A;25oRuKn#BzT6iazobq+yx`*(E@u>BM$<IiE)t1li+j;2Fy+mMg0@&0{nRc~RJis~48bRc;fmuwas-E(-El6it}EB1%F)X>#b`?O7T|$=Spzi}~LB9oU&%#lcLf9Wq&h)k-E|4@qiT+>!*iV{>*23)og*_24Ih97RV4%W=gg{F&*n6$u=q<=-Z1w+4uLT;>Zp|5+U5Ip7409NI@^g1quH%hIem{yeW9JJwc$?;(+VVA`ngeLI)&+!Y`_E|`mC9-U1-1(tIR?BM|q?rUa(OQ&;lTS``QcrBFV-CZr$uGrnR)r~Y0S2xa0H3?W>ypor|8^R{pce|ozAZ@5pv?3rpbKI}Ajm4(`@5I5C+JoYu7&T#$hz1cDmM`$oh|;$8EP*48z|NA|5U4g`dc9M?zywWrBXH1KP4HdN3ZybM0%nROC?atxMNo)X25=2v06qW@bE~mvak*;lv{r?%;V37%L`l)!yYU!exe3SC_N&PaTw)NF#u{2#119fG#3AAkB$sygs4nPiF2uu$7}5D(;OP&*x+Si9Vv8$tnde{-fi1O-_g?Uj-T$OlnFpBDM6I$LC;c-Q5^ZAcl`B;G_LcQH*rdJ3$|f~n8P@8=?eJyqkM(#qlPj`rsCoAn#WHq5GsY3y{QV0YxqGpUt((@?*wb|R!zzUtk<3w2Z(R7_+Dw6-p0YKSrq~zS!KwUHpPuwk`&m68U^Xd6H2AsQ0u5Ro+yF8l9ewLmLS{U#9Va0{nfBnEf`tfSNm8~ul}<z@qVyrGl?9WL!B(M+12lo4bPW8wGg;Uc@UDL<fkh>#rI91{h`&7iWjp-`)&>E3y6_d#F_61`x}yo1(QJqtqxq;gx2-5o`lLnK66;Bt#$kG%h+tMkujL0I3V8{kKrk=BH3l6w%z%~mf{q|wzy_BqgIrUSR3~*b(eZG4_cR*w?(<m6l})2geR?y6w>m+k7~6)d2aW9x)io(khr{qI@P;?m;z!q5+CUgrVO${uG4SLF3b4|_i;ee9g4q0smqVxrC&Stt?Bb#*$S~qxP{XJ`@gBR=Kwf{5<}GOY2Od>GQQ2TnVGN-&ah_(OIfQ|bUKl+FSG+%R#r^yDoQ=Bp^CS=ElCewnlt%vi=;JY2&<1)$IFd`J_Z%BygC}Vm(ydQMKVM{*tu08<VM#8vj?*d}PY}>`wK<s-<tmm#=59;P4L?B~r{}OOaH-@B+$rT3vlH0zrsM_cdGY-qmrv;u5Nl$8YkGYXnpGz~OhwQR0j!8YQIKW9Wot#W@Uumkyh4`J6!!p&%m-Q)VEpX&9z3}Bp40Wr5HcwJ0^NQB(dBRDiQtn?pL*m_?PT10MqL&@(j0;0mtn4a=MqyedGw_eakPv~_X2<36~_w+b7(Drd7NsSYKKb6hM-%PtBkqq1CSDd#*c>4_=$zgq}R)#o~D%|bj~|&Jxigco>IHmQ38xN!1~@SPV+XCyes|^{8I2Im*9`m-YKJk$j1^H5o5VoToTgSt*F?VQdEFU2JVITr{ma4|Kie@K2cohVZB*{#VAH2e^AjuyKQ76ju6D&V+4VE*IKJXIL85&2J-`R@N^1}0G*C7*PPab(E-{6xHtc%7V>JF+;UQyq*Q3wH6E@whL}QKe)d{sfd9D}Z}wpH-pDhEji>LEH+eZ^^@@DZ=%rb77A1UJrSWW=rYIxmG5{};1*Em`EcE}TFQEV_%hhmkn;QWcqLjy##Z$nO<_mxcNK_v&RNrUb_KFMi-7HGL%Fqwc{iYnxfa<r9B}>c@S)jBa<&)ra(*81pQM9W!jj2pPK&zFTY{|j^qcp7;axm?3N*g-qDou44-3eczdSc1~kAcbwWoPcYkfdfo2erWT;RqH<4^eCmxF<z?DbT3lGXW|DHho^7`tDm*Odx_Bjrz4V>P%EX^AuAT>;sVsm8|XUw)S-gr2Glr-a`>+vs4ecPSfzxhjjemgZGGBYH1khthg2q91>B_*Fg_K!~OK$>jedLvmRokXCb>IMW*9}(P(tyA&&vupclp<H6*`GW-==RVEF-aXyRCN1+Xyag=iiHr?Is9HwJFZFi7TY&fnXJsz-8;VM}fgkbD3YC4pTmWIV)4=*Qk4-3J%;VC3{8*+6i5vi|T4Y=YAqMty+p1Mu4;bpS|>?rnt>ClM%BFc!PWtRF@>$<$cvTic@y3E1d%^aDw}fB4<;aIo8m&pgxkR%X{(7b@$Z9o^q;w~?V8Bv-6|GehinEAH0TvPv1L>RjcHyM{rh1ToAQ_bXeS4dRt2%~c`ZM-US%%>W2zI(`6>>~vixZPBJ|qE1{hv7-kuolm^u^3pqP%NIto!tR*};ZBf&V)VgkOxmGon{pJFYH@Tsvfx&%)?)9<(DiO;r{}P=*{nC_&4R?A%VM^$htp}I6AtT4j|P$j=dn+N!(d2{_DryLTQPu6XN--|mXl5F5pq_;@CZ9R+Ou8s6Z&#^np_aD7T|;hWenB~Okjj@MQ&j_&zZvh5k?#AVDzjKe7u5BN1JnAufb;Z7L+^4OjbFB^|`>b00Ra{|Dy$lqIKwl=P<GOIyYG(Ps?mZ9t=%k`^<yLj-2!0d&;XBb})HbjVA<La}Y3{da2w@l}>4UnEZ+33%&-a&o0|8Czi56fFY`DA`QHmCXkGc32#@lvB6?T!*{45nWRMuRkEtch;qr`G+5spk_qsHMDnt}!o`)kp+oWKP}E0h9k95}-~Ce%QdW@fJ7Lea%A`-GCr0pxp9bLTI>Kb6qM6K}$H5r`Q2VI|C~BqsRp;k4sz#X0a)n}l)JIp%0&WBxHlAvpiGcO7cHBIkaZUVI`S%nO+_Qx>uDJnBRTiymYu|F%fuYOk2%&b}wz1WsbM1!#3SorS0P#B^O$p_^tTCH2ZwhXj5jf2F!)9P2e8em!hlK5Q&@1!{@UNtrg+4E?YmS-M6jZj5YJ%h^(6xm?h$pG@!b7OAHE!dj&a!q%L3foWJvKxEB#go=*jF`qRE>~Rp$4E(mf)T+myIf~i!*YPFJI#U&!i$WFt|_T!SigOt(G%y^q@@3HYbyqsF<*!u5U2Kwao2y)|4dMRNM_du32153n+n6HFzpHTBZnpnF13aGaEtTtjscscrfShvzZ8DG<sTyTvH!Tl-$*32_2EP<>^G7D1KVvDXRyGUTJ@~H)Vj^om!PNlU-9-cArKIqkV%$W<kwx1{YxC#9=I6_C}g`!?dufVv3&UK!T7@OF3`srKzPTr>!7ruShv^{wwHo(9iiOscK0$5ML(*`hDV42%2B5i|`u<Q<?GwXz7|`9u5_M`!`w5yE_pF17ujgK5@=#&Q4ybEK0?RdVLtvE773Nq1>#Y{IoV^C!Xs6poIn^3N%vd#A9PFrOt9ZPQ+gwA3uK~_8ti>M|(IT=5U&)0kq;RFtq_DdJf`*n^%m%PkV!~zX6xun%ln>*SD3y8jfE(zn_`crL8s+0CO;Arb^{o);{mUIW3w#7Y?tXa&WGjV$fuO^SLi<^xGgMNIi5lRP2p9`_dYQW%||t>$V+ATY+Rxo^xN=iRtMo1MSllRI-Jl_b|xHD4P2wQ`y+O$0J)|R)PFOZ`lBBZhZZblG4z6+{k-Q`f?sFl9iU2z=ap&AfGuj#W{R~9hXVP1e?Tl>t)vplj3wAdw?d}<}=rWT)WGP%#-SBBPG@^S9|o8ZK%B`jw+nBitM)OsmeS;Fkb1~zWmhM+07&u!dm})ug^e+0}!=bRh|l!52ob6K+9Jh2$RiAMcmtR%eIhjlZ)G4y$@CShj=?m)+{-6`i~|8Qe{+d{HESd7gtW(975A#1$5&vph0cA+=!D|$U;u#Mvl#=G3dqXP=Dk3Y^3P$;dRoUpjHw|ah9qRR(FoM2Vyw)*-*bwiop5gwKP+mJDWU^Egy%2_%j>aiVe2$?0fqM`c5~7mmUC7-MY%&X$dZS+aHG-HM#20(eJD)38JftXpPr~2sKwRyTCu|)3gF&-6GIp-Lh>d6gw1o0%pVSVLj*G=n#M1!(ZRQU+)i;cR1C~8A*I+#jXy!-?o3O-sxCZr&J8M0l-lE7-Qs&cW!0abTJq^&2vb$=tf*vV;Mo83tSqgH^O5z6PiF?NAi4)TRE(&>L^jAAL}sj%EF!%Yh1^nMQV+Cld6?Y*S^R6_og2M1pvcJn!M8=FII$Dq$_R$;<KPOm=A;uW($4j@E-%mQ<M!^SuQG(Go`%sak`li+Dh6LV?ae6o30hlJ_PO@Z7W&q!07jf7{YBb7^e41;>8P?!&X(h?!({WVz=t=Xn;c%%d{q^wOfq8MQ#UL7|Rv*K*~@rLS6UV!e}nrWah?3ZbKMuT!5^czjZT%xmGW&K%yD96;!cowZd(v@MHnJ2;~H-Gt7%xuOk0lLSE#Pk6XfwkEuoB-%QW9w&{(mo(8A5pfviLXmBV&X6~X<!l3F!PiX^7lN~rpcQhK_Y3fU`xL^gT9)?vNs>|@IF55o?Y!_~r&g$bO$AYGJPPVoIee&UK*msEKE4@jz%9?slf88g+wnwd#sjQjKrq&Njc*~-=th<HY!Ttw&6#-V@etb*SBf4l}zIYA+6jWKjn~-rx2Ged-30g%1h%jv<>8@{E&QQCAk-Xrg8i@<+;Ob{6^RWLA>548q<2^Zbu0(^Ap|{`$v@!3Ba!^!(IMMIz)lvD*W@+i~jaYBmsnJ4p#F9Amn(=mLtcoM}^@H9BVN=lAhn0>52IA<ZPUe`si_>;39ncW@Voh&HA9i!=ir&#e3_YK!W4?bg7FRHEV@=238I4*+0xq+Xt_I;@71~9(!mk$-NLE9eXpHYz03kp8?2~=<zB5*Dp?w$K=&;PN*My8^2uuuIS4NP(!7*y?B?F(AAkum6{Sh{*D!x+h#A;W&^#WCy82h@rWdMgt!+^S3j2_T%t%zhY7>!U}4Wp4p&x^u~dl6XiITXKXO}~yIv_9o?t?KA&l<hh_-K>{eY)Sm5fAi$&^W$fqed-@Q{o>ivzcx)oe|tyWh5;YucdpFx_zhXNJKFjfrO0n%6{(SA1rbhR8pf(u9iKJc^`T~TT&VMYk9svjx2y9j!P@1c<Zt7J>r<mW67NpOhwb)Ac;p>0ZsbF*dseNr&a0}vK!9Ik#^O48xqx<%K|4%lJ*2_u4<4QDKC->t?>P_mY2*=-P5Xc_;hDI`KQEjHB>L-}0spS-14nft&?|EM1lwP0N4LkS#miG%85h0#NcBx?yiHo!C~3+$!O{UbCkO>z!h^R(o3RGc4fHCUvIj48(bj%oe{T$ZU)RvfcpZ@b5ZwE9{Sy0v19<;`39NAa_Fw+<?LYnN|Ni&iz4`Zlc=M}YtIxmx)o*_P)8D=M$$wmb{nh`x{pDA0{{EM5|MBOJ<?_~;C@q%?kRY51Ny2fYvVa;lMfa|uv#9UxQe37b1!;?N86?-lq!_EhFl;bO&q3O*-+cAcH$VRM+u!``>)-z9`s?5R*Vn)M#hV}h8-M=4fBq4oNUg|>p`#gvWW%Q4=<uwdWG2;{*eq^&<CSb`kHVB~BG1)8n)oak>T{f8D+4qkO%5L@j)SlLc%H%cD$Lj_gaJ5W*%%j)^_PceI{-$bSV(jYlQ4(@B!8C(y@8@OOWz~HWl{ajBa{05a@%;1oR1<zC^`^rlu|o&)e2%e6};I)SY@C$?Aqqjd-6Tbmty2mqd|y?O|Jtm*;}v;kCBp?*!-r<jIUeN2DN0n`LFrK9v!}0p^i@}wPd^xyYVI1p-5hXW82j9gscG%8DEmnd^N5wZ4JxWKBs_sr#e*az@bnoHr6F(<X-kT@Z!-#vw07OT~D>@fvcBNy@jeBuGv!9I=EKrC$8UVZy2IT+y3ePP6YB7wJ@S9FcQ=X!%j*gII82XK5js^Hoh_9%Hrld=#^40SQ|}zmU{0BL4}>pq=DDOfmO#Q+q(5jDOGT3xFXxVf36bOre9>{(VUy!FTLoBZ6~8#AF=s5n*Qz-OicejlE0Ch',
 'corpus_runner_snapshot.py': 'c-rkfU5^{daqs*Ued~uI&)nJFlXP(?Gk`3~cZbtS%4&6ofFZ`S<d7WiaE3P@yIQaIKO_%9USk{r0||@-dAxhb!x{b``X{NX{_g1^xsqY~AuB*j_D5B9byanBb#?E-!NF;hvwD@VY898soXv}MoaSs(7Uu~|@@1MQ6^rvZ+cf7{TCLa@PYyR_QYGbOnlIU`C^t>TW~*d&u_@BLuEK+ZgI>Dc6lKlga=D4iD(NlCVhwGxESc45kylJKcw97jos>g1PZn{L)$?>#%g(dvQhwyWvh+M`>NHcOh5YpEs>tQ%bzHAFz72eWJ`&eUtPr28>x#E&$_&90m9im}RhulWd%drpp1%6x#d9`h0UzjL+2nar9$x+|=skJ*%SW%DoxOtEn;wIopaHH#@iNKlV8nv2ir=JJ79Twbk6HhlG@ln&6?=ZhPL9LlhYSinz5kHC`SkvPJ=$!t<eTLD%d|dvaPM!zd!MrYm!F?~^=!zp^de!O0dtE1d%OaQlcSS+00957SMee)Q_&?Daw5v)+a{^%Dyqw-s-q;o1Y*w0CK2`XB#W=1=p-DnAg=3Vy{Rkseh<FZ={hMIq<-?rG1T9G5Kf`^yqPZ(EIs)dlu9(!1|~BCIVq_BFXOC|ICx=HY-qBf@<tXp2u10v0ImWsjsN6<K;-$6y~-q>8(r=p#k#I8uOmhH-<oxcIL(?eiRwDz-EIZh=Os)LOyRniCpd%HFqub1o?Qn+U2RSC^CC~6Zbowxou_$RUdv(*6%=O;P^u(L^0Zk;i?pn2drm|X030QYq@Jz3)vD?yIFI2|RHeT`I!}&|fr6`|yhuu9><N5Z#v8y0e4S%S_`sg)2VJ~1CEx^k28^>L;NG!jS61Z9*(z>`?ipx}P4#7Q*-;}q(ok%=rF1^nE}bXarPCf0s5Kxc^0=m1<x_<d%7sMDG6+r8tP?@2D->B_N!xv*P*z^WX>IWP#Huug@yW1v+w1kf+A)&eDqe%?SN&NTUu9?q0r+4<h}UtRE<lmNyolP6NI+ZHhi2@E1+q<ug~5ROr3=-50RxLxakY}|IXY^c1q1fcSYernx(JqmPab?4Oh<~5Wt@U-{5sB><Y`$JWj}aDO&LN4%OB^<WF89aS82Usesp26N%DS@&x$#$9OIy=7l%IsD&mS&HE?6CQ3|%e&?nfL243|uSc?X1QIrfnF<7N|KO8cagzz=p^aoSP2T-{FO*skDc`&7ZC3WJ!Z)yD4L=+6R4f#m}E5ZyUNOXr~u##g2F=|Q$*PPxGB%+*KvpWPrvN?X??FCj`t6;dCal8e7IJn`h_!w@9l?QToc~MipFnN=L!S9;`;3T0H5+ouaUQt2_N^<(VVF(J<N0fshGfPmD2gJ9&NtYbUgc?k@F>(<+R;~i@@aXmFGqz5vHQF<<21v}U%~p<Ia8)=VEyj|&BKT|ZHQuB!>h8&qwv|^_f$A<ff&kKrc#YgNSu9wJK1&~(4Ou*!B~=wwbzC>qIQaU}i|EPg7mou0hx3rvg4~b6t4Q9|Wjw2sdB74_K3VYP=P#ar7O9>Cg-%Y~Rhia_WdtPYo$?~*aNvrEU@b2puVFi{lxxw+EeQrV=5osxNMRed<C6w-7dj7PtBZ8AX|KBE$x7VYh-B<1>=dX#A2-ghKnOt$ZY<IObH?Yy@O?urawG*q;XDtazNnHMMoK~&3Lyd504V8;Aq*u|gI-&>I0UmMY&LT+Mtxi2Y|LZcjWdTA1+s47a{$!z3UFTT4t29tU2nW?jf!A9rUZ>bbOFw>4MCS%7^&FPat9X9vj~mLcyZuL@sUqwl!%FGarItKfD}=|2JzKpnQUcp%xb!mV}{NME?E5~7nrRwa>$M%c}LuZRclb3W#{qi0{w32Fz%C{Xclow6e<w#L@J}Lgk64p=MR}iu00aGLyT0T!9YzM7jz4M&^3$FVjJez9hYTY$|Q#cT3Kz;a3UK`yLbuyVGY}csJ`AL<Ddp~N2_|BJwz`GHlX#G`$S#crR3TgAWK2HCsWiG+T}QH_%5wew1Dsuh7GQ2OY22>cVF<>Hx-#$a(~A#^ZC4QE2-TCC_NsqpV|!ewHdN|VnY99%b(Ay;6Q~fEfWj^Y;!H7r?@FLO&0Tb#ug{NDw=YZL;wf<ap9^@0x-@sY<wa90Rq$GnpQWPZIG0)!G_!n2yo=P?QEqSMJ~2Cx11x;9M%PFM;`aoipGG~4QfRV(SsJ=uXdw74$)Q*!ncwe{w!}W>bY8z=1rn86;~K^xk#=@rh5wC`St)lAcg&&y$zh>o)0}#+-Fjl3qM&)C>E%tGSFqFeYW)=__m25R&;FukR^*OUUG<Jo#h9<teNf&nM%{Y@?-KXyjC|Gyj|=9vllLt90$6!>I8<zEU6aX!NE0piNehqHxakyECaZS<nff|9H%M}!3_+hz!XKZ#UjPMA-HQK7TC!lc}lfZW6$9fHz<bpjm=0IY_Wo#9jmxcg@U28{!-mU%<Xhw3g;vMs4+B%Xeg-TK-AbY7$mULSP5aAqRBJiRACHGEE?0|YS7~0)<N4t<;p#A@l9Ls32BA)4|Hgp#q0BV%*qk#AC}%hfo(O(7kh1Cvo~d0fMrejM*c>&8;KY*YGDiFsKq4N_}ISy$sipDZKD)oI1OUJZT%7?bkos^GXzON6DOk3{j%+RGK?DNg<VGxICMj_^$)g@R8uo-&oPFu(5x#NqR<R_Z=MTQL)>Z&fk(88<n53TGqR!^s9AF7BC2q<u^eRpHaboRc4#zmhbg2lJ<l*+2I%``TO!An8Y<RxKs24yOY*K1U}Y{WZ;tE#4zznan*_}7!+zBEVfP*l%<-Csg#T<~*)o#xdyK@5KKsTYDtE$CA!88~_QBHjUou4sJ6|dC6>MsnVkTjQAlBg}tmVPg)m7M4JJ?AZ#;f-yi*HBLDuQm<mPfvJBH`9bpwL!|`nk5^62oR-EKo&-mE)YKgr;iyA?#4Mmn`Lq+X?Ou4yD=H<zhDt1iq6T1m=yPKmsr*;4WgCWOFk2UG7%fpnbRxb_Xv^c}L2f+3kgOd0s9Jd-`GT?|b$9nz_FxHdTIn9<O0jgOh;*H_^Mu=(18lvk;D8gJFj#%b$L5u>xI-F5@DagW`!!lhW1&a$9ydEz^N<>>y(MV>spwUqN6$Uw-kTpC?xs=^qD1WR&C;BZ+`Gd;;tFH&oE)0ES#9alF--i>KR_OZ*CUEzrwfi6iJEx7q>OxGvV|EIPlgL07rLaOQISXopFW`>_4OvyY_DPd*N-bQy@{JEK!})S92L$3Omd#;%g{3ed9IkN=ai?A;Ilnld=fy=Jp_-~WDtv+4>%@W=mZSe^d(A9R>su=?E({{o>)46o1L{qVb(;dT@00yr>ROsF9^DCxWJzgx5UyYGLOhd#)QgFpP!kKX~*cR&1|-6)UX_UOiR7H$#MLw5P&zZL8c|NO`Qe)q$7Gj=2F%f$2+rne8ph;+5Eim+pM!%)GQ8n%WDSRi8Z?)!hQSw=+|qy-Ec49aFuxr$SE@$UQot=YE?43Cw_m?fOHCu{ciuOB`8;_M%y)2A<AoSud2xp&mTTlR!!LcC>XnwNrYP>A2;#2`b#?}XB4RB|eGiM{oJJv=;Q^#92E1#Pu<qL;`h3qzV$trsLK(s5rLyk$4oS}&5fOc%ACN2Au`IP+>ca+^2nP1WZug||G#=y8Q7=W#Vl(=na+!_2=uFs1mXD8|Ur$1C7XO0)3!+1bmXVAuNH_rC>x;T*mab_PS_D6#*^xjbUOYMYc_@oAd&cIaO4&C_2*Pft%@oW6p&#XkR-2jQ<FJWuC`m7|4H1LYYWmq>Yr*W~^<5JV&v>2k<E9x`!!J-$DLv(z^c7^pE%2@q~9EJ+mq5v5tNj1nv!!%4b`BO&2qN>9Nth{c#Npg;fd$1>3b@O{k7Z`DaJeVy2u3&yUxSS*5JE0KX5zbJ_zg@8*<1Rjk{i=agVqN~By^U&A%MP6Lx%tRdO6KD)@_9bynst9;V;w<I7D6(xx?2c>=`78{wlCWDhu2%NTUGTY`_rmE<@Yo-%xBp!*8+jvl#;W`7g|VIUu|LLE-@PzeX(SFZ4-&Jv7rvwdbW5E^8k*!8pfRSRP^ub5lwJ`HFnJ7u`l=DALIV}zM4!4G89SdUhdB4DsKd>5;6^`<uKG}a2S5vsF*u4I^R9i7y%wp&o1qER@M#<-P}E8x0yieba9PBg_1mpEBq2_I{(40Aj{HUU*5QS<Mcv9_+1Zmbu@7tuq#CD>)%LD=s35nQZMQ^@3tV0Nb~@<byo`JMbC$Z&l?)b0F+>KX)gsMd{g-F~_>{t7#&`DfF*|mms_l#q4AkE$UF1;t9=ZT0H@)m!I9dr*TKzG7MAPDF$$rjGwizS0hoUc5WFeAd32ufUPiG}{=Wg-EJHLRWC{qD=No3?TRs1-`I7y*19GQFCuVhFeRGvGuhkXwm3tizpg$E@FF2*=If<w^f4luztnpc>@)ek}gw9crq$1&=#d5-=fzzDJR1--Do#(9OuY?iJL-p3s7kIP1(=E$eKymFV76s@M*OB4KRJ8L`=r$FA^ieODT2df4V3JThSOISh38v;8b=NLmDNsaj@80`@bXRn<KOZmt$Yj5O9{ZqX0^cJnRb(baEz>sUa4G0gGDG+@8Py-u@i<3`|fdg@Vt&i&riM8^|I#q#;9smMxB(Rtc3$Z|>M9&@?o-X(#X>{f-wBU#wLYSDNyFf6s2#)vf&~t)YY76P;kcv2*DH!E>x`e(Or1wykYL!jZjMfUvzgtSwnrX4{QuvTJ-@3ADh#iY-+X#P^LQnv-kWUn&EPNP_#9~?mp)3<cipWzn&dasD8MF*+(>3xyltyhEM4{a;ADzB>da7B}4b3odd%(2;V)hCx?mlWAUuPR|;YYN9ONY$|Ui+!rAS<8HhE3VOvNf0{a6o1keVx;+t!HE<yehlP4JHMbP5CnVbjX&G^{mq-jBa#@BS$%EnXtBKJ{A{SU|)MAHsd1I`5^=672E{a>a)|=&!e+PpOIppK7ITGj5kD<pz!6Rv(HWWO)?`xgDABIYoumSdUI9x-Tan8xXg<4e(<sAu~WI8CChkrjX{jjz7(^%V9jP(#(Fc~aWY`AcBuH;?(zB&ijP<?je8T*R?E{Dzj$#LJ^Jh^+>r5i3Axz}j=;R13?|2I7-y%iU!6rypMPyY@C5`^#|?nq?UbQ)6TUvWoB|oHMcSjHKsHy+FrS^LVV<qSJY9v$!bC_pt5#gdo<+V;Rtbg!8j6`Nnk<rIaI$Gk>Ra(9ne`OS({kL-+#OhGA>TY&Vsg7!3LIDi655tsgBX6|=&EIFNya4T4!x!6Yf4;aBn47Q_Ql8fmRzO<SP%ijlR_ov7<8g1_};!Ifs|@x7Wk8@+<XAFH3MOa&J!r26PqjsnG}-~cv#b-k8iv1-o~50HE5Xu#na}jK4z6?_#WjphUdWQuxLtPoEs?}vktk{bkzoIk~;0D2s$nx2ohP6;+YSG!Yh;^#w8IARGCm*eoaXm^U&6xxT<<4mQ9?3%Eoz-!|ja>WorZA;{~1=#X0Tp%Cez*YOwjMuabn{h7{SOI=5Tjft*otO1RwvbJTkOYYPVTXDf?<^8{Dj3R!>@Uabnkcv*0QzzpZJtiePMd3<yvGu<&rtqEQw*#<Kcyc;O(^J&yu{XPSlu}Rw(hgkP@3cZN|6MQKn(z{1feL#^6bjBj2c{diY!j}mgsX)??zJ+2NRX6ca@83*x9g!GMOW5rhIwv(2w|o`Pl^X4tE;(o|P1?}#P&~j)=Mg_Ynn%+9^J65AR%Rc`We98a)fNqJlf|&BoEubn*F{+;P8)~{E7ea$BT;wS(ZWgB1-z-FJ)9hsti6RK<(EQYhP^=db|lP59!&)JsmU6D3nwxBBkMlTpzXi9vF!IArImZ~S299pceT%moS+yDYzv8Z9-mB)M;U|TJTUY~wI*AoXn`N)x__2uhzlK3q2w78D2(rIY1=XxNGd_|3x$O3pW_{Hv18wGzU*CVo62d67sBEoH`<e-xMOo}*Tl84h=fCRCyJ*mMk+5xL{=>{?%kGZ*m3$f=<O?jE;(XFL#O7bb@}WFZx`5igw%ScQC496ahXEnNhw;K1Twked|2K)<No3r52lTV>KNIG8u1WyGKb@1^cNv3Ag6Cqr~wzm3j5F(E`rmkd{1@8b8HB?Ll~R7i3^_Y=8>O%Lqc<wm{&kAL_WPp(H2QrDkm$zSbnwQ_d|vOtX(o{Fr>>Z^4q{%acI<}&xWh>Ws+<d9#1p43YukdM~SJ!S1VW@xR;t@4#zA;7BkovWeky2=-y%AK7-dLlCZVbM#ktXu}F@5EKw_G_riH=ur*V)Ey=XGpR7}yvtkLWTgwv$#}r~D;F+2;>ud>;Mk6QVMoz>?`cIVRB@jf_4DMNDopucaU+7on0!Q^K#RL%5nCBKMp~LJ$CDl-gI}i1JXgBo*hq=3+qM`RECR2oyoQDBD03ZzYNQw8YH}3=zyRxQQjq$F&Z5B;d3N_Ru+H<2FbJ~LWwL{I3B*=CW)#hTnm!I$6#gz-F0wb@b3azvof`d2KZMiveL7B3xn$mfZS0_l*0q{U#H1=n384BKGkj5O1m%WuBj1^h6F%f6_f>tXk3FG;kP}zsBa&r9DUZN+^4EMqQv0D9=m3%v<L;&5c!$F`ZkKM+({RaVw99HZ(c<z`~)QW%DB6nyB<ll{T641ys4$dz5DcCx(6+O}p`@AV*$fRRDj9_j``SoVL(_#9_0uI4}jlOnPbC(BKBeB8$;tAYc5YspV-NlVsZEr*=sSrYhIqt`=(ph55gRz-g$Jcl;6kjHoxfv;&GsfxSax*YA66T?O9M5Iz*k!RBAa8dO(Qi^hXJe&;W*r+joy#SD5BI%B;AVZfQMrBL&}IlHeC(JgGlfij8X8O=>yHCxk(Y)zOF`<y@pQLiA_@;BOIt$TIz@1^P&z`Zu5aD#Mo(Lyjphmj-4bvH_fab`p+y{*5`9m=mJ`!tyBLR_&20C~>KOLe7uLoV!Ax@-ccsmBW3ZoF(g5sNm<*|QxYZ5$Zer-)osjEi&yM+^Myv90Y|T+S*4rJlcO`54CT;2uTaS8sjG$Y-DA5)dX^}tCEmrpaci74+=$@9YdHhSw7mfKtCu+xLwKZFqGVs3pXg?(bD^TU+)MD@h(`iSZ!w2(a0&rIrS-M;~fkmGg@H{9s-Ft;#){|ojru`7v9l)8nVl4`-M6#*tpV19BI9fxm1U+u!z|3T5N1p6HO!pxiD$LxH<Hhxso8XYLpBi@{;PVXgi}CBo8BO1B47MED34V<o{@u|^@4I;Hy-4)B+?qYl5-A!)Bl0rV-ODOzkZdsbnpGEHWthf%@4$Hoq9aseXg){SakEL^R_;93z1^}0)pmg&_6s8Sk`gI>A7nB1vkrIBdOsdBm9z!G$}<IQGTCz6oww=7LeB!KUi^{3>9*yr!}Z%^2{cxgKb3^}9R;4T0ne`@hegiD%jWzmFrn&V44!A=Moes+9>QnRYu5G1hp=+Ec@G2E|0mYmVMTT<)2^lDLz5#+BHb;D$0N~Om6k_h8tz`pw=pqvEdW|`Rr19is_U>=-wqPLhh2i)QjA>+v2#=5>a<p4#vjboI>_J8>_DU!BZF<jzlLRJhqsGwLp3=M7Wy~MUX-R6IqtZF(reR5>U77a^9)Pj2+p(<gK?==l%h!cPg~3r7p?SLPh468y!Tnl_LEKz9j5VdYnA=+`R>y$oUZVBf<qZ*4Aa4O>se5PpY-C475E!cy0MuVECe{%dF<-GAExf+LC761ctqlge3Mu40z%yLIfeoQ&&wir=Ak_Bf?6^17Vn&1p3F!g5$D%YYBCP?fi2#(Nfsw-k$&!R3(5OzRg5FLyRXdCJseWMGuVCv&vaAi{LUU}msiQ3t82YHv)AKxd=AtR>)<Z$=wZUVsk%`B@2~lF`YvF#```@UGyc0C<9vV%?FiQYyV8GNR~imwK!Tn9U+vov?vXq9Avtt=oEPYed?EvqzyIgv+IFM{clf*GZT{|HTTpv@%=&#u+!FsZGR(U3LdAQC@=rO0&+#RVO*~7`29Ghfzui|qM|J0?6)jj^L82Mka%|Rp1_kVaBN^M$v{nE?`yGsx_w4{PWs$!JLdl7IDCno)fvNRpEkZ1f@UK$vM9EAX$eZ1J`m$ZSzqqVM5D-=${}WdiuPaeqO%NQV5Yu->59K?<d?#;!3PCvkWxbFfdhtQxwyUpRFfaKEuO#JBxoq(K@Fmss=Sek#_t|ks;|Lxen8{lsyeA(1MzQFES)Ra6dO!yrhk!H2YCPfhCZqy+dmSF~o31>67=mkhraIblLwKhy@-G$)>9#7KWvUl5$M?7Pgw50;Y^IE!TbqMvJtWiWMTHMYZ0jwL9uN7^qfl{ZLAwCnhkju%sYE+(U<>8#O3LMV?egxlx?kA3<;nlPOvlw}tTUj9S5U4aanG;?%=b<Qjw$5FEUkQg8XYy@vyU%q$(=eC%k6s4-33#vJQHpm_EK%@9)?~jsoq#Gdp)qv5nWe}B3d+}h@Rt!#3=)+ZB@@<7z7q+1(VeN8w%nNMxZkG{ts5?|Ev',
 'crawler_snapshot.py': 'c-rNCTaz5eb>{p0iXs@=>IQpydIkWO9HJ>fAY>s#f;0$f*D)Bkd%CNqi_<q&b<ahEd5{(IMQd$X>r2SHc9^u*VTWXivaR)6gm?7-2LA^l1d@8%zp&rAWL9QfdU_yg*AZq&gX*fxled#6&pj_)y7blfi}Cce+joOr(mfijPbZ_3U>1+2^JZ{o8oaZJl9OQ6-HS$PBN$Kn{O>&J#*=umA54=Vp7f(*JU1_0y0kV(rsJT~87$_DB<gg6cs!dX^PoGKOy}KsJe{O#Yvx_DKkFuGBz1bz(J1P12~+9jbTOGnNh9b-gYIH9@5jBlDLm;dk|dhUo6?f4b8nbL-ToWX>FCyR)LYD_sNPLGz3F&1isn(@)}Kem^HIEK>gk(tchcRD5-HX1&ZBuej!YTz8^g!nuT3Y~^=RCUN6p1N9@%d2z^Hs@KJA!#X5jl@i)ZF}*iDD%hWRP~;6-yB&AX`BwO?l4ejfu(&5O6w>BKybyYr#+WrinbSZ~k=eNMWQ{&Z}A+3uu6>bm_k?HxpO`->JWoyR@fY<glJ=RDU~dHcc{<YE%{fGj)&?%5(4(YSbKO!x5a?P)w|&_g;K#dEJ{GTocbxqM<O+@)7vibv5K{MzEu?4+9{(<3Z3W+3;nNsGUjNBpsu{|NX%@i~8dCz~9+OXt~0lkOlMb%AL*nE9x0y0jn7JA+YwZSAYK?!I~Z^*cd32oE<m!?m|=(4*O1ccV1wCcWWJ+GJ6((Q2Z|-PgbP`n}GLU%PeZ-jcH6+P%APy?L*5>&{nE>Rz&l*4EarCxXSiSF3NY1^6$C(7B04(dJ}&R5Mr#)a=!paXKAL6JScMuGRbTJ_cE%CGw#tQa00Jck9~oQlL4Ej<tfRk12^dRJKM~ry!2v`H<00UN)z*Xi^K4y|7MGJdi>qZ80!yV9*FcX*(=yw>zJX<6dX)WFDp5N+WQ_Krb*&Sr_bET5BR1<AZ*j)Z|Cnrul3HpkV2|Gd<v+Qe_^EX8^=CX;hs_cN{UuCOrku1%t48YBxcX5FzdZX03kKoR4Q=2BvwG#GnPzWA1DADT!vIZZFcl$_S-6?Qk96bnk5)UdiC8T(iFz&(a#oVJwrBh;KLT#c}&f-BF715s{wE+gtT!G@*T23m5aj`U_#*G&!1f`&o-VK5zq6AgwtoOXKbk(X25@OhG#jlq!z2m)LA+^%%miNtB@Gj3p@>NgdN3-k*d4a5<o}N8nbCYB&YnfSmZ_sTp#^pPE#NR}>&aVBBbW6eSkhCbVfr@z<gnja32BBXGkZa?Pw<c;`zu!-kYmkY<s?`w2SV;30T#%{v!*5;fDsUM&fC?$3k!i&m@K+I@}+1`56I4$79kKRLb9I1_-jF`3=tW;##eS*^~~cTCeqaGEk1+<|)Cl+p3e&`WHU8jfqTe$&O=>@a%U#Bb?JJP2y^aU%$ElEwq<+9s_i#>yKkHROu>-HmY)e6>4TM7KbSlUn%tVh%zbbVr0*Cqx>8ZeRetgb^<2=-^514X0E5=mwMMC|F-dqfMQ>bh=1-k=H+YrG#KWd-|Q`8$$Q#EUCHQF@)Kmnf8V_)jM)T5N@QwGwom}jQcd7z?yKEdI6>e^r1OFnMElGY$pa20rsJmhcJkvQNKpD>s)y0EZjPtff0&uzzDP;?t_2o2ed5UxX?H;ON;R&HA9a3sSHZCWOvVwsbPw?x$nDm`NxbD6`PaAXaxQUtQL)E)@$-JNHLE6NpD_5nR*i})a(RkgGm^Hp3}Q!LwGGtQ{@i(3*c4{$C6o?+ppMub4%29<@9RudJrd|#T6f)4c@x@YS3NGhf@&AlORrmbTJsjy%_Y3(R!9lahRa_mx4oVNIDXO0S0*z>_zyFAJM!wB#JGqp~_wtZ)i7Q;Xrwl4!;0#%4Uz(+!-`gqoqJ4F5Hx&wH8)*ipd(qbYf-Yb;b!}V$2N~s|j{KAT(%U>js;!i^l{G`}jA9^Z6|O{PxDi&Tnq;#?J_u`(I)D2jT4rkK9Z-W^6k+MeS#jGo5<#A2>pA@<hs#ks$*cXzEIRXKU9|Lbryj%Z^ms4W4TUoB2Vei*e0e#O5J&4x7IoTn{!|tyXamZt)^)z?6*)@*2F%@C%SJfms<evkX7&{B%hy2&js|$!CLG?<~5bbvhJ4iAK{Ni!H*}BoS?kB;H%_JHi1fm&(@xND-Yzf>-Y>An{;gNor{l(|8b_q}Y}Twzl@Z-UwRK>d`Q!eqdZ&&6_YgDj)fD<?uhQQouBRYp3#he%+X&tlFOW&xdK3C&5;4t9;SC%l|0hjiSCeq`<BpPO-`~C7C=|qkW=;JBlVnTCi+)*7Gyc0|VAD+Y(Y%ZOPVb`1^L#`<UBi7vCi`nCd+;gM7*dyqOeJ)GIZL$1xqW-mK@BnaDALqm|<d#Z{xv&F&10c)xZ^^ncq<6_d5?fXPrOE$~-*IP*sYxxWM>Soo)*fSeKn;Q#wLe4NqOi**KCt+L0a*E?Kp7w|9AZ^0(n+U-DhTrH8qfo7h#MmzNu_cNA~=d%%D!?s7=@m{|hB-=rvC#XQ5cau3$?kR}~;QYJudBQl4{>1$be+Poxf%pYTWd)`)^WCyeT#HULeocpMjax8pmXWkK!8F7JoN-f7qV@H-zb?>dpX*ORszt}W(ISlxqq=1t27zstP9SP2YbcFI!i-e$7dUFMC2>O5lsS629X13a25w$n4`s6IyWTd-n#r%$>E+9byR^*6C*~MF%ht+taQS*Un?;)9*{z^-We4gj%#9_$6=q9M9Y|+P^htzNzHNP-&!AHa{;cVoy5?Lsm9cFHtsFTQLVe~}Xf9^N$kYsV#OQQxcAdG94%H2lzN(}vVwy7U0)*N25v`iQlv64bOHE<Mf{w6UQVDfVZQnzh8Y~|uw^3+;;T(5l#%Lpqd3nKh;=$uJ^>g=FKX;G)&siIm#RO@f##*S#$1<L2o$O)DZ<WcwH+gUX2@^sLO2;qhawiQlMY%G(OaknzYPx)($eXAh_7;$*)14DtBgGD(Kjtfq=uLRz_OC6H#XV?Uk2j2-h;|kfmxFkpO7y15Y?0DCKB4Fpt7$`LgtA`v1TIFmFgA5M9UVqE^cr?SRr|8-=y+5}a)Hq_fRCn@uvBJ)gry{f&yOICh`ExPvn1H<prM_Bri6++%hcA;&5k8v{B1g;euLA%FUxiCE1?XE1&YY|{+EumY%4gkvka+S1i(DxC_js^on$dFv_GRfkh2ft=_2(cS6jaB=*{qo=<&|3$NiA{)6xK+piM_izfA)h)DK__HVd8ymrhYth~*iN^-`J80vIJSd?pl3AT2^M>CsVc1TAN}B|xd?p-VC808wSz8)E6rM-jQrSN^qvBoy9dmBH;-Hb`h3$0^HJ&9I^mq{4%)vU}7ar4(r_ad>XETEX?USAqmHLen{Kw-&KpSIUGMK=S9^jK(&xTd1=)h32EpHiBk9BGol1{cETpr>`i@>ZM$j>PZO%pKoq+isifI%vROR(FkZqRL-4n$365IdZeV=lhzxyX|}6Coj_WTa5sBDh$o-}a+(+CP3BsUJb*v12oV)11Gs3h>|oU0?~v$*N?;<mX0a4#3>Fsyk<GutQb~;qYLz`9fh{kPKxQXIduWjx3iZ4Z3l->)T-hdCZjj8Q=Fr?F(;m==&y_ZxDn-D>W6GfpwCzd%Wmc}NU(^#LU$~n&D_hnC5w^m4XesPsqK8!Jz(O6qocHoWY3>Y4vbn29KCL<ks+a`yP}?TG<{4fMaM}!Y4cFFsqi&i8Y@3n2a_hJk%}61jgpstS9UNfsywiawG#W5J3}Qi+%sFg@ZK#Aj78~k9E<g-0KV%;lGm<Tv7M?&L(kLaf9({?_T2WHj2aJX~FzTMPH=E#*a4y0A09?J&2riR;9->3qv2W>kY7~iu7Q?=tU{X6|V*=pz<~0JTA4N>iFIQUlQV9pj>*S=fx99`s+b^~pAOj4Ol>hVjh+*VsjEBQdJSKfM-8ZKwEuoG&8?g^4H*nOnk90ERzN#!H>`d;XD{2tItChj88?Go8jcaKiJpENUfzDSMFQBm6BbE4Un#OZ*M@E6{(rmXiVI5hx&H*F*FX>jJ^z6v~3(Jv%wrJ^xRpGh#lmT<!t_~?fHy2X&c2!6YdS-x}muDVx2GO(;u%ba_*4!eHUTd6|=!+d)F;|Dyij_U8E-rqqVbO_VO}q2DEk^zUuulw#9%xDaCOrao(^^Tbe2QV5Sws_P%p)0$tHLy0C7@cZ0u#y>wH%hWSl$<B_DLgER=il!Z)1PaWsWto>b?qph+3{EPzoQU+V%u|4>pM36Ssi8qFr{@aQ<I9HmDHd<b=x>TI`Oze(@VJ(iv(`ZFYcbO6`+qpH$T+tQK+biZ#{QSaVdym=dKfB&oUTYHN(a`Dt;QPr+v32;|nC=2UtB56NOsHW(JrxJsDA`m9V-OR%a`JTjVZYCH9F>NMvG@U#<64#{W2_)nZyX^7!Ln`OAH)9@|GsOvZO!P8-vzjXt>zR{!8o~3In{l;)U9yP%Hum%DkHfH_7OYgK>%@-S&H!kzjweYN-cY7$dEmQdg_PFUmt{hJLc5;m!tL9sP#LGz3(wRNmoqH)E#>hxdoj?Q_QWUU-DXeFvOfK!(!}S#ytR(kp2A<<+TEHNt8V++5#$sz>Ucanw>_lUF#!N@>0=R&ySFhAB0ve-1Wuk?B+QJ0YY3|wJ2A1d~m`{yOjL24WNWN`GGJv}qy%6(Yz-zMsZt{@qhhS}IFrd%XwqwXTQEevZ>@-EfAvtNqlfkr>o6CkbTvV-LfW;MRx6PsmgYiPEUSUHt?qJP$*3rirb!X@5?oMl$+<iLxqv>9ER7Kg{iCj$5?jSO7!bTnwoPDkvu`{JJM@S88;aQV!_)A&F%i<_UOrX5zV|PJe6t#1z58ezOj%T0fh-n)qE3uan_3oq)m?rbliLm8?TH^f5L=uUmHZ<FkM@zv0-NQ-KL&N6zlSj3h-QGJ3;EO|mqLD6&_uUMg6ql`R?7=!uN^?9#jVb&`Kt<0*bgaq@$PEW6g+l0lS5_jBvkkK5unUR@dT`XmVGNU;qgJM#IokoW-EWerV5YRM?!m?y51J=dbd+>HkDcxmTIC<@Hg4nP_Oh8+1y?UN7kejW3VPt1`qQIHPJzK)NJmjLt5FAYAM^8MZ9jAt=Q>{^8Ov75xF7RG`~QUoR7>S*oRS%=P}`8}Df{7i)5<+8t-z`-Q>UEo`;2D3L{C&PC(zzUHPZeo2b*<pBiItFoaLNz)k~Bamc20yNu4p635VH|IVkLb5p~C^v>~Jak<}Na#_tSPA2R{vlhzw9CPY-`umg-(KZ`9>T5`Y+Oen3ENCqmVZPl06p1*eO%JcQ=)saS?6L%bjRj({d(kK^yd6~Vndflfu70HZ6mb^ju8oAKShusN`U<B<f$kdd);h>)kmv<(&aSX<ye^;;Z-l(J@g{lrcsb6$lMSQOAwk)pEaxIzgWfX8(?|)?*F;JG>B3i{LjfzTH;D*seTl|y)9ddYVWc)a`&Z5UI_|2QJRozC(i3o$Q#H?-A+6tE3E{w0I%xQUV47Jun$<_1lKZGW&$`w#;8gZ0Yw_a=n*IHMWV24l_XQW07U}gVZ5+K)a46r+Cnp;bdr1ZF0GAi>^<#iRztKmniXmn)q<*l9DouyqW4PnqOHt89YJf77oPr#p-blw?2EgL~2xCcd<KE4h@gPAzupY`+mP9jJ-E;|g&4lysgCFeCfiOpy$jw+GV%J#*Y@4E(D#;A-_DC>qr=Q0MXQP#^P8z7tR(oQf#nn3-Ho$2cepk)!AgHaAA@qRpUJxu5;NmgVL;g95by&;vF^W%BA<}pMphgPygmL#@q>E|l7I@y>=5`&VTz33z|AUu`MoNb?4tm{mmIXerD4|u)4v%UFzX0Cz#0KV_+5_}p;ZAkC(U_FWKaPfwl&70y)U&v0ZkL9Yu(2z!yBo-i4?WuSQsCxnrm;7k(oB4}O-!q#+m4H=_XcM5%B03qhBPiyCm{?bzAdpa$A$bC-w#|G7duDAm$`#Jf1)D`d^J3?fnP;Os;Q_S6yQ2T%eipez9cSkNo?h+H#e&W<R?;0c2gDT2s5tcw*2=oKlUtD8OyR#))r3E>-e70bSuM=Q;Q4ep+6Ns!BR{;_DR?SU90X&gIzkw!8)PmfzX&?9ef<P@kO<q^cgB?{IHT?)1U_GY5x3P@h7?tdCfaA%r@BeBwG5}p<3gSor=_3p?wGO1){G09J#<#lgjp54ZFUWp#m$!=D0|U9`(vIfgPnU0R<S*@OI&4#9sOo&>uPYB|7o0iMxFbg5lTw*<&MV0GP#uv>Hu6_={dZaVaclb4O#%RogJr#wwW~7y5IJ=rknC*olbXRo5K2vQ)PKu1=l`3*bWZGhyoJ|A+2dhZ6-vi4vZ5nzCZ1f)Qq#j`dmA0oWDauvDdqX4WuoKr~287!y>FwAB?u^rcqzQ_8%D|rV{@Ryn?cJ!yJY}KC)wvoO!}ib2^=Mx~IB1&pOE|w&PiI|7&4odtOkMv9OXO1UV(JJBoEi5f_g!VQITnKL>wY7YaW96Lw}HH$GU6j;mNkWfiS$o#b!4+g=S><?NxhPzU_LwN`7lp)dig!xr;4+i0l(%<H7)>~!kOrLb~ozKZ?JO#`^L896Pa5SJ%#T-ar*y#S#r5r~<<k%ReCs%eIs+m_b3rfq4Kw^Ox4Ip3VMi{i0@abB^9kFIvRu==ZrYgby}P_DGL=+71U^Y7`;3*w69QJ7cF=3F8uWXGdM8fQdYVeMpac9%&1W$ta&tVXrFe1$yk^L8Z68>UH*ehCF2{Ehmh3=w8g9o2L)D_qi?@IqQQb}`H1CYjE0Ok!BMSmAXtd4(o+5WF`0Wo-}y2RvSX298*aRTZU{?G?ihI6M{Y6l?6^)-R$gDNOE{@4fbFKz4JWKrl5f9xT(;D?$;7B}XT6;pa8WbS<kVA@8*;(2~yaw36=G37`r*bb2VkubiHvbrnXl%~pN;S2u*6k}Xo~2R884N_|V^`VPda@>D>bg%{g8fElZPQhDhw-@ErlV4>A7vOgWh-y0Y-_RuCVieJm^wdY%{|Chtfka_j-ii*V@oJf{M*``9iDHUxlEDKc#XRDs}Mk>Zf<1^=b$>2l7nX<b2omYv^<*qgiq-ObORju0IK~wdnK(C*VLbaTq4#T#fElUE=hBsdopywaNbQsnuVdlGAUy&mw((Sr78>@VojJC7BxqB`J%i(;H-72Rh%T|-k_SxE#y}YOmuOL0e9Z=j)rOJfm2UCU8V(lCVmK)AE)G|YV)?G)P({7h)akf|3X*q+s<%gCT)wPhn+<V>L!E`{;^jpE@%R$ZbZ5^5<sFO4r6d4jI>j+tun_JiFRYqq8_Dh<cpUT4QVmB=$^rKjxNEi2}YKS+?aD@j;9V=y0RB(kC(ntBb%JhQ<Qz9$S>w;WRX~fP83KpB%_6cH<FPO9|g%VZWQuEc)Laz%wRc9E_)o_~ke6spY?V4%;9S;-cNTgWiVdA<yReC(BqMDA^)0z6zoFRZ7KPJ%};=FUv8ZU_@N~Fw~?0b*+UVa7DxzW6L6v0boQX5|@Ok6)xxBHHw<p#v-CGLZ~ZH#(bJU=IcdKqnz7Ax!)0&_<I?PhK?ndLwSzE;jN7FIZDmjNz2iKV!BXnC8HHRcr|EIGnvCZK|@&8%-zxroFrq}K3KbQ}F;C{?xRr!(6nb|BYVwVaZe4$2-x`7&h#b^gjl7~&EXBTmoEa5<0XgnT&4Bt38`vZPj}#e$hhlg4ByRA2jS5X`X!arS`>LPoY^S2VIMoFk0(Jxyb!8>&z-r#<tymPhn+oWGQ9v6P&ZABG{1gbU5P+p5&o>rOhX4y-we#>&!&RLfOy%I+{MTY02bjF4^H6xHqUOSduJYb_^io%g8LG1_%{>s4}x!&QhB(1WRx<8(}4XuqnBluPTnu@cj%TsV@nj$I*p6v^Z{UXB#v<~o{q3a0GsP_jkjA~ZXh#pQ+E0+gC4%utL2d4}BPbS!1eRF=wCvzq5|au;}>**@}8VAfd)nJ;)Bmv+aXx`)4NhOHAPOJ}v~1S<!&yfLSCg(8bqW5>{x-Lg$qt-dW*mdCZcsjSk5y2N;TaqHPKLz(I7sur?JgV$3TyVTTWEM9_^^|d9MZdZuhEylFvy+KxxvzKcBjefmEs=AVwsvL~W3sconR5h6de!>U-pkwH<FVDewk@JHd_tvT`f6BG92&EmUa)_!>LFd(PqNO6ArxyL$V(%yVb5*2gGwkcDjQZK&)hT#nv36N^88D;p0W^XVIn{TOAr_V~4iXik^<vUog-v^U`O_W$1z0gS89sk8uUB$`V*NWrBr6=U!<}^uvr-}F%@sLRCBgcyaOsOeB~yxJ7TXfjQ0D2%=hkOt)&1oyxg{o9oL2j6n(e0Zbz~!s2cq}QY_;{Xf~@nI+m~#t_|ut{c%HM2OhhES^I<;4Piccp%|b}Gbr>ZlamIy6rB%~2F!STNvz7Ups#gbZfzalTosKp4=Lkpk#N@?93h1u@sZEd#TbgKQ_c#<|kND?q5=W9SjYI;t5ksdC_eRk=vXPCuARWVKk#NpoPAILIgb=ihp<-wpQWtU<Q$Idr6o@&dX_NnM@!u=__bUIr#($ruzjL+W_l$L)AyNjHC_Ky9ze8vu=g{BwU6$|JV01jD6pUjUcZ$%qLkhYD<1}J59T4=T;C0GK7}31<V``)8R3O?(OuCL+GH|X{=Ln?sn}8~b0?PvSvbc`!z8-(3QoWF!jQqM#jCWVsL#EF;&&B*?#2YLgBoK<}X)<*_AM&mvRbIHf3mJG_C2V)@7L;jwNtmxo3u|^lmH^dG1Pubyv#gdaRA2M`>to8%JYgVyeOXp6P#_ijwn-uOC9x=8yIINdxt%jvqBJ5)#~eHe!g}TGoj+Y`*@(<mbhE-bS!vblT7cLLxzKA4p~>vb7JDXyk-dFu2#JWYU455qrR1|_XpLfTm`cV4C43F!V<k?3Af&?m!2OgG8&RSj+WB@<D*)QY)Kv3$#;z_K8<%&!5ck`c4CRNH$o$f7maL2d9!y=%a<W;-DNP!QHQy)7)Ie;UX{l+On!_d=t@AkYEg2uFa~e{UdKl;=n_~d2QCJ64N)^*(YB;+u4&y$^26BDg*?;uzN$}x!KK#aH|9*OT8~?A}@1Jfqw$3)3f3Kd^YoCASe!uP|0M@QD;B!0biU#?l;UF9{cH?GHJLEDGey+dqx$(y5egm0n87=%o;^%m!yRG}^rt8t6)2Dcb+>ORyBLj|&PsSX>z(_<%>~KOH#l%{rnl;mLm_)W{*3r;2lU0Lkn)QEnVsrW9l%n4*Dsm8&y!Q9ffa7EP9D&4%>$C!|Z9htT*pA%^WkxDSKCei~3nw&tC`BUn&udbb`4avhF}svmS|snftJ9GE25h9Qvn~lkIvu1viPkAg+CHXmU3V8dXlPSK%By7CY;``#LtX;6UQY4NH$|Xz24T0FY>18F)g5XX><{JEouE6lOTnc^MyA%&8D+%&0>g#^5MTK(gT?c1j-iH`=b*B+?o&3#^ne=M`2qso2l+O$8KzteHSU7SR+E#Or2*V!cl(>f+ThPQsS8i;z1w$!S8qT1)}7$j-@0)pynyo}VWmt9p(RhduQ`)$`pd)f)K8E^jfgJ#VECB>V1CLJp7YDpT<!!-krI%16$;4qt2f<qMvI%90j539&m*CiX0!8=;;J9bxsvU3ExGaQ|L#VhYsY#DXOE)?apE$VD{gWXJpCwP5)SA+pV*sQt!{ciZjN+v9~~Vvoj1;YWhDh)R8nztA#Gs_QR&idSkTPcy8f~(1Nn3~yoap$2UIn;X}Xj2NRGtMter_pRXZh(>B(eAmp`4#PAGReh$0(7!vdl3_S|r|1dVCO7H6*Aque_3-O9R(>WSdIt)T^oHN(Sh)ry`!DkdvSkOm9(<xLN16w0aif^ECp9Ow2ILbGGKfRKB)%W%*risvEsLXzdmOi8zBR?T|J8>Ua2>OT{fK+=qFP<$a7hHRD2J6+oJokJGFG?|y_UI%h-V)AX7mt|eldO8*pH^YpdGrT1=*(KGqMp>?TRKze<u5Z11fYf!#AvR@zJK?js!(mxy+KomdZu;8+85N6UrZ+i&8rdZ?&Krl!U=VRNx`5@lhyHo!vK+yNoBZ9YVVAHG*bx%vAU(2Nnq5*1T^kMy8$tU^=b(ST249&XkuN0nqf0O=MH_j+NEnr3_aWUsC2i7NX+^HVaLm$T8C9y<!`{MZeNU$NWqaEz>8@SZvId8nzf^nO>gh3LfM&;}h$*2Ot52N8HV4HB6%2mmZK)O?cSyz4)a86R$bR%x{gZtgZt8Q_$50xOEZnMxP>EC^0K5a}sbqtT%LkQOyr-5EOww_~-c$bGWa={ZfeuaJ(d8w~vA%sUt8l=k(0xyGn2F5bAIdH%sD+>Ww|{=}&42v(TYvqFum92i`IrAn>#48)^mjk`!MpUTUN7^r%yY79IrnzksE)6|ij^sridc|8g6W^euj&+a$Z_dWW{<Je4=ejzeOkt2+<C_bn=w^ptDYu3&V1Y*&As>ZyCS5oD5Y2q%-I@1sPTSS*8{SsD1bg{Ofk6CqQhH!|IZt*-F{^~6CiF)_OT0xL3c0Z>1bF@^Bar3*HB>6J&gAyQ&&R99I_T?N-4cOjC3=&F~;1)vQgXFTz?UHmGI|sedF_CqvV9M0)=GKnhz_~AV#@tQb*S~!BSROt|K7jcE;^##y&3<igQQ~a7q%moi7lq_N7z$=j_t%#utp3?<|$TZ-VajkSGgk!2-KUmb4nLk(8}ll-A!KO3R35V-C}{g9>}G-1pd_FPTn_%RQsFs_jErGpo6_^MbmHCGk?YvID(Y=U3ZBcIb48RN(U(PkopUCCSFRRUNofE*&YIax=he&XsH|B_m4)ou5Io5tcBL`|GAqLC<O`pQlzd1wFcJD%<l?du%>bgX>S@@<qpo&|K)3sMJQ|r$7jz1i0piFzhV5;tDmqgn4tD#AOh^^7`#<LXP;%1qz@1Iep&=Z)iuc-%X!)J?;9bJqNTG%LO$CpBKt}G0KoVZO##<SV5oKyaZ*F19gua_{H44q1>2NUQxcKm36gKypy+i3peabT9IvDFjLnn-+?tiY_!|#Tc3kmb;z0VJ4yHLl$4iSTTOCfGRaiZBT&0E!zJH%S^49$?mi*8&Rm-~*++*K9oV85IH#1B6|QsTD^{6pp)88T{>t5+9Sb0&j2N)osaqsHa_}!M)1`X}I&~)lwwvnW1!r@0vm}qWr?n}4N?rWV?#9gEO3rff&a|CLCvXWL;tJr|iIe98_nV-$Kd2_X<vip9&)DH>ChMU&8R<XiP!Wc_7bpa;Mt__nfZ14heK?H@|MvRE&dEr39O33)vmWw~w`mK|%(bHF0hAa|y^vb{U&AtLSN<(7qDqziBIcJ$3(-<te3^d9RmyIXH4?D9O!a>&>*U`S;d&goTz>4_;K~A}qbc)JeqxFGs$8njrfGk2_|f$H!9KojXY8KjU};Bi-3veW#w%Z1SNS@Z|5t=)2TU>U+1^4uhO^edk2ukna&@w%CY1isqO(_4NUg9}Bp!Src1FS+F-92=La+FGV&==}iY;-l6Hb`E@~vn6!ETv2oH04nDSP7$i-F37i<04d_Gdpz>lH3zIW|b9fcYNO@^rughJ3-`F?Z6Lsqkvlyfe9uMTW-X;__Az8%p@P?%c9f5|1Sl=dJ~50d}^xyjXWFXQJJM^D1nwElt4-vmoDo*6|WElp*A^!5b{RvN;YR{G%@UVKQSB;i6o7YdhSIpbQf?C%VFgQUO9{#y_Lr*S@%QWphhbA9!^0B#CkBdQYr+3Q8{_=F7+-2kTonC$#bZo%@&Whr6Gz?c6_F-+itg=C?6w-uC!@t-kI0$v;IaGO?S_UwPr`uFYS-S46Y_(ms$c3hQqB=6-54q`<A{nRy!yY<F15va$&IuABO-zu(`ZiKDjm_YUZv<8^v)R@Z3{Wj0@EU1`YyOb7x!+iYR-FQ_(v#(<ELL{qtDA=ysj+tt~73t-K(n7ZuXl_227Q~itg^;>d>^($OM6Y;$|ogOS^Woh^deYF9zq)W_*^$G^nwR}W#Uh85^L>sJldG3A_<1AOb6zP19nOm98=SDCY0{mjyGmJ6yym>e=!>RgQ>`y^XDE6pce&mTT)tvP9;^t&DZcbt-PNs*BCmq|p*@&;F3gJUFwy1HQ;&6N&8&%_*YUQTsT838jT<}j>9!IVnv5G-bOO&Bc*0!f)jjhW-X>ofKlv28my>LE5a33+-{?B^Zxu(E<4^ylO4q5owCSAKi**x$xv>AJ*d{=o#JfYg5USP!BojD@gMS-{UKfda&Plud%h)UpW!@lPWH_eq&NdIE4J9GnV<qj^s_J|m>)VTSHBM1+U$(mj)A-!i0T64H(XEtDYr+dBZV9yHCm*q+gXHrWv85WZ0FB*=n!hX^>`Bn-DK|1t|=J`lK@VpmiPrKoe&u<beCYjcmocGDuhGX>iQWC@RifFkuh3%x2E;(Y)XTC}zb9>3WfMtEm9iJVJhDs2BIvP<NItRJ9g!dwJw-KytfnH8nvb|j4?z1{QsoMX_I2bbEB4t@xLoT(WC3fX1H=ag)<6nx%-wbKd%5GBsd^qe=uL-+8i{B$55*-k$6Yq$fma1p6fHL0hqE0RM@GZI<?7*Xpm}hY;V}dRQ8*;N}f?}gG5F3Ck&Mvwv^T2I#neovv$daw`qK)Cufq&`+8}co`?Skbuv+|a@rj}TIS+<q{&uSZ^9VGF9jtu0)P1K!%%Y5Xl{$j_JYk>#m$}m7gEqdvSqx?a0-AJM5v{}v{oJ)%6qdv9M&Ir5M7c*02+u*L~qPN0jCt5+D+osv44gjPvH&XwcuhRErP|LKrZSe`nBZs;Jlp*Nu3LO_yXxkb>4Vw98Q1e!dpw2Ru88|;HlA>ITR55;*Wj9+&6Xt$yH&jV>hj*oFy;1y7xSP}H6UFBeDUHn4!R}2#wEeu$tqXAfPa_6s7li+JyNRfG3iZPfZ2CNO*&D-}mq<zP@%$_X!$~^KwV<T_YkU3Fh17^Dy4f;1;`#Jvn4qwstZG1R#uG^}RRkcyn7`O$#q(ZQUc*Xo-GT5V(9k#ZT3lyX*}YO)fc{&vUdY~PUBO7Ypm(=+2$!AE<}g<=%=tF^m9t=D@>f^3st{~cRflpoaT~g=-0NhwGee^Cq8V%J>Lfdd&$Bj_sAO0W|LEpCW@mTsuxqXalu66=2UF-z2{CLSmZ}3&Ufd+yRK1pXr5Ix}wWH|+8$phn=cvs!b1iO~1=Z1cX8lV<3G_DKo&>#Gcz<!_YO57e&buD{!HX9M11LH<)XHReH2BRKB*z+&NFLhUb%KIAeNJIZsTvT8so7(o)trk{GPTxgQr3ytuv@F=;1?@WDn%MYYMkr2Mrvw!2^S4BV9Rns^DT`KN4IeUI|187*f5vFtZ~)4(=dH7F9{xZfY05Qs^;G0sJUQB*M$YRIsD`$sOjR=+y=s-g7IF$Ap|2}&Vn>>%*T+SG-(=yqv?0X3*WZpR)$!%CKlss!w$73Iwnq`mg&>Noe#h9=&wHf&ZF-?`pHM{ee~lGfBVsIfB1)wzV+yjKYI70_jgSoQ90nl*aPeyUv4poJ~+LQ6wOPohbw*JF|I}Dlb)t^tHm>BD>^-w8N8xd6iJm-y4`rcEH#nSmq=0|gQuqVb#*~gaHY2CqAtpoFSkB(a5<3Yg_ai?dIMKFPZLhJOLq)IAT$KoNP$2CPGkC#2Fl?YH{2p8djHHH3En)DMJ_P3FrC?CRg$KhQ}$Nqk4+B^qL(SsF*7Ia4(CnHA5PJ$lzTqqyedQ1JCpc|D1L-z#64QLb?O(|=8w=g43R3JVJ|L}c?l@2gVpg%hGj9J98tUhoHO2>M`_u*=6uu5x^A`3#|YOu-PGGGE$X;8n1WaHh?8NAohc<4NQ!|=2q`fFXIda(!4@tk4whia1a=Ycvm=DkQD9GA?m<WNI!io_{7SGcz>{@9@0R5r1|VRq)@Bs|)y<>}Iz^q9Gxd<K&0(2O52uzroQ8TjwXWLJX;^xEP;V;xVsuDE(<BJm=R$aEOgYF<6+ehvKcEwAy2rUPIP#c`GwFDGfWLg@IFSV)3*FGCK8Z-B5^)kbH=Id`r@{ks7=HXuzyIV1fAHj|KYQ|>Z$0_Z_kR8_?-H?l^38wx`NKc`_*;MS>HEL`$shjCCx86+{PqtXe)7kE`Q*?4_LC2O_T-!2{`4n5eez%a_S5(OioSjP-T(aYcmML!Z@>TKkG{@+w*TuppZws@p1l7rpZ?zWi&9<q^Pm3s$+!RX$)El0lOKJTLI3w}Kl#0X`s5$J`{_@9^5l2E`Q%4GrMDmcA;qi)li}kJemFw}>*I&t2=*TR-2^viF7S#~<u`|qeux+8qxY~FdyhZ(-ZU8Fo3qHYjxuDkSiE-GE1eJCJ6EcfZEp(uXu=Cp#3RFFTo0O8uT?>yo8P1$Wu=Oz9wIY);Ge>NZ7GA(g}H(f6m~6^L<y<@!SvagP(qQ;w3qm(Ck53-%>uUO1u09{C_@#jiV>5xxxA_#0?R6TyRckTtsF{K`K4#qXK>ChIk&Il*j`Tite@8k-7E`yDxAA@E7L7KMk;3!kd$U<g;jdy>20`QIZaM|rPB&`(O)}nz8saAFR0~#tFL+9EabuDGJO7YC^ZzMz><C7F}X?jTHJ#IZ916W27yhGmg&~<Yy?Aa@?e}!*EzcdKh5Rw`ZM=;&{jNnaGhVg{NOs!=w;=G)plGp<|!xlptFIf(qaaR(KQ$?$$@8-&nMw~tHa-{eSRCs*&iS$`UBy#0?AhX;BB%9bm3e>r{e>vGGRl#4!ZvU`IZhyCu?q8k0xXZ!h2%I=-4>Hq#_H@^ay+F!7twXAHVpEzxw$<fBVS~zl#Gve}DYv-~agc|M!#M`|*Q0WiJa3hL3(i2iN0=e*{AN_=ET22j<nIzZ>Jsdi=p(p9EaC9u{ZX8=p#?Mx4@V6P*OmhI}84M#nA`^V}!Yb54XZT#UPuiUG-RgSWefT{(mqm`=)rUwr=$K7Ie|pS=6^2fgkLbfD`mUghe^2=G5t8G)<XH{6OzDeWi_tO0^WWz8h6s)Af332Ti?VAEFQ>f3967B*2kyp1!;B#XBGHaU&Jw$0m?(+q42WM4r5BVEo)W*BMVptr3AmfiM12&f0%w#%|nQXTg++%~6Zt_APla{<wjnbTNo8=pI7cGR^_@1LmGjbE+glZ1^J7Zl+B8M%@zVY92Qtf+Iygn9WgmkBjJv43w{{x&u{ry-#OF_NM|S$d<CXG7PVdD^9><cLHMRLR!Ro;hI^xB+F@O^4Yvc6{}Z_uX|C49z`1rXt*kb}3`31w<)QahH#ycXTiBvG?{c)KxF<kzU(G(WtH0yqH#<t6j{4bBm9B<qovf;&M_<OsaR37*(8~D6r(plA@m!S8>zrmBMmb7R$KrZDH!<Zomnaszv<?BHducCs%#wUO|7?Bw7<WZpxB0#RYE+aRjg<7BPK+s3pWw$H@&%>^Vt<#OXCX(pE446jZFPrc)=v(40cD&_bp~e{7mgE=$3$IcV)&OI}Wx>iZl@p~49c`GuJxcDxL?ZR7|o-X3s<;0?Su3-xxH`dSe(R?-+l+e#8vPIqqaU=qMiMq`uAn498k#)C_zOac%gv3thpi!b?vK#9A&a)}&lhx@70w9$0GBbWYViCO70=eN^OAtkJ9Stf|5#eS?Cr2(a6qWmBh;B_y2hqqPcy;IA;TV|sh(9WjI;s{u@4^pQDNAB%rQbnC>+Kaexl*a?QgSag>jJk7WzG-S_(|vYhChB6!iKxt_Mj;z4OwAqrX78F>Hl8=RXu#AGha*nnzUjWEHlEH*!VOW*`}2)g<ZCnNvJ;K!>+Ksl)KAy4<Ilf_0f&o!X(#Y(LY54@)(_t|uQglQcXG>gK3-|zV*$E*%>)X)*m5B_naLHvnMtZB$D5v3<Xk9EHNIbv*H7~d^RpQPQb}C)CrX#BPfY6kgk6XDUhpa1rux-eci+7I`W-r7MO|VK`)q-&4?KJ3ph?Pqo3&pIPj%)q*SBy%!XyYL^D}SVD5)$dj7<?Q|Mlg|=IqYaXAtj~0GXXY=9+Y}agsZ`{xZG>k|v9k4xAlN%9x*JFd_b`%kt@L+i}jfp4Hj_c`{ok=32(cP-`b$p!t!cGW30`)N#6gcIF1O>gE!zN`@;N(Z*%cX&xQJYdBTM=4=PE>5_J$S^Q-i@2139Y)B(!@4XZ(I9hiCdnZ!E26A9gE-xH~n%svnc_Hi9gks^%UUg`$@G_<gW-A}D^~dph<4ZCnHxggLj;Sl9EKS*T?22K$2@MzOS&hUxq|9PwE~6IEJZFVQFI=Z)$TG8O%+{}!4vIA||DdUDSHtIs((WPqTxH9XRVgMurNYt{t)Fd}Rj2ZenH>9(e{Gj%&-Rx&8>J-)Pyw^iz8RKxL=*w`KEO;TqAR35axM(4n1ge>M0aPZLfY+2(E&r3)d+`}14S&kz2+|egFEDb5<?g-;nb!z>;cA<30h_7>m1{DhA#TFv+WiX6w-vJjlh@pqBMJBY_vC7ahwqQQuerJV@K`{39sKkUb0&V%voTI)u>HMQcWp%xfA+B!becjk(3vFTP$HYeQ2gov{9-|BaBfp`n)ve?gSmIpY|R}4x{6S!sh#=CW~>zN&mY<d&QK2cwA{5<H~1KC+KYj&=hah+*Q9il{<{YhTYu_o-_8OEU}b7(^NswGADD3N13r_E`cP<1;fVzQURjK>PDejOu*swnRcSXqnorVDmP`oU1>)yS&*=oR#jpS-e<PD7<a7KoSM#MA=R<YXfQewK~j3_q`5V$Lho_zToku8C*iCyJ?}5ZvlQu(IlVgx>>y57EC@Li*%CD89(OB;uaKM~(*}5p(2#r3<aNn>uLQcF#HmEJ3xPA5pNgq%(7dh$P-fBrBrgmWWV3MwkqN;`;4V4HM!n9lp+}2!D1l^ooaQYbe9o0;+CeKY#DIQrsFu>wmC(QviVk^Gz>DzEWZ2#|BqpQ_Bq+S1Z$ibifeYY*i6s-t9OelIJ{G9;(=4C&`>bR(p*#a$NYV(KPMUWh@=ERrx-8tvU6jUGvV0uRAt-^dOjxu8lg0N_)iQ+{a-6*WCf{m9c~TD{*rlXcgp|bZh&h++T`^r|2AwuJcf;`>1{@A08=VAv{WY;^tj5MO?8&?a`2;E~9P;S^?nf?%;XAa#?4)9jlS3TC?sIz*%m`+&@0ys&jhbH}!87vYvXMWhA=2;6C?YDem7`MnnP=BIGxT*DMF+K(r=@baclIej+RH0jIae}u)gGsZy<~p=MJ`f~CxE@TwNG@}!IaK%{r>7#I(NSK>dmjv0a$jUi(ZCNc^8JTM0ddxK$KLb=!UA9H1s*(+Z)&&2^jJclw*7_u;vu(xu`@qU}Q`@c=7&<A`7K+G3ZTl*He{F1sB6|5NgUiH1yr8uiyMi=hlBjK5XaroiF~Yor%KmPU2|PUt4-Bcm>*B1obY4PjB5R#i!D%zRp>ix9-$!&WqjwMXBbsSK`EE2#+OEj%bwCP*53_GeABw<r1hA%}7~swMA+Px01++Ig#>Mc--U9Oql%trT02=+&zu&TZd#NNgkIayC&5U=QUtx{>_V}A}v*@lxb%bVqD^_&8)B=l1y^V;4^#Th^_KvWx4gPSdDJKvhmj4SC>hwNnTnlgMQQ@FJ9uyU9q(h=ufj?r59qE+Uh**uE%!OMK@|bVuKF3L#au}SbTihnQ>AqLh#3C5#R6DdAHy=^ESA2bDAWJ8H?6XT<OxBT`HFe+`%-v{eDdcTbN}LL|BO;z6dhF#JDdP?1MTC*Pyu4x>L$h*%ei5PTRxk0W_e%#OGnxBqHY;eUF181<BF#LEl|-B3P<HxkegK0+jk3t}YosSpm(L7gD;(Y|k=0ThHU65?AQ+XG;L@<fqiS;p;-{e2n6*<eJ<$&DnH^q|}}EI&q)*XU=0#!w(IQVa*+TWlMwZjXxX%)F;=jU~~7};h;k{0JdXV<V(M#O@&Vb&JT)Zqk3UN#ICPok?F6Y$pF9Lh+HMhjjTiUVrdYT*#an}D{t$V7bR7II?y!112D3sFM?>#c!*X4-_BGv0$oEXym!)}a1dXsDIQzMCNrHaMq-4dqd=xt&rFzGQBOP$C}=~M(~Wyit5IEA5LB1vOk+egc-x2`d|q2S`5HQi>^>z0zimenW<#MYCiVy40W=cj2DwcHnUJXhWRfnC9i3e&=|`$}u`>-zA!eZ~0&o=yvnp&ygk|z!w1;l5T~OvCt)nR39}4NBJ{|A4)47Xl0r4ioLZBUf+iBg+SSN#QzyYrW$%uBv2}_jbksfJ2OtY<xjyirWyCr0z?&?+9MCGh6hjcJP)yq~I4byw$72=iGfDkEDaFk`;(AM#>E}GTVoujGqw?+wE7rm(YY?uuXQ#cWxAu>fU>acV^Mrpxw`(?eBRspF+&jl*g%r0LEMF~&|q`a!dgEYtyB>FYrUzOUlvNk-Is$zH5DcAG6YGmr<dKv1LA^`in+T%{nHnZYN$cxnsF7U;=o!kzvh=N_`?uY^uv+X+`I6$%VK+kSo2cyz68-{N~fy!n%EvRR?c8VR~vzz@;kx`=`cPHuy$C-^_O5t}kr4ojdV2ZIC(AkUEWKO8g4quesi{eqO`QkMYHIDOA*tGvQcUNoyV%vg4HFeh+it<vXOH-DB44tHGBeQ3m@-1=fq$_iSs`_+R;eKLpRfOoKrXW%3X95-OMgZSm^kQ<Q)QD3!#2^h5V$<MKdMpAXE~Mee&|n5lS)lF&sfkjeJ3YC5@z>PbX3PaJ8pWE);@S9>?QoqX^@DF~u0SK#syanl4gXH%gbi#3Q=Mmg5*FZG1}1+S6M)6$qOPZ4VvI4@f+-PEsOyw+V>pfJzFk|i?7B!AyX9|ii>45m19eOh|A3h9n<GJ(+(L{Tn0Y)Fl-b@*nXc3??d7z5_-3qJV01Y!mps<m;^|^e(+LHVwL0@@XUlwnB6-5v5C20YpF;bZVx3_m+>2<Ns&_zN0Sf*^iT(si!Q;?PsVp3@6l>^Ua?zsUv%yy$fABN9yamxHqL<RUTfZ8AF`mA5`z{B{LIHMZJ-9i1{P5coNWBhbo(0e!yo4?D=v{WB{_DAZXEZ*0Dd;`=2b#&p5B~xmjCs;>7f_R2%1r^a^yPc^-Uz@+xK%87u?wj4<k1fiLvy@vTFj0RD)9K>-*XhspH0a50>x$$n`8QK>3?|-Za)6tZzjPp5Me~V;ciEiqER;{fFB3_1=`qq{O~_Pi&HGD2Y<y#?>&C_Khea)-}O5Q0z!HjgsXm!Cy%Gy0<AdMcDI{bpDS<X&hXJcj%n6@syD)!S<u51V>x{M!4Dvo#*ZJqFAEv;2q$j866`;I_?=z>Y{*c*=Vb30J$g60v=l2p{OHGBijX{EZ~C7F2Y~$z>IJ=h{P1sp8}PSp244}s`xy#-FUFXq7#njS&V(hyc>|z7dXMWodT*A^P)=1S7{<t;lZ<eyA1sXTKhOiK1(pS0DCY`LrU$=gbd^9(;gUm0t>WyZM)BH$@Imk&qCLLR<^iqbp8zN=+k@aWZs$Sp<!oP25f^Lm`-Yjh2eiu`1ncYT0sq}z{!f?*sV*1c1SQi3xB0y<JqS*ZGeUh9$ZuDAG4C>5GH){Zz`k;nGUin#_u$PXhstD0#5sBNvzP-+I$vM*=)G8pOZZ#A^;>#sJEl&H?mkPr+h8;2WZvGYOVzMZdZR?Z?cD_Bk-aq!Mj!q7@x$*;_E&2*myn^nS)&mu`kjZk$L`hJ_kN>u_tqP)-@Vry^SNXb;7IIzL9bWA;!t4GfS6!k!{2~p$FaxVg1^7&_D%URN5pcYL6MVVi>SKk<!k%Cc|}E0r8cNc=ToMqDdq0q)n|+wdHZ<bZf#vp?{00@7#aRDZ`Wtrvu}vDMjEmBx^@~Ed~YTc^r2pT>a-|!GI3JFsY=H+-KIdkQ0z4Xe|GX0(;Ow@8=%3jw_mYVWRtumZw%7Kg^dXvEwK?{jCd|FH<TFyj2YX+nCU<?HJ<fQG<xDv`;&Hk760a}=x1f>5rAp6#fRo%e1MVEBsX~K_!saU7?kZsE`oNu)TpqKIT1`l&LmBDBGLD3Y?6mswg}&A;!T}lAf-37j2t$gsbsQ*+l>t6JkNdmw&Qsjw@ZACs|ZS-j+oze!JKsx@8Gb^3(=VXzI~`rsbTG83Pcs?lvId=mQ=C6of)&NkWq7NkExT;q!NWzHa{9?aL)QHS$$M2az~(+y?S$hG~KI(mv#B;`REPbeCOpJC&x$`t7zgi5QR+%Y{GHV5|xGgl-TSrp($vGB7CzfA<%N0omx80KusSqn_2#rHx9$d+1kjW<yqF&%58KU|KtvJx#g|IomCBP+Fxone=VFz?WN|w<>Rs0oM`WrkOeYs0L?qogA$*v{Dx=JE)WY7;Kg_qb)XHa04>f!smsV`8pKsh2vr);Rb8w+xT`GB)RL&H64UNQvvqNl?b6h36^Xkl^1xMO@>*gee(L;PWybKQ$mexlhsyGM6*CK-7yed_3Fn5w<sG}*nN(bi(eqC{?lqr!+0_8c!q3>5P7Yr`=xmu2q46K&+sMGAnFCI8HJ}5w-i>6R`CCcsT%&6Z>HG!1*Y2<pwqq<Wdt|=jQe^i>7LGmnl~N_-Nv0gfz{0R$qud>8Gh-9e9@ms=$O3^ebQ2S!J>zd(?M^LRUl;Ni+R&^vr4jCK@TJ@xWx@dfu`}~Z7!-C@-l1Z1NRdf_=5z@|OaaoSS&Bfctc0k}(n>~KS*#M%UH6cFylTT}G;4>qDJ^tLae2r-oX#`$Ia~^CIxB&N08NEup#*$8IgUR>MOH#gB3KgF!N7$E(p`FE9Y3&)oaz9H9QoLlECB0D29Z0tPo8*-gm0`NNzk?gN2HEUPl}L$Nw2LKP>mw2zWXc|R|6x;*SB)1B$iF<z<sXk)@oKFg^g}2Ywb$q#4|g&ys{`pWm#U4cjO`s=SO53a~eLU4UJk{H-U*2BfM5UL<L0b(m7BlV)0xk;11R`{;?d&i|0d8%f>ndN?yDx2&GwvjOfgIHQKsflGV5?6#8k9x7eb<TH2NzG3O9d-PbE^nTvK%I-t@}!m&+yT<|{xJFQt^EEfZv?axh4OG7+Ru87%;T%c7_z}Yzh#Je4kIuZ&0t&MG%3N|p>ghjs;aAV+^uutep(L+PUSSdjxw!W?yi-}rimZsC;tVkVYX~^?8o%cyzr{kL<CxN6zVQ$Ujp03r`{x4ME(pd',
 'diagnostics_snapshot.py': 'c-qZeTaP2hb$*{;Q63>^_KL&V*?4V%a{<=7BY0z7R=cYR@nq0yHj89yn%#6?W`>#&NT3HJ4*{}BjKDC2NEQS|c8u6jf*`vOc^KLMWAaP#ol{la)!ig#Bnv_Y*k!XWr%s)7>fGuyH#ax$r_#@*@I^NDqo^yA(I^UIDTbjxj*~181M%aBkHjcVrWh2Y{#hi_BJPSX4x(Zh#$z$_(>(OKIqPk1Zf=CrS(4_$PsixW<OWX;k|>fv9wu=n^nm+G5$7`PilH3&MU)T2AUBhP>{QJg0=+z(N;AoP?+W_={UnxV$|Rpg<~dBtC;o`SL2oK^AG3Ya^In$3YK@tnPcXtP@d#hdpjqVSqa>Z0@7X*vkF>sdem|Uz!bomxY+x?}G#z>bzC)0^;>=HDP%3l#8v=iMI#&<$m&&|IV@9SICH^quIXwoE%k#V=;~*hA?>j|4+Iq!tU7jn?13Ak@=glXFX_|nDf9^+x<Y!m-8TD6-7zBF$Y$oGjXXIS)DzQJ6i@vzP;DzHhak^ME_Y*DOP&g8Ck~0~5@;uD)tmC$Jcy!DyX>yi{ee>-s%qMDI0>s*TrJZCs)}L1Jzzz`?(*a2i%W4=4i3XWMG&*W@SL|3AXdG#95B=e+)97q*#r`9V4dE;{5j59r&Cq<QxY1&U{kNVm7iyag4rzK4TADvNEZ$(A%dA6<T`>w}G&Ian1Odszr1u1ykH-(+?8Nen@ZWbd%(FDaNyH$4>OH_Ye1{uKYBEGZD%)4UyVz5n`kBv1dfy45EmB#7*5i)}WCws$2eA$)62}NrOF*p|=;>UX$a!BKieWLGWt~%^Cl-yfB9)$>1!1`Vsvl+2CC-Lf7-zX32ht1#d$Jsw1zT}lAu&*J&r&XwVue+^YMe%~2YjZGW?BTD)T;KssPRIPsW+B6Pwu!qmSovca=ev=V~|EM3u*&A@Umh$_0xGLDS&32oV5;>Je6@yN5wjQ7%aAh6D8x!p@%R&O#+{lYL7=a78uQaXlFifM0bn+8a(zz+XUtvhnsNb_V<Ou^Bm#F!>S)gH08zg7`6t^v8@8sa;>v};SBsN%$&Y(ir7E(!^j^*639B3B&qpFlgs~rEeS5a5ZIl_Cs!{%48=I~V-Z}v_{=Bha*!k^VtV!B7vZirQ4_LaE>d6QS1+Cg;>%}OFMdwXzZMx9#95M_K&qTYMG8<t;dr#QqcJaOX!?#Ux_aSI?{JnT<5XtLXt{<%X4VS!G&EF6>N)~U?y5=Q^OCqnCO4Mp)>$}|FtBV*)hD!OQlx<t4<BSA!MPBAkS0L>@Ku=ylkK2LqwUA?CxwKv-hLt@8HdHRrvV?vW74Y{Vg^etiIYAWhrI7dj{RU>@+4z9!5JJ!a!cyTa3?TVfy|TKkGwS$EW2zXm_{!PAbMKpDl$}(R|ns>Syk^+2P+<ajRS#MjpJ|w9swAwR-K1T28mjp+DQhF92xnuOb$KWr?*8aYuUUGbcNU|-Jl*hFkzOEkK*SYEx&&59J_8$ZC=mSDj5C@cn&~@CWHhbS`v+w<%K#kK{B01G6&oD^-Q(*vfaxkVdhy(=`J6K_S@Q8X2C=bi>L*&Y&Uj}9MgRZh3%3IK(}gU+0z{d;<m#74RQ=^4OEh{QblTUkJXs91@8=^>k)BDLDjk3w^-b*Kr-1u4A8wVEFNI~D2&4l8$$<<O4K2~yJ&B6y$``E-~s6GUGb-`wLnF0KldUZSQ~a>V%Z6>6%qRtp)Ei-_n8Mh?!gEZta{b5Ozd0Yd$(y}J_OWIlG5I*LXd#~S_xqX{GmzL0)QfE?Rx5;cd&FN8e7$_+!F9k1(=y$jt(=nKC}?`&=*^@V0UGjl4LE#sO;`9pVbx0)v%@*yY<vIJQJDJEEhfhF3<)n`onLpUVM`47;1d={IgiZldI>S=DW&TO_IxxFo#;i5EEh5(#v1P${vPS&;KV@>-5Co(|aNS?7tP!<wt_x{vy!u<8jYXXjaZUVweCvQyNUdQ`vz-gOCNTM_1@3?QkG;djyG+=_!bqL#qU4?i6ZJoJ|nmh<CzxNFMwk4dp1B18F!zpt)ebpTa?eiyQ?SpUTf^#hmS=ACG0{70dZ<Io&08J-v3vAXyuxPW48^>3$?r=)w$VbG7FU0RnS@RN6rwPRtps6^#a$Ni19z9w6Jf3rW6%e-LMD8_@FKZ~OA8A7*vml9h87NjdA_sC91lZr{cEx+8AgGN7Old#||0EgTeKG%T6KUSXX*b`#Vh1*tR)x`ZEVWFi}}U)czJ?U-n}SLA{A(zPd2MPpMx4oAv5R3WVPXiOhFGHo~0r^C9`Syf#t{2zHfLsEcE!37K}JTJIgebF$@3dBO|^B_q{Kc?DyVf4FZ3p9%&D4v#(QtlW;^i0R?jiY4Hac-%txV2D~hSav-2zWOvIU+$(GwK2X)rON`#Tulvc${_|NDpZ#D+xfmwZ@QUb;Wn~%K=BKS5XLF0}VK?uX|0D|M0=~+mBxp@fdn0gd6?**G1_@=a;{UC*t(-qol5C9C!WBNJ%3EB=&K{W^DoAT@zICd8$F8ZFJcj(Z?{E!nt&mxrEE2oAWSNQ36bAUS#w%$@7_4ef>${M`1oMTk30=-cjU_d3Z4KQ;*$R`o<A#kn_=Zj1FiS=5EcrH&!s#ox31n{+UN4H_zF`zjOC{ycH}~eyzh$4l@n%gMi)La(hO`mo?QWQHNNL;Z1ob(Al-A<z5FAg~N_l;7v<|&Zb$<ESGZ^O;I>9%sl2}R7tdMW@WlusO#g}fHnS_fdn66OUGWr?9d^7;4TF*3^8Fn3^im85O7wdxfp=fFJ>#H3>>>XxW<$gRQZG2B4&`=a91rAXO6L6*R3VMI5bhQ=c?`Z8x~Xb8K@TyyR$X}x-d@a7&G+cr`Q}>CGgsmEjgJ=QrtzEC{cSNolHRYBY!#=`lR7m-75^-de8xnJF8iSdeF@fl>pL~$oXi8b=oqiuuenKy0^BDHXqOwV>U<a7@wD`#fN&gtf!4Ofx9TwEH$Q~q10G9=rbDHbR?F|RNBj}^;sh5lK0F0soKI|?x`b09zc~?@91oMD-ov%R0&Ht|EOH8QqPVH)04@FPcJC>^1j$<W%d-M`qsfIR3_^>(2W^KR`_TwWeq^yt+SfaKndxl6L>#RCl&ymdLzlI9Qv`0If1N$xR`Q(WmruC8*(ikSyFW3z_U@c;(>7Jv6b-nUL=DjhH<6Oq|JV7h}(pa%)r*Q0BR%LT3n{N(r1;X*h+|3MC@D2j)PErY;=eIwk+;6(W{1923dMaJroA@m42o8&X&DSjbF?8v_z6z8>2dxZ?evMxwvjUORe2xF<T3&m13y&n{P`U_Qnd-xJ(yO6Zer~UZ*$6j@heeHHXs}6cpTLS^{*tJ(L3~<f}>JkS(L3I6xiNrZr1w)X=i6S^!HW06-V&NpDw?SCL9}bdBz)y!mpvNzJ<p3%E+I1M>A*c6Ny1<(mdfGLN=6bLtU+{*xGP;RGU&!29{<<R2%O9|J+{K+n+nNho|=iQ*>G&n5^7&$h|y!?(FOzx+7hX#Y1c13-vx;&l<j(GJD1utRrNseI8u4#v0PEPq{hx$cy4L(Ro%t#XZjz0{}HGInDR+qDV!GEwjwce1|?Kikz84>HfmT3Od&TCSH}h|$&j5NCS-j+(Ac0^Mz>IY;J@D07>fBH<HC$`>3}V!!ttsD^w#lYFawut)zRH-?+p&C_u6V2|$71&K!{^ZiY}<U%Wp;K6<SfT(oW-nQzpXz1)^`5aAyWH`SVA+GH2+?}1%{W^~IRuS$3SFo&q=ex7>UAn8>n#gcG!LZ)lMNZd87Xzpk+<s7H0f|)KJnZ5=B%kzmZr}cs1%mf{s7Lm9Xn`aOhvKE-aJW13hh)6^-^U6hJzEg}aomsODBlHDQ2XFVx(#M`0XRggQIwo*ou&S)@5l4AiA?1J9J)YyESFmuJayT}cOJ3OB)`CQ3)EMCmV~6w7kk@k?-n^m&fa8a8R^9T{`EhK3!;BdW`RHB+z@*oF5qBJ9mwp!D5`#aR>Prj7d`PqE8%!ydj^Z`3(JLA?BZJP-&t?H_)rZrp2DKHw>_(Kb#`!Q9AVB*e);V36IOX2&qb+0c7^ePc?=YfI0>&_d@w~qkT3Ior}7U_iClg{*MhkB`+2T1rJTFawq+Yc;=v!l+Z`0+QX><(p92_gazOc644)O*GL*mk`^#U>8I{bad64Df&}4JJe5SEOV?r41Qik^5P*HI8;->|pM<YK_$&(rF;L`w2(ENnj&q5#-Dt_XI?#pMs!VffQV>7vPU?-JVr;Wk=I|qA^wF5`IENos|ib9?BYxIPR`3?|CWgC^0aph&t<y1sWmTi2#_$T_Lts}qvAtO_p%x4rjA;rJR!NUj80TX_FngRypoF<nahh^(q_shroBxibwvJt5(^}G#;dZFWH=%M|WD)iv%cyrrlqTH4T+Fc#4b(a(jIO4B_(|Zqx`8uZybnQ61g7~Ao#3OAvkKOuBqtf>(8Q^G)bds+Mi;S)|F}y?6CE$)m2MQn`9@Ne~t*a^;fYRosOIibC941>?>Y(VP<$!>?HRFJ)o}3muqF%+<A>9W@*}<OHaR*Ov0G@%ZCzrnxfA-|fH%NY;f2Iz=XV7?hm=f4-O<t9SBaci8;of!$eVgr%$qbOgHl;N4$Z?#P#1eH0t!MNjE<j$tdhyTV-EtQM>s<uA1s}(8$Szxa`FX4r2njfj?$59b@H6ghY>9U@-kd9bw_nSSq6Ezr<*s<^ewV8c&|>^ot1w}kuWhcm3*t3RvINCASklz3>ABE@e-|Q|t2HZ~zljV^*tWEkANa-)^})~(`D9TBov5;%%J(>MgtJXv>Ht)U#hCP!KHIqDxqALzq|a#c1$nj%R9gWJRHv7N$sc~}H_#HqhE=Lm?^tK26s*Qr-Afqf<n;XT>C;C-olweIYh}dOZx0`6cshxv0S1$T$KZ1gkg=HXtr@efnkD1YFim2Lhz!3U&GQMadIb>rslw-P3Ne96;cPkF{*bQd6wPt~6{lnpl;}_e&^BO{HbonNt_l;Vgo=v?2~bsUnQ8kly!>y_mPL^5a7Lg=oQN*IxVwZ=+lmkb&+djye2kl}lgodZ67LXj5En6NmnB|-_i7hBiipV|$s5b=+$O-kGa`mhHL91NV?$G3pTb0Yig-Qz0RuFWOd!p2T{1485>5YA({ykN!3flUU%=o$|Ic{0(Ws1V@cYlfN1XhUz;vvUr>R?5TAAu`XjQsP4e0t~C`o!zr4>rtB}K%uiE&d13HzLjPGd77D7Zmw7ckx4#s>Q-d?~9B-#6y4h{u?rK=9_3hKk3Rzr@zT+6i$5Xzgw&gjoCPt1LCG$qa~kJqmjlsw^R9R+?vH;3rT`FaBXl9Qs7Nf57z%f19u1Be)ijX%bVd?#k;VmR)GpFO&~dtIp!>)f8ZlX;)E!c%b}@^1$d$+exg*R1dd_d*)V)7O&J$Z^QC<`!{xZ`%RyB1ICBVk-l^7q0E_7#`FYAr=z}P`%hCYSft3+OHTN!4#><EH|&^!Ky|R3T{QQ&Y{AGJ_wwmXTVqQ-QSQ>2v2LP#%slJC#?i$$CG~X4GCIs-&pdD@6EVt(Kjuu%#bRMyn{u<t^3UczT#_?8oaG=p^>MnPcvGLg&gRo%HXLzJ#IY|GmU+Qw6w;UIe9EbbOxKq=>u&z7-5pgZ9v|sPx@l?Ga+_^;^eDGk60l^-BQ^AmB!87^Pq<XzE2C0g*wpsH<;=Pky!Yd9_9|WeSOpgj5tMSORJlWSv*p-_kGuzmufBHg>EQz|dL95XJ1ftbd2UlhvgFn|j}PxXc>S<9CCV$;s9tyKH0qOE0BW8Qaq2H)?Ma#*_bWoQ-6a52p50f3VJRUZS5@3>uKsm>^EF^BpQi>!x8D?u<>l1CnAG=*^qhPBE-RO(u}#&{f_%d(H@8_Tqg|n@&ZE3VnIxH{dm>GeqyF7x1J+){sUeGHa9_Ue7Oim|S1xw^O(CMx`LMlnN*xZ0A!0A|!Ft}pe|NsCKkD^D{GJOeC7Ep`(GacG>AUXcYfW8AqbLQ>qs?k#3qC-b>&k^I7gq;o=0>*)bZ_0_i`>P!8{-w1-Swj6Rg7BxQbVgsq)rCT-+~r5qCg8q;AfJpUvsl2n%9l`+i^SZtp_w(kGm^h4bUihMic-+XHYl=-E3YgC1SPp9<7L11mG8B;rmkT?c2h)oTRq)=vc6z#5PTMpb5F%qM^H-OCaP?YinfN$W>{-*HWuTUI{2W31=RpoYX(TGVNrJC8|buJoul^+S>Q3He$v5ID8OI)rGl+G9?$VtVx6ly5*bZ@=;2J0!O$paJ0NUsVc<$Gsl9Ut~vF1Rmag^DI^9S9dgXD!Baw_0^0LYJDX(T`3e+Efzj}A;A|bV8Uu?Lwp6a<XvJAgm9|_~YI6Oub@BAix~A+e+otrC*^9dth^3Ra_Lm^EcCt;P9`gE~o2~iv=(Xx3w4t_-L%B(PB$gC`++dY8E0_ygTu4FiGQ1kXcehH<C+WPsoVJo>y%Atv>$_O#m4@@8dn)gR-caBOA+c^JuXT@>WhF4*z5`QeUz6MZioRzv1yo<s>iwgZ<yROfz~fj;N4cI!WAtWocJwGo)NE8Kqh_m>J*s!D7E`=k6DBDcK)gvXkWGjx_28)&jGCQLUpZ!}d(=i(F__{MqH-Ru8LjAqE=1KqYsE57Z}vpUOyI7uXb~to*|~GO*1AMN*SrlP`ZD9gcKI4DUsH0*_&S{8BNfLNaxEUOszm0LS-TMV2zQpp{gwFrv3`TbS*P^~>#{s}op?5nppsWy@5iHXTsErr%}AIypf%0L%R)6jXrP<dNab5^-+S%h)4%W@A3l2X@lzGrnv+luEuDLBKX~}GKG3|7)>Ip77A&Vw*ag+a<uH`Zv`$-#0<{Cc3a#p@AAknVl_;<3pu-WBinZNt>gcPU1|zH)sxo7YWZe=3q<$U0>^e<Om(2OLPgI#BUVW-8%~|P;Vn(lJGdHT;TK%;bF)c<FvvG<yl)d*Lzo@3ErYZ&4Ag*pj6qs|ReF?0tOlz~&2p>7eNVLq`5>^(1wCzz<O@fzTW~x-=`lZYyXmoxjqzMd}8vODD3M{S%Wh!VIB3u-BX%2DWC_FbMTBhrr!ykBuj~~DJ_z54>wG`2{GpCtn(uU?NoFLmd_D=o88x~2xt^d!Y^Cj^kyzoPJO1Nze$5fsr^!u=O%^vk<B*X(sKuUUjF~=`&sQUozj_EgNmt+}K_}#{ts{!#k7Ag$*aFnDp5w^u`tErU1^8J|Q<fzeAuXD4IeJRa3w)(neZ3TO(SolIAh^&6~223Lk#<dLxjg(Pp*k^uwyR^<t=dGGZ39d@cTDl2-uN<a`K~nr=#8~MgYV+*NC+>2OSGo@6YW=S($Bp0EaMHo@-_>Y^1?oo)GAU2Y`<cu5wPmx*b#KHkcD4CVuI7bw&6<=Z>$vt0l3e=^qptYFiHod+eDykE5u=C<8!^$5eIsLj#X%`F{X!loFhf%Q9h;QzYt(Pr^zNl&uAF_uJ)I#c8SuLz{970vmDPHlYt5vH6v(S79cOC`Qsbys*x+&uEmT&F*SlOk2Lnln^;G_#$>xpks@%Q)jv)={>9-2$r7@e3uxD7<ihM2{gF?SHyis3X))-oq`?Wgra)4<%UhpfBkY$fAz&wv%xA#1vzvnr9{bnLDW8?n-H^}Do',
 'export_snapshot.py': 'c-oCtOK%%D5WWl0e_+)Exv;c0j3h8r=un_8(4t5U)X7B^2x>WNG1-?%u5H!ufA0**)x%2M?m^;m-ro#A?qo8#e(GB5nPNp-cdGDH_S-MB&VmE`V46GJwq5UjVqXC|X5qdE=b5*v0LM&&JE)-DIdjk{t2|WOd@`AgN2ZQ`?H%qr*llc-+f-(~d;j9vW;`lwTeD7i60sz?!PoI<r`T<kwe2HQ4i#VT<`@K1^Few~R^yR|64O&!Qiw{B_iR2MF?@bx&fDB|mGPBnz=@2h%C#))IQHr6^5XL9{L|UR#npSP(isixK~+Yx-~amaBN-DkxPxfq(dDD@n@~aCcF+iDidLi2g7?0hUGa=ELpxA)%7GpWzPAlqQigfeD(wX2ld-b3j8+73v|^?VJ2?gKM7G)&Su4|sp|pUuWs4p1`Oo&Quc7fbK_WD`0zG)sHVY|rTS%E5n&wJtsgjN0vsv4FJ`Jrm7WCrQ_K*xv%VDHs7yif+;f!*$L^1VI&~}dRN#fex7J$No29fqT<8k0}RTR*9(0LH@WCu?j6u66)&vwa;=Bqig0WKS`3VvUrG9uQreuzbgQe%>bjQy|(8&RAcc10Nn>@#)u8i&gYeqDEdn~nlG(mf7JfzytM#SOWY1}aT>2+C{9cyBBED@~&+nO;0j9E1ft5v*Ld)IM6_1|BQlT6*dFb!FTJuobAb&n-UOsz*t~Myj-S8+G~q!{GPeb(d-Aw9$?>Rqr=3jD-gSj~HdW4H<VDRYS~F6rL?AY5DvWwup3o2VUSZPAD!2^jP@IVHYfVNytt~ZBeOutrfHABesnOIG}dix`Q@g`%w43$8+bm(s|{3$5+%gJ`4w^W>Ak&F#x(6j>r5MveTA+Y@6BOhP*&%aip`M_>@JWtURJB_OZet2a#p@2A;IJLuwRZu9C{oht+>(bBo#`4y%o;l`l3!4zAcht->kG<612D-l!K-5}C$-c!$9;UFymLK|+)nR2T@wI7?oNz21@H>5S~`BGaM%KIBQ`qiB^>yjoWf)4io-PSh^DEoC5Rnx?6wMf@AM=RSY522%+3BK%I-F(o99$3dmL3a=<f9LE1~e%G;AB$BveaE%}f&hsxV@@1f0-PfHHcFLe}y+!hJg)xg?@h(6PK4bjvhVKWIlFvx>6|<*<0z_q0_<bHoEf(_a3bNd{118yytda|w9}M*-A>E|w&bED{XEgFBgJa6?lWW4|e}$qO?~k_vVI%sAi2fPLtO(bJ<MY6+2{If->9-x=E)N43NGKgX84o_qolM^yc(CV;?wM2KO9;x4kEDF$16YHTxU*qAl=rquX)2&BpI=_~Jk+Q<+_>Txr|D_`4iH1eq~n<oCKp*8fc4hH?kd~^wM?;BOVV8h+=6PtjQkO1i96=6=_kk!3FqZ_^dWrE$yzn0M0u#;_y`=1en@GGMQd!^;`00R-60Wt!<3jIe5933N-9Xoz$sGFwI(G`NX2iI@#tR>HBWf',
 'longchau_snapshot.py': 'c-p;PUvC`AaewEhnD8EE_PiWYy3^r8X%DARI-g>lBuhNU0FM~V?#}Mcadu{JW|kC>CSV{>L0)1wFM%K8i(wcEV#Edl1d$N<L59Bv{Ymnxs_veie@p7@<X|s%d%CNutE;N(Uyru8w;yNe+&Nf;YiE+BWt5iAI9sRFFps0c3Dc<)Iz{x=8V}+yLB~8>MmzCx73D>ihGmS=pB^4QbMkCmM*jBp_SOtt04zzONf~Ep;h=Yxm(Bs;mQlW?eonI0x%pn4%CKozMrFK=^cekn=+NiaSsF>dXc@+dzb@m%fM11q5mEo3%mO`5PcZ*FUg__PuvjGVSpSSO{q0MvNPl03<wB-f;TyW^X`ayo^*uN6?YY)zJjtenN+=N4c>(}FBdEcktym>-xg}kam&-&yT9nIVYwHIOpTBtY^ocWa+yq-R!4B=5-gdV>efr|?1zIoMbOBV>*+f6~C)v_JP2Hj6o}HcfeUC4<?ms_#bny6L@Z#a)hX;pGpVJ&igziqch_Jg+g3q)ZZHIXoPm*Z+Se_P>Fo||%V3T8a>%pTJ5AVNt7<}^R;o}E{kmpXLVv@%z+H^vqIA5;HY)M~3Y(|!3bMO@X%;VyO{!Zg?o@PaKKb;=tQCOZ+V-lZ&snGN+k4}m0P9t})_2A(r_dkEER(_V{C*VJK-?>Flq9i=W_d7n#RF)B@rDyNqC+!Ou6Tgmb?cw?Rcl={CkJr<AM6I_zMC+Dg{pVJ^9if9eVUB04qcT4aW?;N%8f0mZN5y(ceGc<=bh-6ckDnfVFF3saUBPiB3p`~?h$P7B=yP$4KX3Erd;Ix6f8OEG59qVZ>2sW9lasI3S;;8O;`t;r1fG`sImMQ4ZB3(@6BofED@qT{HQ47>0q1p|I-X`Xe&kc9G=!98x#PM6KgrIb+#5J?>M(mLaA}q=!zBJX^2+FCsUXr<EYM<dzv|~c`Q!jNaUTUck5)-I!J{vDJzu2QObbpP`Newd<?hiJrSrvlZ!g?C{tk6?6r6#*4%7RK^y0nYr2zaQRe=33vpDsVI7P1#jEEJ@fCV+s(g>O_RA4j)1KmxncbY8&>g1(ZnOg2DjwVoqBj@Ote^bA+Fgfv*kf?6-UrZE(J}Kg~D8m$MR(<?3oDUt$HP}}RF=Oxqu&sW2FKnpP3HV>QB3{f<<$1y!QJl`{Pdv**;Zn$x>4nv^_>I?P36{BRO<w`GxECa2w4fEnIoMlB;D@VKlukV%c!gj6saZ+I0j4MI#bgmDQ<_h8+^@+V6PCLSYVGwoxC3YvC(RjbT9s24q6MwGu>$8uPUoO+MFYl}8OCN*OC%Iq3WR~?HSE`%2w;nr8cX`$*Utkw9*9lTxo3!a4|?UJd%N_BMNUX{Mgfi&QrfI@^el9C1@V-$XnF3PhRIs<b(>lFX`M}-Cv@QQ4)Ebha<3JM0MoGKd8Js$aA>p{EyMFFP1Bx9{Z@l%0ZK~UU9+soI;XYLDIa=e9s%u2<<VY{z*4*ak^o@nt_a1^$MJNu9RyDv{_rq(aR2ar5Nsd2W)(@4@>UiD=PyRiErZ;}<<L1Y$8fNW3y^IdXf()fa78`fgs5106`m(qNK~U9K7B6){?XoXr--b=d4%q2;E}t^vsK|9;|m@<lOLutt)4s2BOj+3&R;aR16oL3-76yEG%wC#CdW8R7a%935Eh9AAR?&(OI=(JY>*pN9Ca8v(|A%6*Mu?P{eU>*VV-BX=N<@&9A!y3+4xJGg^r@21$h#t8IBZ~n9ZGo`aU`OIFFLiw&}Eey!-JYk7gEtoQ%|>WG1b%J<-5kfhS9-j!N2v`Vc<Wm;gnx2Aa$dW`>$>qW|cS0qYp_gnimzO$Y3M9ps>{2)p@C!yyKQeL7iK%%DcdcBr!|dO3+!rQ>}sI_K>eI)~@0NPVq#$(p-o-W0+fPn-NMvIs6%WI61i<eYhX=p^A7d^Pe03NTrq#MNZ*`_m{|(HBc)DyHIYdfWZo-SCLF7~*N<@RJbyx~0z&nVD2cYp83jNxJz{NKZhV<!MD-#M3qN8*B~H(E^7AGf>WuH&7OXF&12B|L)LNj29-P)QIISc-8(SETUPKOuc4zu|%j3AuJ@iWSzrbMG`Mum~u69M{u;j(r7iSR)$Fwk}*1mRp{keRt}vd<d`@?Q2|cb^Jtl!M$XwHgAfWI&0-)9pJ0VGr)9CrmS!<@Qdr%&0~32%IP9A&qbUm}lu;ob9;1kvTANCZn!GgO6z54ihbg_zl@Vo_ljuP_z~qP_i8R*Q!*`E9W~U>Wta6eh`DO6srb!YO1%HdD;1he(%Q7$Qt);10qECjTae4X^np<=nPs7%9Z5C?bN!Z4kVu~*NEKJY6GCPS<-W;Bpz?&w|RqAhN8W#WupM>Ws;<DKku1hfM4oE$V$-rx=noa+~LrGu>On20Dj5R<_*OPK5kFY9O6j&fR<!l^<W8jp{b`-PMC}?e>HhV5GK}fPsZc&Xy=EuIQpMXP(@Ut?6cn9jTHMp$XdEBKbs7gS{!Te#gbjrn>e`BNe@Xfz3oke{0`ft}&qv=7P|B2WH0u<bN5iXsR#nr1{WUBMvMqIe6(S$XX@R3+sB##B#F8drliPLpd+4kb$)LF1cxzyRITch55a}*y0bacv2%VyA@n~<CjkAw_ka+z|a$pCoJ&_ox7-9W@FEcI5$H>H&`tw)B6)aKO@BAi*H9v#=(-*FB;djZu~WLSARg^TVqSlS5-Xq$0DA*p+-kM4yZ`M!T|H-x3U`e@hyMkmgf>jIVu@4N3j1-TVX)eh{?4mhF~op6TG6&~PF#g<LtQWojg{TFy9k+o3p2F*kK!4|Q3;wz8ClQ6-WY%VuTpqDt}W;1{u(%AiqRGI;*o~@2G(i<GQ=}$)z!)?#^=ixXlcGNXs>Wc7n&-pFqt<+96%{9u$rm7myB^Mh7=N;#9G!G}|%&X?)gHm+zc)loifYLk#cZ-?_8I=7nK5um?vUNU*h<_qq!G2!rppRuT#q+g<T6Zb#31-b<)#JpCa*z$78XN@$)E?MTTLv&p<DB@DyTM`qKi7sFv9Ltnt6~^-l*myH&0|(6)6K_;jT&=;5Nn?)>(Z(O3lcJk23bzglll7U^-mEds*Z|%w;^?_JO+uXMSyyzakefR{J}1;jVzCAV#|Ir1GM!78-lLef)|KIHsT-p+^8eRVaq^G?%j6ojl`ohqX*RVf#{ed@mf4hoyawfi#0@ze0AM*g0?VOEkcjIr3uk;d&m9bkN;^~;yTno!{rp9Z`5K<J!mN4!?=X;L8k|-P{>@1mzj%+ER_Z6JBP%?^K}UDterKN7Y*p}w5Bl$0|gSoc^<A7#a#zlTdZ4zj3W{hEki<EBK_GS$|YL3h_u)0Hx(prLCw@RY~;BoVtBA!boJ_&Oa1K^>B1Sm`6lC^i>uc^K_~kAch2vBa`pP}(*=iz|L~i&^X7kAmRNg#D__WVZQK0Bkr-h=70_C_+}JH3()I1-)BdlTvQiz`*Nhx~d1$SK&&Aqtq?Jeq2`pC7rygJo-sYqd)f*T{)}c{BKiau<+;Kd}qWVsSjmOjsY=wA9pcU}9u*%+XUMwIJQ>StnoF$ns*<`X_orAmK3!PD%J72~rIc%Eshs+LPC1+=0o|2hldVCxG|JEwmYw6rWW28H$Z+`9+Bqh!Yy7ym~&iv~2FJj1o!<+-r`FZS|##gU?S1PethBWleuOU}xc=gNDR^4uU_xd@FACao$V61b`xwW^qcXM-_{J5a*$Gzp&+JtY=TmqLrViAHo(iaK#(>ERL=f|gW`XjM^`o__JYZaX{>qrB~4LY256=V=ASp0h(tS;6`S=s5WC4eYU5S!8?+7i}5=a$W&EesRXr@R@b%{XhuNi)uwamtKrY0j^4>a3P4tEJI6Z^mh}{icUO8ES_T*%bbVbBvXo6}uz~kgjN1$f1KXg3pIsS@<|qIPhqsNn$DOW23ef#?->=vSLt6;%Fn2m^ZfS+C{789G0eUz_bX{2S|x_Jt5Ydik|E%KB&Y3bF=-ex3)tRs^}I*Dh9*TJe-u?IGdi^(fnFKo%j}LsRW71i_jRvpeP^dx$APa^P!~3azv$FrGj)U^mEBuKH%Q1Q0P!#M;w8EMk$EAFGPyeG7K0Je@ou^yIM*{d7~<bO0+z^)e+IDx{xs$odu#;D1+S*gT?Yb3~+uCo-U=bO%KkP<N?f^B4)j$DhMQ^xQN0jXw^=PK%`Vcmey0#=X#9JRy1r^Qkc5SfcWiJ&3Khiv|0>B3cHgw-1j6-Ysn=$ZJ~2NI(3V$KuE%`;zq&PtpfJa=CYY;(ipRKl02aVsb)YTbkrQVLaYqr+dS5IP?Y3XlkF=^xJZCOPlGX)B<P7K4!1^RqFCTbNSsnlYaZMOvv#n@dNpMiw=qmL^B{{0)D|IEQ0q;xOC4IKKylHj?D1VH6@ruKyo%Qj?YM26;ylJ9lfblcB_Kihpk4%`<&TF2O})acwJZR0;l5@YlW=Y2-XH?uavaas@V){O#TuMyz8e55!c|rc7iOf)Zy4`j|3G&;O(lLsTaa|Xx~GPQHSx!-I+~h+q>@N#29<<tSRp-2QPTvfc<a2Pj%&yCil_mP&wX)+Y*rw7wet!UR**d_!O%dCSxX~VB?Fo^6n2F=sA=ncEE8XKGc8RQ@_qq@S4v<bF4$vJN`~VvRymS3fK|9?H=IJwOEN{~RSKlzyl$mBb)K~1@qw~lI((3xt+GZ}DZO<Hj5sILuw2`F#(cPhum<>FbE9fvc~rync+j_TYQ0r;O~EBN_`i`<OsKP<{hfHn2Yo@jK5}3*pc~C)lA8^_ur!kaQrn0s)D%(aAs-CQQnyMJ^dW5**cmBOhxL{TgjYHNnR81YSW~&RvI3t|_rA&7`+R12luq8Qa@~5*2X{><;?u*=9y@4D5Rmwe^U3}17w$Dk!}9=b>qu*h9MmFfvtF#X-(+;^rK&O*O*Z9QynlplkbMvAj?ye-l0Ej)2XDjsTH|V*d7*)el>&I1Df_KlE)5lwdTiL6s<yhs5WrqJdB=IC;!P?CL2+8)OJ{Lgr9JMl^3hNWMlO_q5<;$v_jVWW>#&+isn+H=jVXr_1<JCjA2TyKXQ+tf0+?Q+4&u6V=Xe+XQai+tlcq6~m~b6?$<AC#s@B^ZHWI!K7*y9V=;AI(wBD#bw1b`0EPpyJ+)lk`5u@wOy<USUX4b^es;biI{T$w{-lc4Um{h^+R`%-Qh9mig+Fca)b?Uu_QJ<c_5v8=w*Sfx&KL5Y=MArdRLunR%MOI&B22g`bhViBXDe5%jXxiY0rkr;5Ffu46;N)<l2}f!kT%NO#(QNx&Jl?EKr2(pLt3%!MV_75@C|QQi=H6jioRJ6BXB6Fd;gs9Wr*1blck25No!@UAG0I08-OZNK;yoR6cqj_Rqv%stKVSJb3c4ZP@pVdd@m^O9`x@@QsRZ;|PoVQY2_Ck}-otpR95)p?sd1b(c!qUjWSvsV2<~6RgUyy&?JfJHmk;bgTtG)9{babiIuL#cQ=gnRGW$bgMH%Z~R~xc*ahD==;-Bx5&&Ax6XNjp5i0D>|xDOnl)|Fj>O8p~9aHQi}F1w0Vz^8IEFONygEH{^9A<zvV@X?s1luM>4JSscOZDVS%qpQLz6w(S>0d}c7mTHotYN}(JqOQxk=F>;B(gqp}RlI0s<oNfUy&JgAmCrdnuiR&<h8iC+y^h&rMudqC9%>}2ajQg|3<WB3*FtT5$H<y$F!-tufq~+%ao9{Y9x<#4tf`{7qTsmB_ul+^vH(Z?;Wt;We>UNK!++4Oe|Giizb1CR?lGRj_WQ#(sjA!?XD9*5nOgy=U?%o1dha9{G$gu3&4o8mMHtl(s&Q2KT^d4LQ*GERs~0@a62fa-OvMOV^Bw7M)-i8ab#PWY($RHfr_EGcuIgscFr5M!Su}?7IH`GHW0Ip!<1Pl_rEC6@tUz}{;8GzY+`187#VdwMO`SS3iKdqVc9WQrtVQN<Yk_Y%bv;d^p@_mr@g>bRq7O^Kj?6bY+mK~~-w`02n32gn(h=`=c~Ey`c%<UO!>WT;>Z3}#n`c`0j>rYxFp14)ysan!w%Rs~tcSfvN@fZiMXG5)=7b_A$qPp1#i*{dHozy9J*u>2ZQa$WII@4Vp;@fSrT>&EViati1)e9!GF;oM4>~i8HOkV83i5^?&JOEw#eW@>XwH=WqgrQ;0cX7weB5sHsFR(>=jT)p&*zJExb!b7ULppv0c5<Kza2oqKPVRA?K>Zg)NM{*etLY<V!B?g3X3+7C72YXdx}XMkGOPV;IjlpRN1|VUQXjVj5Tktkx$vy()2U&^HzsydsK_L^~3fpNeHw#(r}*qX=YR}-hEeJcN{wJzAK$Bw~QZ0-Hyy9$YXQafMYVP#{*ps65H3U+E+EIi@-SyFzeDUWjv9scNE0YEx&PoP@X0(W~ePFtNG<t<}v#VjW#kc{5v<pg@;I)!<Y;@nnS`TsKA7JC<zpqw^=$ZEW~QGFqcAW#IP}|dJu9qLrsq{*rBQ}LoEO<tP=QW9_4=hZjqi%-%6EBAh#X9%y90tiFwN8JEgL%J(iGDj-<TX0yH!$QCC}!ZU21JDijFR|MS2uQtMS&tI@OBmt~m|^?lh*lauvsZ1xHhRrF3fWPu$Cz`m{Rxs+RN^f=3sO~`ZuFl3<NzT2O-UWO-#Vn&EzpxR-Qhi3_8`#?~Nl#$$GXR>it7D2{=uVQ4CHpxwSBs+u2Y!*A_RyjM4T+hr8b;FjrPD7U-aIq@+q&}^$ep?&AE}|kBQxOBsf9xn&5;aUIDH*!7yj9&L!%~8Pwmb-M^d_^0?+%BIsyTeqhgZnLIg)i6=^h%vFi$G!B7Nr738Ip%&0tJ2^Lg=D0a-1QK(1)vzmRs%W|iu)HnZ3Hh!)uacrN!Pq0>fupPyDUaCU2{c%CMpo<84qszV7+xkg=6o^I>$wJ5I+pDp=%6I^w~g2>31D?gWQ^teR(jN-#1nMwR|aC0z5$syOQ?fHYAfgz-#@9sa@+$8Y+G6OV9LU`n#F2k1|-Hva}z9W6<gDSad%%ZkM@ro9W*0$Bg0G%SzBU)X3dEA&sZ$w6cCK|(40~AnO#)bn)q`+B2t;|%z6xJ5GQKOD(LY`h9$Mq|M5;bK1P46%0AIwnd$y1ZAfQbu*UM7mgxF%NhHP;FoSDRy!x;CZ5l6RLT<XzQNw|1x%QP&G=6?*b4AC0OGZD1V%e%6T^{R8@IbYp?kD8*|PJa&1l7lxqxl`0OYxF(twdb1okO77AIMtRO9=y+3yBdXPJgw7bjB0)yMBlH{lK*fSAYRz@jOP@n%I%wR*IwfnbfG3Av;Nfuyb2*~bDB?5WnQIjD(AI(MQsaP7oVkcx<6H&G;~6?=h=d-j!Z7Kax}%Ks>&W?zRA&psu8=MF`bapvCq>X)Vb>;R?Du`mfw`Ic8Po^WNy2Ytz?bAvA_I=SDfl8n85KQIS0&?nUAXpgWJ}vv-1#Rgs?C{iiL<au^-6)xO;DqIwy9Q{*0_coC{3gom%fpTQ0JjB+iptA-g_}fEJ|#{l6WOE{dpigimb~0w-Wz8BDdCz09^#{5-F;oHQ}yvdr+CNjbUBnm-*=!Y@HUQG+7<<ZCD)G(-=7e7#-~0RY!rK(wyt<E>`OJAeAR*_8jRiow8eJVngK3YJs8y5no%?yJ4&yxfa1$tOF#_9)T*U808Nrn4~G?D-b>99jX%;q+C<Mg;eW$@9%wxH{;y?OFqW#?ClX_bauYye6Y9I&5Qgx2+ETP0%EZ|C);}GK3$!YltSlFlB4P@L1h`NsoMJ`S~*jx<*cmu{!C;OcIQi?)Ld-nV7W_*xYlf+UhulA)UF`4?$|$t>cwi#R1K+($e!OPpLGvV55Z_Y<kT^y4?O*}Pjz)|<4|uBqTu@NB%%%*l}EC;y4fv_oS1qpZ4Y$|`73z5|83<#+#x{d0$`^bUrvVoSRE<}?@^gGWw&9x$4Rl56N*gzIna9X;&HQ*`0eo00B(ZV_0SHoG?jSgjeySRK1tGX+|}F;vSE8X4CdDjqm{Q~$z1Osy{xmGu3dCET|;^|TIDP4H>SB%9)^+-+mSZ!m!T;tZyte(aipC30_q{5N?GmQ!|Pa+ca0!t%2z0g6$l%MkE;)(ju3!-0!gwP)e-r~)w`uK;8b7$fe+H?OnEgNqo7CXH_B!Ly3}?)de2;1!AlyR!WK|x<*r2T?!t?o;2lH-Dw<8IRB{6_hoZ^J3Yq|62SiFwV=i&xNXQzuk#W-oUZ{TcgDf4})f3L-PGSfc@7g3#WGyW_(c`F&dfJTA=|Kq1ZH#K^e*FPCf<|+iaFVLO<g8z-be@WoIIN5&&2|)`3MlC81p$8#2K<J%l3s;KMf-{|wmRO<V6z5ShuaNYJg)hGpXwj<q{cw>v?GRcY3+^<Z_qaE*6YLZ)Cy^ka;`>H(Fxj=c6h_;%CSpLznjGx>GauE+@qN4E=dnIina}LXGg5K4Am@z0MUS0TW?o~>{}QO4LZ_WvRU#nfuedlff8X+-|`lq;>d*>K-5qya`FjO%CNfJB*Pehf@{C&$s%^OariOYzL0@OR9A6)$&}x=>xF7$Z<YuVq|P`K??CJNrwph{{)dj;LQ9k?FVDtFD>Y4nz6e3jo*K>JJbJME`SZt)F6#2zWrsF4Vv)F9cBt>nz)3axuKV>}aN&VUP`gk*1KsGgpMK@wdY-*Q2p)%L%s5yyPRWoiZIqOM?>6bx9;i$aeyM`F3YJn9$96bKcURpcxm@L*Y!H0MM#I=-CtSM|n0uGYAV^X5Zirf#IZ*biuL|F{tH{;vZC^~jr%)G#{OYWka%_!ibe9OsjOhsCYsb(MdBjDubjMW~XIBN4@d}ut(ihlubJw=;8+Z5RuD*IdSZ%{=9W}tV^3)*A+1w%E!9`=|^`Jph_HBh0q7XKoD0CD8LxZyQf;iDkI&L8|Z#rD}EoG*@9qp@r$95J^J*?PHbS!oNIf<f?+Rz$sXeqnFR%g&f<wn?VdCOFZF3h1diW(P?u*c>k&a@;l%_L~?OtpHh$*90~qS%dE+Q71v2JOp*bsgU#M@=8gv^uDuM8+&htx@Y$wMN@YYz2a<CY`@~n+Qe2=xo{*mU-E=U5W`5)_w7R3fwhuZ0TyF)BmET)4RfCJP2kJ?ClWgG@XdkA>(Tz&4i>whm8(3D`S;PP4cFmQ<KS!IaE5%DJM|p5-s72vRnnYCeLruw0m@=nFd?e_$4>C2lO7bHs!{;xsnpZm2Wi^p9qG(VR++gZtR6h;`f~0?J!|`zsJzFCtIDaq~<%F{%N62kr0^MgFjI-DXnA(rF^}Oz_hbhchE%lD?6#n&!L2Cukk1_{|vU{b=G0axRi_|SD$IkBpW$$mAbRv)+<+#=(TH95X`nAU8=|K1@fI`MRCH=HD5%&@Z~GKxsa~5nJ2nomV|TqE!Q@?6rD>YUQ$OxpqsT{Ra>pn3eD~ED{FjJJJmgDv%(5P>orsKeu2(nANzqRR<_!7U0wIQ?b@pfX{&n52G~(;*I1}7km{bY&!`^u-bNqllFwo7j`Og7v3Y1;SMHpdLw&?1^f#cQLj1sl_~!!P4^}0r!`6XNnd?K_+8);IW&lpY6=J-BstK}R5u1@uaB;G4>#`H;U68`_m-Ya92d@x<(<+=nw>2T#^FxHTkM<E8Kkgn1;o2~8hl1+0kyg?FS>5p^=}x?+T`CnRg*<hK3_`|@*RvTLUvcKY;I}0_dWtv2opD-R2jp1=a;B{~^(mk5H{9UK;GvL(UbKUvL3u08S;23xrBj1cos_!UdP2$ku|0vDj1qhH+7AelS7F|18TTz5h-2A;FOio6`kpEAlM>k-2#UUr4r5^%<`VYNM+TxQT&gJwdZ6BG$!7%x5mCCBADM-*g`>5mRlG>}xA@)-Clq<L;WU(^RLYxf*8-Gb5k-PnpR0dcqlRG}Mh6f}fwv)1L>SO)C4W4)9%g-Aod~v;4wzm=OJXW(6(gI@@_q>l?pDU-4Up0*xeJ@N-e!bwqlF8t<VyDSQ*<fhCPnK|s(@4tjV?fp5w4PSP275ENPU6a+%d6wz_T~vEDeYQGx_D#MmyI!?nJsI$~e@*re5btR~*PVa^KOle{cR}?fmh_|LC0X%bmVj<NZx2QMh{j+lf=6NnJy<Dti^cuCiWO>L~>QN4Ez&tWGcYQ!x$XWjao(Uip{c-BYJ;{u{`g<MmM){>2;k97%Td>YKQFJ;0FONcGbt-#fbNsJFb~D#)wf(JPYE_3r17oXOSeUxj$_@0*|F-aw83r{BWhnUL3+<IS}{iygeI>YJ3(s?G_$>I#c?u3rDMbMWE^yMO)S>66F&$~b3&kGpcQg=YNKIp0C~m6Kk*`t4G^AWPY${30!UTOrc%a!s$rYUA+xe*ld2?gXp<Nz8B78uCBlv-|ynp>@Nnz+<_aw5x9;6YvCxPBZmxlNt(LdfDggJHA$@KYse;y9b}%|2%m9@Y&PnhyHTf`d2*)t7d!ae*v-sHKG',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYs3wcws#0n0G~P|ZR;<o2{FQ4U#>rxk8X3Q67ViRv<gMS?}RNP-Q3RVtTjeb;w6yNQ^X-7ID?tLw!qJTd>_KbiAPWC970CAIIkqeDdi`8;{@<ay_V2M^w_Cd*~8*`6=c>A_}lkpz?JG%1Q;nyogIJT0>IFgVY%>mtd6MKYNsc@d=R;9c_LHYv(t5M-M&&DN7;@IF~4>vX${A3S)_o9EdI+E`Dwd7iAxc)l&Sc>+~en=CJb4~rbydYfg-mmibqw#@QDFe%2^=`tyknW<kUAIoKWuJzX2<x<}E<khsi*(3#5NLG_{8E?yUX?lW56v=E-CgU<2<KHq}C0h65&*?^g{|Tn5zgLrTp}#X}djs#Rk-W8v<fg%5ft{PD?fE9p5S-@CjrnPhqa<*o>kIB`n=hfKn1-(X!Mi_Y>3V=aX30FCT+q*AvrJ2c)^eFlaWTx~pJ!Wu6~Gp1Ot#B%mQKrF@4J`p-oJYDIv54vCQq|GEpHBBoaf2G)$vi-`}*ax7f=E|-}VCdFMI|Vu_?o05F*?U7v*X>m~5cq=>&T_{D^P9_*j3tT0Z`9bQJ$;@bK^<{rV~#*bN?hGg)730p+meKQ9iRzaIQ~AytyKTjSf!MGm-k@M>KEy5z}$;vd#MhEjLEUJoW2q{a9uyNUoZ{UKFdWJL*6i4<1!DaN<!$tnr5JP5;nyv(kXJnB=8JONZ)OGSVYv=?8kLoAB#;$%H5uG4Z6g|U3<OXsV}WrE$0=SewTK!ekKa=k?OXWMcRJj8WaPHq6z%Q9JQaQ6V%v$7cTCMrg+vvnc^hh6k$odnMpu(k=V&pcfw0Ze`w6o64sJeb3f%0&Xtc{*99e@<q>cW=H6W*KaQb?~R`oVFr1#s~s+$J6;d4Gi8g5Mk@_M}5(wO4H>82*Gps@LP92u{_I2@xLvMX)JVNJO@lpa=@)Mg8oCkO-woH87K(Qi!wp{o6_t;^SLOq%{W_+bKJ1Ybd{P0Oc4b1c$}`&ay*WTWI4A+K}~EoxRkLeL$D{nk80o_RqOEZVdO3aF8A$SzwfpU-NhF;HlzkHG^7I9K|eSoL{SKiu~3X8C3G8@T)`rZ0tMtUo6N>oEOMLf#}`Q%h3l*wpC_=)NjM0O;(n0MYh#F$k0~rz)DMy+5R0SO=?;oR6R;`Lbx}^%(<Gt+41x%jW)RGw`?BB1sbVptdU0CJ5pGhFzVV=+1V=8QX|GID==HnFa+|!&^DK|TSL-W4MMlIa(Dzw(y$%Q(p$C?0ASmNWd8(W23}AzVF<xabqHLW`VP;>z$Ua6#@jym$z(w$%$hSa*%DP8JE@sE?`F&&|;z-eO5M04dn%|5N<N7YjNqd5EN+cTnId!JXtVpy*qiD>ZYyBt!WwcsQhsgXOpv|CojAVl73Bmx-PhQO65Lu?{#3mySWN2w%#zX|&&naz4#Y@1x@q9UpsOvc#s^bEv&$?U+Y!EH9EE2_LgVz(<ySj4t*IseLZ|s!tdJr_eDMrxuzOK227X3P=>+AvD37ruL!DKz-p*$T`1)7buVYbfkIsO$42R3E{1R@!0doae(LHFe2G)ZQKu4Qn(oxzkI!*j8PGiM1akk_{Zji^{A$tJ?_x*s<aps#~qy4bESfmjtl>fs4ExAV#La-86}1l9ycA#?NxSBfBfdIq2z5LBGD2~nk~h<flqW1X#+8I<yN-6)NPf-gn^V2rfbD1w9xOFd=8ZuUqAP<@8J<;Y;qVpB6%hI#mVpp|eYPS(LKHGBv2L#`4)4{Ts?ep4nz*!T7(Tl&a1)>hnV5GxDMK4Vf%9J}Y38C7Cn$bemGTLh`yvP@-u<JlHMo`moSxyD~Gm*@ny`Y*nW`}@Lh*fJ|*V1S7TIybGiZaTNBKN-QHb+4`52Ij`Bw;-rQt8^V*4Xh3igOfh7WmhO9)vEaxfcu9Ao5urEL88BaFxfSnV6!v_{=OLfcCtijptPTdQtxZP($l&eE5eQRhj(>cv@}5L-smb2R!D5l*`mz1Q<C}O=aV8q#(fR+KD|m_fDjVt%wuW7J0?>In3r;k;^8boxJJJ?`U>#u<X3bKeR*_*3+*lZB>3ggksrwry(^MEVrD3k;qVdkE`dS^&*ng`k;Wf|eLs32z2B4;8|3fN*Q-}N?HmORf><Az=4uo5U2dqgUm0Rljd2is0Kr*)z5yZ%0Jx^dzB9raOKB~S%P(pbs7=R}97uX}>DM+Q<XgUhBG@1}55pMcUnW#Fbdkxqj2^#4Rz4PC@uj{&c?UiZKXo6lI0Y^E8nBpcC^2Xs)%*9~2)O+;$DvwX4MeclurFYfev*NG5EAmK<C6OO3~P5|&$IQKwHkZP6_62@%5!iz;s!FFXG(PUltBoQo9C<y{Tj3=iBFOn1h=*rH?()Fwt6QrfNTX@;#SqdIw`NST<~vvnc|_Y$^`@mPk5G;z^miL1WJx#!-oTVpqGUQGQhAXmOPU1QyP{fZK9brKji{?mVHtu?|^c&_kilQM)Q;PO+?B~0VAH{^bCSH=!$?sWy>wL&Qk|k4D@+2n}MEVo{^M9OlZizf-(!J$`x=Go*~TEMGb~9s#ONc&$JK#Ky%h`P*p`D<1Cn(O%b3b3Rx1RpH_`lz(Tz-w^a3QH3>~6<-YB3;ZIqi_SJC~#?z;Ojd#X$0UJI6h0_?-*HLw1WV0L90U9>t_ELAm5Lh%^;L8E?YI@|tHeP``<&G*+S*V6>oeqzWPJR(Q4DipHv$5cCyG)^fAV`~%if2pUaN;7F<kLk|&F?&X5|wGWO#ZSs?*DLp`p2hd51;-}Jmdz7V6{bQF$t3O<Qz~4zC1bn;TZlr1-zmjg#$@GqM{YlTW0p3(rxtXA!@+Zvg$u>fx(>p?AQF|hri&!mf3bTUrzF*|Lec}2}d^xR-h_bmsldcJsv!|gD;`omlzOcU<|<4{sMGtdWp>OO}5R|mf%$PmI#<)iai3pp8oO4*~5N2Ok@@E1eIz44q}rPF(VXm2ql0g{Ob5uCnUX{hqG+D#p%Rco3Nf5btpZSjm|?9fe}_Q_fZwbz3F`3EGJW|F?;{hv9$#4xN2chd){jmy2?DkwZ%d)Rt3rqu;8gU1+QT50n(sWH4t72>&74P6pF&bL#%Rmh${T3e+oxZgxeGpuK93SO!IUDN=%$*?Mr#HO!{ZYbYjn&8jD$nIYL<mf`C3gsU%Y$2THR_Xqa!25zc)KQN0hNPZrAHa?8{C0f_x6p1(6@mvyU>GT-bM(&xJy7hW{?l`|At<|nk3o;tgs1#hT4=otZ#SBT$aya00x>lwvT(p6XbYVJ}O_Z$!@%aDVWC@B^yPex*t7@)2`9|NsimzAmE1spn)d`9L3IlrtEVFQ#UxL%+E0yQZ~eucC7`oo8}#ukBWdt&SjQ0gkpwn%cN+gOKTZ`NX9(Pph9TM_?mZXyt5(P|Z$ua3Y7XHN-3VbVSf@H{So_0htE;F87hOFqU~y^CluP4W`|PPXL&|AMp%!gI2|3Wf9oPJE;Rf`m3%;Xq+j*pE9JDb^Sz*J=(P8|gbBGHPIE;Fbnpq;QLEy>g|~_5)Ql2f87)8&dYxAm%M2Mn5!!+=k!63LiW}1Oa}v!j?O?1zjd-K_1;Q0v;`+&~nQVs7w(CoI6&8nwd=nLR#4bG@VySQ*enDAM_ZnC`loxA#Dz<in?J??>dGyVIKh*uJ~)SagwWKT~=y#V@Q&2T6E}C>of$WK0Mn`nQqtMnXa>Ik5tmDDcVn9^>j=iTNvgG$a=Plx9b$?Py|Zg_{+YTmQN#(S_SCRhz-=o7eRbnQ=>zf3c@mK0v;&T=D?)^okD3Ud@j=eA<EF%d~RvTRdeR*%ELNOSRLk0H?(9xRWq>3Qdc=xyl13A9|Vth{9etvG|Q&No^5ym9b1;qJgZ+!&^bYDxz(;PinxJ{9St)*uM)A?pj5+lM%ChRcVp1)jY9qn`e{R!-!Fjmng!&TAp;?-7ZdL+om{K|W728C+GwEO(8Oj-8rB3D!b?Mv&y$OE-G)SFO-EAh3z>`n3&*9jPvHO*KZipWP-mo=wk@Yfu$9{%q0OPvTf&nEE^x@A$tpC~=@2naDY=1u*k)k{>P?^&3uvgFhrZ5qkyoHy0(i5<Hv#c2pVFB+udR}_$uk(Un+(YrV{(}A_rdFX8)7HupMWZO7;k&?s2_mo%}@tOiUa0(rH?za4Vyr*te7F&cE~n~%XR8}JhKiB>ph>v?4yHf16o?4JbN8EVqyM-<+4fh=N?H8*NppPOzL;=FG-@WYV0IIe$2M9QPUjOSfi#Y$!i#?H9at^z^yn7z~wd>AQK!j4E?5JRJEE7yG5JPq|sk(RhXlIwc-AO{QwzliimT2*KgPLx%xE#uAVRk#K+ZPq8=4MK#u!f1?&ASX7@D3zH?3z0OJC5Y$Y21SfHen!(-^7g9RB$LjR#|3uF>`Qt(cG(K{?+)4jW1>yju*I#Yuj_0=4LkkmFR?JNZF5nx^Mp>QXTvU=Uttb<r+<EHI@U_fwuB#ICDGiWt5?D-%2G1wdCb6Yg#Fg=ZpH-!FZ-C|KgO;xuTfQ95e%ciZi19wq;%5H;XO5sS4HBw#e=@2YgXM@0~$j0P=HA4SJR_2b!sD~Vn_kdNUCm!um9eZ%AO^I&tU$AByJhGb1S7U=8C|(<utVR+PY<n1n0k0>l<)}F}NgI2rw7X_zI^j9#Q@gcU$Alc1(#9uEim0wApCe+ljN!A7U_Y>T>km2`5O_*`))ZHhC+Ir9feqUV7_ifIO97^>&Q4-<SNt?@LSjN?kw)fMtNQ?NQBo$=WSI_tRiwTl{j|tM_dM#HT^tU`!Xr2@ltV0H^71FN)wi1H9b&d9%gvZimysSa=6seWMyRCe)Vp_6Hdq~_z-Nt)I5`|3k;KDab+Gsw^;B##C)giM)P!Ew5yEpm0hYkS1p%lWU(XO;9GdeS-9G0JZk26LVOj6Sw+i1`dI6<aReKj-{P{tv^>9*VD-bNmJ1Me#O11j|xvIzN$wX~}Qyr4p(NTZL%4j1a{Skw!{U$5Yk0h+Y%VIG(`SO<|-@seiu9Hb#@mA*y`AqH@9<B$RejSfrDGmx-Xom=Ale#rsg$-TvdYieO47+wLdGnEW6S{v4nxbS=c|_o7Tt~iM8xe7VB+B*1bi0p@VH=tHM;s-4P~6WVapz!nz3jD<aUTs)HzwTSBS+Q$T=xD4+4~_XwUBig0|P%&-nE0GeAdS3Y`9MM41nDCs@`+()NG_jM<7@|Iy%9B9^pU#5&!v$%`^@qzEP^@bKOMs9!3p}G=`*N>!p#?s)4h9hZG}<oJx$j4e`ivWTi5LN9_}yttEvI{KG?MD;<Q^?d^q}w*9ly672JfKdBL?NAQfgSZ${X%%2CpLt=n>Hlh#g4n!gpE=me)p4#Eaa>L3x*3zX#d=FKE0G(Q)J*GGa;K?V`QjP#EXYUeZcvP#v_`$8v7HN}#jHu%u&juusUN;X`fgjuk$a`UAUfCR22WhOz_x2WV=8R&uf?8wC!k1d!0jz4P6%PCN@cX8k%6`p5T7T0J=sTkDab@7E`bAvy4JE956M`xFlb5-Jrb4r*LeqAC+(HaMd4sR}13~>!D7_(%+w(Cf-~w=AJWwO<z%A*LcAQ~N?be~h5%nq`;_BegqWo&e-|lQ>LQo>s1EelxA0HvJ+_MjBQ<`kVO|8<fx0mRlik>m6p2kM?6x#UD!6m+}x8K~tEs)3!Z7_q5JjZ!>+f)|Z4TD>FaThu>cUwjm&SK%!<=5o=QLBtkb_0~~#lYKijB^-UWA^Z{&=ffJK(XF2*zhYc*%S#5Y)i`W`L=Rqti|1N+ZZ<i?j=(WoY`S^L45@n_wfse5H~@-Wlt+H+=Ghl<l{%=5FQ(AIr^|{;B<u}02b;d5Uszlzwl6K1a{z9EIwigYkJ8=toN#C*Se!X-0hKKN^uN7TTQ^g2h-=Uvp#s{jxl7=Hi9<8wchCrr5*J^nqU+QM3$jxZkEo0qr=nLfWhfYLs?dn{F2<IaD0bBsL&g`BjfqZ(4nEcA$=?NJ%-kaMaUHf$mR3e!TZzGg3$w@C>Dyd%0*Agc`Qs}wbbFzT6b30{yZ=c0QwBX86%%($sr*7JcSTjgfUo4Rw<i7z^5;TvmD)$0ue6@BO1ig;)ew&vQqSbf+*euIsN4gvV`DJ1BimhUqGTL0y)L_+<d!1t*^m00`WxP8QboOj+t1&AfoVk0qC>Mz@1=y0dv`+v`7(D;q~=(-1fEKfT3-f;e|m#8O)rM9EsWmhi>wx!%*AvL_X5?>nvdtVXH3_)acUP%8k|^L0L7>5<a;jHnJ){p!mqn@W`rg|2|=n-vBTmrgyTfR@<OY{p>X?adWQLT6pFvk)l?i?l4ggt+whmJebkdKpB1u43(V{F;xPwD;)hiH1o&S3Wn_ny#4=xT%W#n!F?A`;9bMzt36Qoir!ygWN#I-WbXpyF8lWl{N0V)KQYD$scbH7d%!hUari)T0LKcnEZ{o!vr$c3smGS8EBihsx9fc6LuU3EmFPg?*U2><s>}gr`Hc-IjWNa%=|q<>!6@PbwmQoIkCxea6h4%*f4?JNd<l!LapVa)Ccu$<nS#iVkVK!3(=+3L#k<+s*Hzb|H?24nWwcsZzFv(uI}z@+cxE1uY9j2$>Mp7^wSn<wY2pNh-rIu^QHQrFwB1+Ek>4O`>l2zd0~olD6!%7wS1%5~efLcOOb2Y(>B7c|`j|wsLSwL`INaEFv8HV95q*V7h*reu#@6Bx)i~<2zT~G{71XvVcU*C`M)@QfI%ucH8C&R5wSmkpc*9Q6h5=qR%24jT5YQ5%FR*d2*$UYAx;U^+>;)lSjg<2lOgXBCfM$pDknF|4(k7FY+qszC`BRMXC{f+WPCgX&94jC8rwxm|?L=s+DG@PjAhJd?Ji8Yt7L#IeNb%Y&I9XG@*{IXdd6K({6QA+<*z@z?E&C-W$gLl!I$d2{fN#$p2N|gC@@t%^IiHpZ5Le*TfhaG@3sKhHQq@t633wtKf0>A#Mu)xj_&O>vAY$z~JLjZhOqVmHnpCtUQrzj4YSiK$m)>sCOV96nLj`16bVIe610XHX1?tkB-AMI!ZAM4GNryVN!AI7;8<3r?qiNaKO1ikadZG4M#vQ0{(5FrcYBa#fd*RXp9rzAv;M&)u*ZWY18qAPVXgh7xBC277PzE+H)JRFy7P3UuA}TfcO!#)wmz@?<UiP|;R9o+|lQy?vO#c>9zt!+&Z8)k5*>rHn$#(Cwu668Z_hVhd>fx8`N|v3>m8(y){O4JHpvp68i1#O;tvZm@UPH)G0$u_ObY%395jg=i*(M@KdvZ<f{bQ}_pQ1pwPby<O3?=Q6&BOIPGYi=bdrU$6;9?^|;$dUD`zz!hK2-N==Df#fU>n&wv0j_WQBuwMJ+iwjoxOIK77Meu{`^55;SPPc*2G+A!;1WJPv?f#O>l2zawXZc^BO8SlFz%!eP&rwRE3Ld#Aa`NBMc0h+T9gs)rqqdDw(YuO*e~utmsD9c(yaZQGnH8PgdIMXQTi>6ojY5aO?vFk{zfsW~9cQmh2t`nS2iBlM)VgLA(ItkU;!H6Hj5Z)oKm_pT28H(x{Axm9d(14@g5Q0fd}x*1+$%Zd0no*G4TbE#7n7Aci`V%5{CAQU?d}R$_=fG)FqovDsze=%$Q4u7qQkAYrC4PBj-=CGSE{<0d5#TlRG5_PBIdX3##7j3&-7f&XnwCP3HJl!Y79igBgWvwHwJR4b*`gYC>;^@PEqQYSP4pv`z~<h5_cRM^H~a!l^^B#j{A{@Xmez%6c2_bufJ6cY-_yqLLi1@=JFnS(iXyMhgZe8EO)5_6)MO8`LFoFmJSRM^|d5jc=A6WC}_aJF6ICS7J1aN+_f8@YlkYTwX|X1@1zjd8F54D$g)Tz3glRbgXW36Sz4cS`F23pW`UrH?S9#RqH1^jx(#>YSF_q{*$Sc`pR&@&t<nCpj($y*qL%GZH7b>m;XWR2#7(<^Cmk^;(O8C*LJH!fR^O7<^PNHCrAnFk{C}JqaXDA|k`oY`fYN;HAP)3WqLmLjw~1VqSFlA5_)GC0Kd@&92mYrqK*$SY^Z=LsEwgg#*c8I=e8-sZCnzEBPh#3qN(YCTjv!ID*Sw(oRRFu)eHL6ZMb#HN|Ah*A9K@-{EF0;P@I(ifNj1pd7k(Ew{zOvYuC%;-ap*1QotWU>}o%>HyvIv}3x4hc7!)HP2&LC1h~elZLU{ijXAlxcOb-dCEV|T8h=df7&=9Y>geb4O~LJCg!*4ZjKoN^*Mm%b{stYX8aVwq$LH9P@TpQ?zeNvhX^;5S&49;01{!?(GIM0k~e5;PG|N9()RAtCPxdIdL%y#bkiB1x4oJ|hQV&jVQdDJy&x~_0{8+_WdQ-*bFekcu$rpl%)Oe^J>7yTXffEYL5o)zwsSCyVf#1OXv=QqKJ-TUUUwJ&hsL=K2Pm<#wS#w5Bjzo8#GGu_X~!-)^|P*v)HKKWv`H`b7y)P5LH2E@gAF*=V2}cOynd;<YL8<p;qoIVa=O<M##QB#)VS)=@mG>bb`0h!NqOrHs_wPLX1a;C{a!#Yr?tqoQVKzf&MAZoHMY^)7uus?nqv%UW&|3l3y|&lW1Gu~;k>FdnrNFnwrsNk+zO!GmcH+~xb0zH<ru2-phaBCPN+WAM3SiSJHf%QRo}8OQ^KKLw&jsyTz*)KmG2(EWQgJkNI0hspZlRNIumOGb#u5Kt`B028FcmDo8CrdJze17tFodtWSRE|3Mu&kt*)JusUnyyL4Zx>D3g;6B@WrvM4FJLT}2b%7-lCgceC3URLBE0>~x)6Olpvm?bS$h%NQzO>v*b_o;X3j&B187c1Z|Z!#=GM@z$`ml<6JuZ|Ld9E^PY3!rH2)QmC|rns4tXNEYw7wo!6ok__b51y-Fxwk6e_S5d_1tv5zhcr+%@z46#{W`<I_tGd3anxSAo!CohT-qV5F>4t%czz^-b3CMQ+BQht1E=-{tCte;?f}&4eTUSi0+r0fe-D#G&Av;x6iE~5AK{7N6Jv0Kms6xO24MM^Ylw|w?%;Dw+a`TZ?cvHlJ2`CbRX$mv3$OnBK1P<cDSNrkKkTy~L>(w0$PXUhlej1M<+m=)LIqcB{l6}SGt}Xpm+6btPK-OqqH<xz1M#tC&W?+&@i9Kz}H{~9FDSPXVZN*TBN6OxQC9WTbx{T<PxXazor^LTD&y7!VmrIG-hHaur!7Spz=X5LKZ#x}OhQ|NdQLy(rA_O`r_|GM4MGXeXuhXHFKhq2+OBRz*Kd@azt&?rKD@u!+v`f^GBUhmr)za5k9nuc8`sXBh+NzLA68vfv>*;KqdKMeXBf9{{{0-$NtFHQq$Rg?464HvL0P&};a@M{zca@eBQMwBI<1ulzUEi`|B&4I_dx>JbNl7%9G)H5&$tI$<EUf=|fVbnIy#tsBnHji#0B&Buq|xpmwTBdyCETN651bwVU;ylv(t<2fiQU4;ElM9|nPn@j>5+Zg4WrJexox_gUwIm)AsPQ@*VRG0IFg!l<35r<sTm{3L2)nbX(wHLp-~Up!fMgq0`)}wp(nuevk0!$kptrnDr{Ky<&4s+LqSUR%ZA5$I29PrVolU%EbNCnjnmM1Uk#60-D`)X(QC>r;ZHfnSaFYxWBcV!smaRjqazGko~6FEFaq+h`su1bD~%r0$GEFdhdR`^_>o^dCb8#_1{!*9iq*3??1q+I+qCZw4Rgx+Y<i7d;mxjMO>!vf?<9t_%xaZt+h~BQ1UB3C7}XUVW`&Zx=q`BMTi54-O?u$DhL0pX)#1or@>;?apjM78xy}z*F8mO~O?MEx7gh9nAe4>>3H>y5q%bZvXWjINJHdMP)qs)o?Q^W7FX&MQ2cQKwiL?6+<hE!DbNzXs0H7=UD9|4R><b!*z~M0;Sl~qnw2KapLUT%CSv<gMjeb}pCKNyu9IikdXJsC<2H6Ue0AN>D2pO#^x|^Gm%WavcE|J_0pb4B>!8Aq9$g<XioAJwws%B4%O49LKIBgR)8f4$~7ioltG`QuxnlXpMndmKnc!PINf&%D@q8Ry($_@pl<lYu#pQCN;?N3}ayW_rE$I6@I9g7i~iCD7if@28`4^R*7`deVu-fqu-k30@ZR1jYVk!cu}F5X(k60Z6anU$+L=@9E`@B&3bUX)&4%0HjZs71|+eK%w1T<HdBJWoAdPtVhMy<ElXbP-=<SD|05Y`R0xMO(ZlpnLQKarmFmg{dRi+O<f1<Xr|3u4Fj}d?HnNM>-GpSnrO1B#)G8l)DrMc3E~U=NDZIsQ1DiFb?ghAuxg_iZ~tW0((RQ9gef+0;`U7Lhw$>Tn|o-lhTB_9MzPCCez*y1w%Ryx{&N=9Y{9p_H+N|QeFI{VKJYh!e}ZJI;fJa@zaqO8Sa;B#GeLN^VeE}x(4ctJ79hLc4v;U6Vzg?8?tou2HXt0cGjF-=Tv*r-g=<_QRgu0Id1n#VSA4ptJS8{UsimJM_GSliN(kJUgz4Gf#sfQTwA<_eK}p!qZ+OR;M|wgYIR%V{+VDqlTqF?F~&YZq%8Y=^2+Xv`xL_MNhzWm)sYHY0^kPXp$OP5mov-y`{Ufa8cZ_hr5wgeZs2G(Sr?NzU^CqE1Lz#~`#T7<9wn;oa!7@7rAH2K6yAd@P=b{j){N{)G;s>kxc6O;;`%WsJhDj<sh6CoQxmUA#hDW0$sP{k#-l<FnN!yX6S~+NlN?mO$iSF7H^aOBRxK;(IvD%c)$$kKXXP5_zd6m*`Oa9$rgN=T;XeMgGT@rB{64MU&qcZ|hV!4?OPbw~t-*hyC$f49yO5~6I%%(38dpbAxw$x|IwgJ5I<|*h--lK;=ehP~xq!!!n(u*wYZEl7tGh#`@Q&SVFN8^>gLJ`UiWPP?GD5M)s2~i(VK1E_e}xxp8rOfg)uFok4DvEA>6#T5agxb&F&5nraA0E%PoN7}mdmi)Z-Rju{N~-8@87?CH~#SQ{fF`I-@JHf3!pd@l-+O?9!8%pK)@Icl&W%OM?n}OMn31OT4|1V0!dUL?QKsqd?EF~_h@)SS2awTb*J42Nez2;A9J5|tnC;KOKy5UfV5A)!TWiZd}QtDkv({`B)GstQhb<8afqhq4s0Rroys)C7hEr&g{it+6bcBsQ^R`im(YL+?o%y5xn5WqB;jX{8MIRC2z-%oo;r|WQEp9FQ0FkEawh3x!8%#g#_%N_(iLP1VqFfOX?hk`)f{}o1QWP)I<R58u2hOZeR7>e*{z7M{%I(m012(%omX=GF1e{+58K}<l;V{=XvE?k7(e)J?Q#J`47<=O0S}3pSdU+GisM)z0-CS{)I6NcKyC@*!vKT#xXud&?r>Mq2XoW1-IBRAu`x`$6s_;88DqI31z{tg37}Lyxu`BQHWD!3)q8q$hPA4K2EXuA1CJRM6qWUbgUsk7ZAzXV9G{))UO(&Gw<cSDF*QaH_&88IC+_jFKhNMf(KUgiC!=QbpsOWP{EgiA=O$G29idf#W`hnQ+Xmrn$pK&6iv>XJnGC&GZz>$>IkkNZbv<=v>yTJ^I5`L44$qg9^<}7xRE@bb4%@>vV!xhTVRC9rntnaW@p`0Ycx`PaRSk7(szg^$b7!qEVGhdBESUI*{BQqby$EL8o4^0{U&`Qe@wfj4+R5eL|Lwou1pnt>|L_0&`(OXtG+6)rum2^5GVta6Z~rxgEU16~xBp$n?iHRUA`rn@!%1=tLu(X7!sf8+mFn=-ULYRROVD4TnI;*t4;|U|u<2V`rlFy=F+GFA95_MiEhXLfwfh{WaLrx*0SX2~An}?L)M_9wA@7onRa1};%zI=wYj=WLiD~)rO&49=>jTBbtaitr@49j1GxxetkVcNB`O-VQTa&^;7h6cjXG%Dfh1`DeCF;hGzdUN@Wvwlms9Y**3a?Vla{UmURjT`T2Z4QSS3NB?1*xiJjK1rTWzxFBK`+e|*I;AL^VGdg&IM_T?FK8w+f=K6pPav-Z{NdEdV5^G(yjZ`KEtkCv1Fd=wc)wBsHWeFiDe_@jbFK>(@RR;J~osi`=L>V$c#;^d?ld0i%5|TjfGaJ1`}d6+%j6>HtH+q)nRp!F6(-0zWBg5jyj#V20j(6^k_JZ`d#yIFe6fYwlTI3H|dq>GHOz;opbe1lF1_9Wd~-zRI{o(mt!;%wo00FnXg!xnQwK@E-gV1R_+I7V;ZD&I@^q=s~Na<3Y+#@EcyY`JusJa%KQ+$IFkqbk<OfBh39575NG7)<YB%$QLZ}2s<ZaY;>~-tj7riNt%@Xr0&OIS7{Ha4^MkL#9cd`cUjEcH6hC3c19#DXqI(=LXNQ@w5&BI!kCyx!lUt!%ckyO@QO#CnhBf<ml_wsnt4pv+Mf^~RA)O3Q<Lt5)Pz9WiQZdMPU=1U}J#Mw#_wSv08WN%o@MM8zulyi%gdgI4#E%1tvV(XdMi(#seL`-JW*&>~q}pbNvUx;bk7@=#8}DSaZr&58?J=`Uw>IFfjx1iccit>rJF~v?g1bsKw2%FXm8o5NZac601J)|GiX&F4_XiF8k~fQjD613_uEOnX*n0Cpt)NP0zCJ!HvA5yD|9P@{TQ_&L=iz1tJFimmQ*`UqU9Wn>ce`GzQvhle^xCAIM)cRtB#6$1H}%?I?wv^yuEo(vOEH|fo7<e}BfjIa%AbK=jo?E;7m!IAy)q?0-wBX0SSNc3j(Ryf%hh&;WutNOW^n&_g$3!5)SB({bfE&xRWT_bRD7t4?UjcTt99*Pt6NV*derx~$dY(Wz5`Jn{xSOXP_C>tu0`T0@)vT%L+J`+rd*-g|KWU}7;|fIEG^QYs+X=_p>EN@!zO;^pMd7&tlDbN%GtAh2{O#1VJ)kol#{PVpk~e%tb=Y|%I30fd)0`%9#G~m73<O9*sGEza;hyze33IQc`)eKGPDGb$Xf}s5y8U=J;<3c436(;&dsH-!sCxTHf)-Ly(T_8k}u2-n25Y&OVZwW*KFnhMe1GE1ZL^nhCcbvsW+d4Ib=N+byLlR+-vt|X7o1*k>$1ZQ7nfbarc9ebB`MByJv64FTQ>AobOsQ60o#Xk$>&9y9LXCMh=&DGl)#y?_T4m)nqV!?b+|Xe)IZ2jrW*_UFrGP-#!~Z|N7Z~ZXdKFw^i1){j}O@$p28Ui)J%vZIFF$udcHzt3uIp2+u;t*R?u2USVwXav6vj7Lw9qC19TE#&Nj0AcxFLKpc^K*^~$8;5-fK>p`jh(cCQ_pm$qg-*D0JyMz&;zJ+`lTf-iS-V&cuzf%{pa_~44Oh7liL{kV}261+0n~LrB9P8K;$Lq2fIG&2D#@pW3oGH|dZ;z~tW-l^`H7=1`?>Ei)_?q5Ua0QT<axhLvVRd<M93RC+79YV3BY6j}4vxxS{(>$ytlrr9Ikfh9o`S3R!AY|b`E=4~%l=Sra$OXUpdoDrAA!OT<l83yh4T@f@vGCq#rX<Ha;&A``Pek^@$y`>d-h<>+oMKne?V~=U4Ij&(U0bJ)%G7|6ST~^<@nukv)y_65lk*n_K_(|ud3<(0q(h;<yhpz6rooqjn=@{4y?3OlwX}#fae!=fKM?t!mfxftZMXFue1R-T5!Kg^Q?hA?x%waqIcCHtxgyQFKw?1B@M<fsm4hzO^HKIMVG{S-Rh<&$G{z<FTVLEk|t^ijahDzyOx^`$beJ;YqTNzO^Wd>t4Vm~+OuhbewS+p&Z;U+Rc&fO=Z<oMmxE8if=%grA(SLt#@CA!Sk4fsD2A2ewOfr3R3-G@Py(Oz0H;!IZnN*07Kz)$?_IKP=KMI2aU7a%gDpr;AR{323$C2t)q)5Y(^Y{o$r?`$2Mz2H7cVq6jBKXKHl0Oep*`|snAy&<26XUc=#4fb0uNuiE{AvLE=QYh`qiZec}97q>1I0+4rfaY>&1S2sfVixV+nxH!Mb+6WwO44jkaF#tsi7108q%=LGa?`Z(conJ^t;xH?KdueEkAld)IXL565}|z`uDue)i2bqY#exo~zMD2^*Y%<sgh8*)d70g^2_7c~Ywj3q)o~Vpt3gi)12S1Xe2r#5+O89joNqS8rd^dyvBH_wPTvc=PRt{jX|A_F46RsDp(S8HK7=BMFZbLvZp=lW_lL0jeVl-h=wx@6Gk&C^<R2g*}=jNJL;<fEJj$3FHAv*&R;8=5Q$n4hs2^5I!)I5QVyNnt<-(#)3RC=rtxDD-3y&M&z@14@h~YZ>Y@Pe=|5c07*7s(ZuA5g~*7Ybxh!10AlCxtcH>{I-U~A3t8ucgyZ1HA5mKg*_vuZL)f<_*{B^UUc##7q!s7rIhD$1eyAAY*4z%KL;x`-2YGSB@TL8;)@`vjmKu-ymR;1jJMP;{V>7jHl1E1^QP9B9khzh3K)|Xv$w^c4T;C6WC>};Xy#JyPts9uS_|ld$y((UzFlR}lhz!%S^AuCKXmre;K9JH0G+$wRdUeGw<qqVzJ>1tHtlo?z>DpbdGdiCYD=Sn(T*YnW<#Vj9P-LiIr96xJwszQhHk&-sZX-X-gM)Q;Aa4&)CWI`yYf5JJP+L74RGOi>G~|)7&d5`W9~#n=a!P!o`dJ5WW$2f;qAz!D2lc}98**8L(AvSVv~=ZI)tWLO*>A&rmG|sT;41GWE9-t*`~FWTW15KF&uYpQRW<1!h6xzOz8f?9&`wDpk*#vFo`azb(&K6Nyvh$@g!%m1bK&iyzi2vos85IS_vrKm;kIMED<+*LTK7Fa^2o%~WK^{nGaA0(FM`@qE(Rysu^#df>_p6;h0Np#BFn<=qL7*$QTSk%TpbjP)xnk=G`s8M$qLjjQ007k0p2GU2b*-Ws}6fV9N-NbqvMmWjt0LvI;#BHs%5NjysafHd%#PM+pelwc?Vj$7$V%z`J<C?chf+@3LyRG*~e;V4nj4X;P;sEfcY8r3~?&CfT`SMpdjG(1$qpG3B1iiry%;&OedSkc}fP{@FD}=57<0}I2v8ZvDI-@06GNxyN0?|y#HzV{^j`jv*%yG9KU_`;cMf+f@m)WbKIh_(PD}ySY3?cZSlja-@gPdf~glUX`w{2GFp!mqbZ3w^t3@v##q#a1s3Ec-nWh8*dR|lzPyH5KOzTcVbg4{7Ekm1-OFbmUXEYC8Grxk^@}&(-!)KSWKe+>o9b`NgYfLONG3CqzXM&Ik><5&ishf~5!LX1QEy~63e_BhcavI+^-^+s>wwsltWpFjiBh~n!NgYme<vkti{<9_>vohBJ81^`Rj5SiC>zZ&a&(PHO2|!Kp$%9}r)ND{Fsnld7HKPfz?!yhy9S~CQk4!(ZxNY<MdsUYub07tv~Q=$;M20^dJ6k_nO-c)`U&??5zQ#gzhhrY)s>T`S#y9CnuQFfj%ss^05H~8R5Mg|R>gS}gXY0v@>TI4z~<Pr2kG_kBPVF%+cJ1UX2+-0(1DrvlS6*}1iiDK%H)V6dU8mwLZ8D9)gcOBYcAZItus{u+^wKBVKXfvP~6QU*>{PDjfFHjuFAqK93DB^)~yV8dwOkr`Z#)?eEc{l!!(*a)jPdC(BrXd7WWxwV+(c+)l=suANO9U!Z3S`nhXpf4}8om3zD@%F2>8Qku8KF^}OkA403yYTZNR^39<dGv7JullXzs;;dFG2Z#>zwPkgzmn|GSv6+y7CfKORM&_=eTplan(4J}LPhV#b(GYfomP=Wk7%H;$$L(3#suG8?RiDyiv>RS32laJcDLZuo0r_mve{MGHZs95<xi$-1{6J1&tjoj<gTGI-3XMwcr$DP?4^?uuW9!5Ukev&hymNag(Bk#Ja3UYANd8#36lc(BAMb(Y*_TPoL+=FHFP(ym?-;X}D-e!?-T;1N64OqzFEd7pLo_j<y=znm14dvSLYGm~iRrHv|Ts_foMSAG+$u@^c%SCm+0FhRVx`!InA(-9(6F2a}0|r$b-d+xC*M5tyhFtoMXmD9YK9i3S`i}4w@e~t^?y0TDEU&wyMH5j}Qb7APk$%l3Ep@R>k`3nEiX?%_39yJJ#-`ul$y-~qVkBTO0OYu}5fV1})Y;xY#QpRhYP0*_huKJP*-RZ$rMvXhZDhOjl<_Z;9a!C&%A>)u__OgfsnY)gCW2^-',
 'pubmed_efetch_snapshot.py': 'c-p-^%aYs1c6)vWEH3~^9L$XTsB22pEm<1htBNhDr15<e&7eRMoPh@fcmZ%m<GEa<vdB8Ce862~o2q0Jdy$1x`49g|&ZE20-Dr@M?Zj2qkU&39pFVxwr+M$*y|?LKHfhmhN$$Pg%)Uw&-q{yvGr#nHcXsaGWX+}b`l}buy>H8`sJ*0Ec#BPy%<|NmCD>|yna;1)@Up3+d-v`Qvemk*8gE`-50+KA^5$ipr}HK&p|^Oqz!~}NB5BelTczri_&)UT-|xyov`be>mPeZ=%hf3Bq^eVFA2($z`^o7qzRT9~b5_bv-_~UzKd+MJQhqLLo@EWs@@yt&c&#<5)X){+pj^psG%0o7R5|pG2rN0mUp6Jcx6U{95Z~A9JZlE+kbs7|snvsbt2|1ZDovxaJYB(_oWo}?slBuF!C;Xtu|eFV@0y?}7il;ec<>+K{B@PqX?2}?tLz<YUsmK<k$TvW_OIMDUb0wZ#f4X|lX(glEb45LdN*aYs2>3wO_CMfEHCF*-mKgdi=@g@-c6h#TY5O3m(|`&*btr;|AUU3s_=ZDo>2eV8+&*D+8=s8AR@uPi|iWz<Qe~4H%D2MuJFsO^0(9#*LIWSS3%0_4%<1_S&9&ky@_}i(YneCrk$!*OZYy&%yL}3CmOb6DCl4qY85rfMO4vl1pXF4ISnUAr_(4eZ_+9VJ&BFl7y|G>PtAc_AfsfxPK!n0e^Y!@_~D?x1+A4J3B&)RDdak{oN*O)cP+vr9dI{(Enm^-{Mz?qyQFF&Mp+mV!tn{BidI7uDc<`C8?J@>G*z~SK1{-}c~vrR;wqg3^#%M#>^Jn_@3=qSc<lMWkXGs9Y1L%&JoQ_w%(EszmI50pdMq<U3ve%_{b%B}w(5k(Knf5qMHe|Z0j$OaS})3m=pE<@4m5shiE@=KVmcF@;gDCp*H}%~<f$A?Kxu+@jxBghxr*IGCbOE5Ovh$w^=tb~f&vcVQ_<MxO)YRobuf}V541+@nd@1iNxBM@2sR$+lIDEXT&9(4Vgs2pv(=P06v1!jS(Nh)9(bD=J}=9w=i;443Hs*AuAR+OW4_1cS%Y<V@4!L3+oyLtWd96Ec5R>wqF=Ou-I<*mh<gC5Q}RshN!Usp6p+z!ljkecGlI&WoF08Pot(fw_km?&7fnu7Nd_qSLy~XOv#Kg9*#4is|6Ac*TxQ;%zWUN@F0-G$|E~?}GM7L7@VB*>|MbH@3A&(t)B$QYmK)SUg&BItUwCJR@c%$cABm?Su-oh+<6|Uo*MgZR#l<GM0R83)5QHaQw23ZKV4Poy*C-wRaA>_fzg>$rYP^f8+^pjkCPN~!)NbMElEXQy)ekL6SuSpMRJ2HF7con7Xo7Df-0*}!ll<Q3F$eYRqO5L>PA~J-Gx2;!*F^?GCAp2|r2H3F7pOsz6UomXfYkW<<<qxsU;OUn*>j&yif@TZxtmg0-ub*~{z&W|YASLoM%t$N4Q|1RHgDLG(Z+v10ZP5$FLW*PRV=}T$MPc%in}l(#BaO$06cZ5168`r-XYV`(E~aSVbCAJnS9{Em_D6y1BC)3ogp<+l6VobJRl%KopET5M>fRb0HFLf4}W(V#%$bLQL7L0220-y?j-)VAuUIKReMWIwYHc?7CcLXmRrd=h{6Gjy2hh5bG#*`N7sHP#43j7kW;Ds@Hm%A9n1H4=8M}vkl=(je@a7&XE>^)P{DmbDsJ$A`$ZYI_z`T<7@cqu_^Yh0LE(=HIMtHb9HT9n&(n32E__4v1Kl0?Di99-LDHoDBTp>SSF4l&!n>9?N=n8*=JD`(3D}xGKwBMbc&>kiw<$7lBw#QRNO2^)Do&SZn!+yJ=4Ao^3L7e2#0YTRDv~q*@dsxw&tIJX*++be4$%fk4K(~+as#Es{KtRyxVLw8`Qv|p<@DeG{6GH&Qe%#?<6pd{g3l6jWr>@5b{?S_hzGE^kN^sG1qUYn+%i>7X5s)5*hFM`z($dEA^@dRboS=Wt2e^hfd)L>t|bTqp86pVfb;Q4>hat=RDzafM}7MPTi<@0>PNOS%aSlft5-Ps@%!Rp<lV_Jw~DyLE&&c|VEy*4iAJcq0c9*UU_~H%<hO9NigK0Y8Hu>N?U05D4e2XPK8ekd&Z?B%2y4VckRg?uhQ&~>992g7aBC?-5{Oo4pcFt7-4yA&H5>w14{orS5J!bMp@9a0#oy;8I9`g4rchQEvqxJQmTj3$(oR-Q(|359we3N-0X#f-4OwvwZx&(#!o05?Lm=w_h70$>FBCh3dcv&4&^PI|*aSrcKL^heK8;)L15Hj(QxmC~6I)~#fy;#S((q95+5wVvwg*Dfg3-u-f(@qvg^Io<;1-O!E5_{I$H#rYhl!&)E7><R>ep>-zWqf4OBrem-ie1D<0(#0YI+4)yH|j+{=+|PyqQwr=Rf`@Fc?(cA9?eizW>((RIs0Z_{SA=NPsonM8cCK#F*kGYP$x5!5_}vynXTNC5q8Co%SP^CP&w&{^0!0)0b~wzj|{X_cnv6iW(htN7oO1&-)PViesLMsEy<=py9TLuPg8+X6Pj)4{DuU1M|B+WuGY!&*1Fk^VhFlygbKgE}LdukB*O1@^41Pe3nI^Ek;Fl8C{gu$8gfC^gHqTn4yW5^NY+MJU{#5>DOPLzojjw!Q(pn4tM(W<OFsFTzpq)h1+rp-!76h?s-`(@JaLu(n14twr=qICzi-Y*CzN5kvL9IqM`TMNrXf)+bk|nGI$@I;8RdFirAdl8fTTfi^(w*OOGc`hR`j+7?~yWt5}Ws8IOr43)L6c&gwQ^Cv~0HJtM<vU`c?Q>oi`1y9=0uKr(^|W3>)ycyQ;Z1v;-6l5)4(!B<cJ6u&ux&42m!EdKn@=Vyq`(@(wofWwC$i$4R_LWoHsX~VS3>>}VlLZ9F-xE1&B%Z(X&_wV!bt(3TELx_$Oeb1($G{XT_kL<U`JM|uqJy&CV4tson8i0QMwx_q+)QvX-zcMNxV1t0l4V>jPieR+J51mj-GU|ZXX@F|P=>b4(U`Ihf!I8;tJ#=aC2z>x3GVO(+ZQ(sW2m;xmMToa9>kJ7=B1S@kEchw~Qw?Z6TWwaP_Xoyt8p3e`og|RQ1HPY}hEqcmA*ewu2Od$()!JZSI935nIrNsGG&bN~6`PMSiYTkWYzJwew-2?I<9ArH0L>CAU8U^>3wjykxr6T8NV!4i#Ct8pNgkk-&pa&x@i4@_eu$8uL1QiM`k*CU)AMVj<rBj4HKJCIf50Y6h>J8`)LwH_dOUKwmBnT?1Ipa9l@=4jhyZS;gLToXesLDKdDqZ8Q3LMaSU9*zGx{muop+M|uSf*wO%c0qVP)z)akC=r7&=#v9Y5N&n-U(mDI8iPc7@sxkLl5v9*^k-CsFiKr~%!3U^-&PI6mFCM2!c~p4CzD5kY|%DwR|y!IX1qGHFUdDfu;Cm5WWD24UouG-g-Bbms;_PLV?u?a>&*jsg5H4ig7WVM-Z|q<8d++^pn7H@vRLf#!D&kAI7<R~jV3XsN=!cy?LVO_4x6_XDLqb@Ulu;|24}U5FO(=&#>=iS7aD!}6E(z{qr5&{z>UdV<qXV4;He@Vp7GaY%sx;nG2Sh94`gKwtwm0LSFB&tJq(U%!a|?(EN~FVpw4ub#g6Qumd@8HTxGtP=c8#dTIe*aH2UIF9M01@v%Z^IQs0@K02w;GqMy$?c%~gket?Q}Bz*w{19CGz_hlI!18Z$>{Wxlar}J2cF4f39beyw@lpV#wWy-H#x#b3|vQ+f%SI*Ds0@C+!y8%lrPBVbP^3TTQz8Rdg!5Rn?7m??%2ZCq=7R7KtSppsa%4nC~pFpP5?Os^DvB>vPp7<gs@{syMY^euddSy5OQO~s(}ZqWOS3gkb&|Ze|&PHJVGjJk5uK=<%(dey_+goLu4NGk?SmlFtieF=)m`2gh!sR(WT{#XAWo$cC#k41NdaMN(%HesL(wp7F;%~T&I3G;`L?*hx_s5@1IQXKS3h*;2&%kft6j)6OcO6XSM(=Ez-c>G|QvkU{gVk5X=dqc~Ym#GG7F(LkR{?lZu$fH?!ckBLVhbH^mj~L;-(m@NNNI=_355egG}Lxp@E|!n>2<!!3MO>-y^kx`74$%?zeR*uzfp0V(l1+0V){4>~XiWd0TZE?_SFjN0EVd<Gg_qe#}KK?X9<LC|=gBVYLfr1YjrgEyN32hcYkH#&}y*y9)uWxgDG=`|!*fDWYdiVb=5BGppL5EgW-1w2AVSHn71n2sTY9=q@4kP~4oPeY9j<;zIEM^BOXXf~juPgy%{3y#95H?Vd>rZqIHKOpKLH)7H0ZM^Ln0Z;@`J37l3Z9hIZ4skfr$(jFPvJG0qLasF^yD(!Xh~p)oGmb;d#X+y&v2Fy$bpZ$M5*(E4TVK~5a0mz(K!OS|gwg&H%WA!;<91**UJX7fbl@0JM);H^GK~dU9u(;ge75Jcfoc-5QnoS%y5sS49udb5@Sjit5aH1U3nqOB(+Vu|A5(17BSq~@mCy1_<<B_idk+4VJOyS5hiE9&4s^;w#Gx$47(zCrBv63rLEoqwKbHoxYCz~l-CquD5hG7RZe;#MvQ>Pvqv}#;kKS^NAasrvU8abt8hq|3GQgwvHKbD=J-tZDW#B(8=7^6!!fC9JSS2}PL>-?-C&&<=QY;1Af>V%uUoh-3_=5edHRT2Q?wGVBGLV{BRgzXljLjm<R`bp7u)cT({I)`Z2n5z>t7B{(m~(`D%-T+9hD716xFw-eA48o8DZ;G<u^^S02B=O3if-n4S*HY(O%M|E+#(!kSzu&qC5u}@Yzz2|5xH<Quwm4YaYWKGBKaSBPsXa7e6uQ{by;NdAaof==PVvf{T_1VV9Sr+AcH)5idqwC2WV(A+m>>e_5Vkt1%VEk1MZaT0ROvzT&&c9@UguDTa!Q$fF{q*mDqvTwwr0e{C}=hx)Gu}Pt$b(23>F(4R>$Yu!TI-Xo01opaWrVCaZOxf?pS%t@!u;ki%g^Psy>d@JJh?Nj{V!hresI02Z)3lJZzBvI|&ofCnpDY*y<!uz;w6AIH$K9-mjBa)K2=ijuBPwsQH-7yU5OZ;nI0dVG%Ak-;LvObuY`IC5Yba`nWAKWM;GyqFRd^|3<BHQ{R>SBWy^a2M7QeGN&ilbXP&#U^_84H-W&5o^t-MEXdf8siGma~^~Gj0YcHcO34}4vo@SdJvL^yd$7@VQWCJHC573>{;6|Q0<YJe9?{g7qnSQ?#ieVZ|h_BdJvc0I^|V#Q-KYmaaLy5tukgzG}Sb}_UNPcKX7n*3zkdRc}NB!XmYUlO29(kWBx1=72snm^zRTOdtkn%iRI?uXAHAWv=S&Bszzv&dY)zC0AOg?$RuSRVQ9VWs@kJTQF!Ocyx;O{(glWSuPFgxs|Q5+nAP*kDZd>P%?D0IUd&J^Gs@X=X+YzlKp0AActSxke<-s%C>)Ix3s!_J`&slqW7^P2Mmi6MtwAQdikjm9&mHEl`{V|+Jq#EqgGKwg^G{~ebiE6qBOi#?JW&8l4Zp+$d$oaBB~r%da0}SLhY0Xeq$$eRMz*1RI_!ucpdn40oA}DhrdjgLdO*i)MC#mKi$7E@GgFChs1~~6Cs~upi9F5FAbuhKZB0a#tjDf@3I#(lQ5u{M%_V5Tga_b~?QDup(uUrNPCu7kbktoXXes(I-94SbdA>3QYvo7<qv<rgNoh<*OX1hhg}*aJfW4Gp(1WEeH)5Bf<=pwM=E2qmH5OvMN1&558q0J8U95LSK+k`s;K!|z3)1Ajd}Lwin)8V7d^kmyCGiFbpZP4C;F}EFwvkhKC_ZU|oD)~vz+$8veIIeZ0K<TAtODvtSX+zL(+KE1+@%t#EU9&}LFK3a<kTCmU}`*hGoOrv;f@nTBytJf98`h8MNjTO@}R^5upx%zV+?}lz>LI+KN-qgf$nfl!tT+6#T)~N>9S7&-lGJwrYpSp>AOq78cbwsw`QQMfn<l2)ZA-LjRisfFl+-_#vxH5weGMVdRj3S(A^_+d0%D=5Wlo}m;wc30{v!?cK|#D#-U0cNSqWZya%a~kOA-J5*AZ>0HNAWL(W&wjWiTHE#0%E8S|f^#Kc(qJ?Ma98VyojtQMklcG6*3oksQ~3Cq2eDlJkri}*>5YPfszEkYHN!r7L1f%FMzCvq$^7%}Gqy}KBq8!Hf`Gl5L+XWZnd&D?fd-*iA}Azcr>dt}X0>l>7dqTrlz%99uc6Afbldhn@CZwN=d8%I$%AQg=K(s2C_1hmYt^%Jj~Q!&~}8M=S6%N}-e_+5LrH-;zCqO4uW`}ivR9PQi1W}a$!6D~HO$7yY)=!xQ4qXR&5@*@j4SgOvUb^*lfxvd53-8CU1nDMg?%zeWi=4r6?Uc4hA@ZrQ1fs>Pd6zCf-x?qCl$0xrTdLKXhY>WI3LRiSq#p}o3r|=yM0Oa`}V|avAFL%hLA5S(Rwa}0vr<MSa^o~J%LOAAQ$^}H_+iyW3-U0CV8d@}%+c`<kUb6JR<Az&*a4=0T{okLTzxHI$N5I8Xk$GkjR3GVd)8=3$?!;>?akqn1rA&0_Lq^w>XZ8($IHTtvY=)j5c!Qkk*EqY3!)b>k?A2cN{h+N7evYMoN68eZS3)2LO`<s7LP{7s*g6mbo2BKmK%-&nP);Sh;9GwCr7IZdq;HGb+>8!WmPzywpVw7E$_9TZ$VNuza#1uRi&rBOZ8eJVAb>1*jOl^s!F+fq&O!`Om=A<Tk5JtUq}asIA@Zm8?eN{`?t$A+X*`8!GC<_e5lQJYW%}sXVs!Oo(&G{|vin_Kg@=FX!xxdTd3-0voQxh$x9ZzAy7<n0#*K-Srd(z7cy`;QweC)mlMSijJwo0L_^8`+x<bJ;KI!3shK?i08n=TAARm76=~#T^X9UL>H3EqWQ{^A)U&6ueX_bZ7XP=CXZ{6WA38!2xF(iVy0fx4%=jct9CG-%oGQB{$0gk}L2Rok&`yu|Ff#$TL6a&cV+ZQfD`%-c|lbz!?ImP2%|3W9MqSt=$!bb8Gv3f|hzFHG9IIP(ZMB}0NQP_J5a|nHc<=O4rQD2muwQD>~@~AJ_l%3jVg^XFZ9b$yBbcezN@4yWdi|rY|&ws0;THt@u!(t;FtZnGFi-P``UHwaZxBGp#Ddtih;=nb1Zc+uL0lJNZLyq3sJ}yDjqdsTOo!>h1TNnQRAjdA>2&J7L6fEfPFS!?$Ue<VO<O9rJZ#kOrUObxa8rKm@QF;=3f#he?SF?tCFe8X`CkaOc{#||~(*eX_>d&V8>}(a}!M?0X@RLyN1aRjKg|~@`lQOq;5*TLiZt6?6$p@}j@vjnx6_Ew$zx|c`j^xeeK}QFwU*vT~SB$z<1)VGOsZx%_N05?5p-&kHMJ0qhG8t4^mSr?>NQqW5XfEjSxKA?da|pk$Pt~RbAcQYO2CGzTkLfBN+X3c>JE`q`<+tX9l^@!C%|uq91!k5BuSq4bwy8OkhpsbYvS568w7hr6y(SpsnR2A>UVp}qjaYRjgDAo4W?L8uumLt>{AP$Uul<g$<B+oD105o{DjC^2AmzAC67pU=KgPURDD>sjS2OvOs>c|qtK=%}AeCokAVmmIB9|-ptmvBo=s?`ML5^~xvaBs2Q#)eTp=l41y=Tj;EyXdY%oi1pTpDI_Of0g5nyp=xY<0P*=BcVvgY2q=s-R9(6%MZ4xI;whqqLwgS1|J8rkoWhpHacKR`4z77Vo<-xw*9q6=Dd<<tdB$&~iHGlE=WE0%Y;E1Y8TC_QD(L6}fcD0NBrxWbWSwWKmg@I1qo?fY@sw@Kj>JCM_;7^efWZFqKW5OgO39<3TS%aFxQ-YGFg{nkZnGCGNc#5rard0zy+GplW^?s>NgiHe*gz($m5unwe#qPT+Kh@<S}uGrY-m9vKup;%=tRSSH~@Qf>~ph;Sw>9n&=#na_BT$HqYh8ir+pdPp2)-1_*HB}k0H(4YprwlW%tT6?8Yd2PH)y_c#Klod=TdO&R21HLpf93#NRU6v){8pASu#%%WB>2TUbct#qPWT4N7(gqKdcp>%OX^4iBb7p2*ESrBlF#4ubnHH0~E6`L)0$uT;zvA)+5!w_nO)ZjXDV~_JU_qp^hr;bn1+*XTsZp;(M+e&42OYuGueumI&!#3ZmpR@TrVb$HOf&bI_#L*Q>4~z9U|}&11Mj8PgkQtn#tPgq8V8uQzmvg??dv<3WmM_6RMPHu=4pLqttug~Z?VER>V5^f-bipMkFF&&3ca9WIr6)O(n3=}rgO`LYN=oJOll1^xPM}l=4B!D3O%cb+!j=egYZfdj<g?PyB(nCaGZlY&k#c8F^~m9t_GyZZz#8SUc!r8E{`bE8?V|FN(oAC&>5sFh#gLbRW)^)U#Ee2=M>bEXEm$~)O#$@Ys$bF2S@O=zK3XS2oCA2Tv$;fsPvGdpP2WhRVOl%wZo%WjF|$f?*$=S*DVR({9sQppowxZIV+dP>k4pOp#NJUUjq7}_td@zG(l=wCmCOjlcB%XLq`TkHLOfI0>adC+}oQe0qc0}4}`nfY7+NPkv|hlPlQJkH-_v3_JOt+m{`79x~d3Q?~p|Y?iJLUT{;wN5gUo^m<Y}|l+&@7WBPvENj3;4I_x!pjH^v#zs}X1hWfVC0ubvW3fiUghtf<i^FX?uwx4B|NjUc3kU^KB@UnGOC|+dtQIF+KlY-SjXpEsk9^yna?DQJBdWoTaC#QoVt2Zg~BB{M=To!udw~xqmfV>G({X9~DVG%?$n8*RQmHz%5nu;pNs=R6MT-BzgG!sIqd>CqV5`1KCV`h0GtIni5v7Qz&ICBrBwevd24f~;BgTKt(oz`4vwy~ro-)N*RpR%r=qjf;yNdtS7K;d8j!<Y`d2{3LUoplIt#ab)`%RuSK@qj2Q4HN$$4A3Z`pz4{W<akgUp#@-Uabmj}Y?-1Bu|uS|S=AdU`#OS!Z9rWV?%=2dSC^&kVXJCKx=eKswEH+Ld99P`f{JRF_w>+NXHpSF>paJB+P#6zos8%id+&K7J8Z_Ryc;u{iQ4FWK5rr!hgSZJGW==K(539_zHYf7)5G0C(9m#X^J}G?TORGn)Hx^bXUaf}yP*_oYVCp#y;u2yZ^D6lao86LqUcuoLKAV*126_~i$B7H*ZN!eRQk2n0@yxU0N|0ijm(YFmw?HJ<gRE8!;=<j*qMe^Nv{nXGHh)a3rSO}W~9GdgsI_xG7*OqbyN$aEI)m%ZlNQS_p0!BHOpPtwwA&K0dF-yr)n{AkUjw^Yya+Zl0Qf~k@7DkYs?HuJT;{#q=T0iaf;2xaFviwYOS8A%UewnD#@oR2zxH(8E6UHD2VM`?SRq|#9RAr4cY0jkpkZ3aI#(hMKU-V&M8?PIPJ;kLu&2rgU){OLIsT}jCWkCFqOIhIdI(@E#}?o*SSy+;_c)p6iuK6w!6~j(g^ps{1prZ8dtD(hF`mr!=eMd3UG)1B_QaKkwKp7YBZF8xHKv#4|)y`ltSj^Eb#C1xzS~8|B>nO_zykX{uP%Ri)%`(J(XLX?Tl}PQiq=ky5`)8MH?qKHU|P?6u9Fl#r)k!BRgZ!gRb1MJ_UR65r?k);d_T%eF=sLA1bKuwk-*`{mz28z2r;AAjOMAqmb@HtyLDO-4LH9Bhg-9n2Mb7;!dsRIdIiB^L!BX)2qFWTU+Gb4NT@`wOLv{%wlEN9Wbv!rv~Tl{F!>^1kiOUjM|4G+$RCd2tTNMNKe!qn!;d!;<GiDkJMCKO-oU=Uf=}4Jb)N4)|4WR+?Fw4t)v~L0<0dWYtP%+)hm#m8@ePN<!CUWaWu_ZD@k&c7xiA%)B681xcXS8Vj$jWJNrtA)S2qSsl}kp_%Ijo(lShqSKR)tpLuE(txFht+3>BdhaY{*dcfgF-P<hjKe4ega3vUi3r6?mZdITFH&N$#tzGoxxan#NM>cRG5?&Yx5BC>yI8&sx`~WIzFoEl@;SfNHs5dTiNMSz?bRQblwu_5giDj(z%lXLN!_%o6$F?rcsB^Ry6S^}f+bYZ61WIE!N8$`m#9mLy8#0eNmwxQXKG`V~I7%ToGa3tc(y<^|JMr#x08~+otP&zWtF9P<)oY7onu@*%<@xTJ$13kNyP74r@0-%1{AzP`Yl)iztx`j08<k1&RHTm_-C(Vbs;&cAO42?s&$=mvvj{Hep*xk?iQe7NUr;bh8{%9rCgfT-y`Xm7<_2juzuA|<^#+0{A>TgpkJ}@U?+;=x&}kQ&>Pciq-Yf2#rZr4P?Q)KfA5Ptgo%FjoaK{UBbe*O1Zz<RptF=gmCOF}Ae*x6{j4G~5rjgx<W>F*VxtGBA)O!3ZUO?CuzvqPUY30dK$UL;>%O~*E023VHZol_q%U>?&_hG~Wr`tV;*eNgf+3U>N_nYg#uE~Co+3vPD+SLj*M%HO{CBIXQpo${Zh9RjSW~UFm^Got37HGoGh>O)2>A}McY91_AW-0JB?BwTTqKdq?P&olVHv*#`=jlZ<zqQ6j!N_-)WqFSJHFChL(zRr`&MZOmnOFsi7GjcEAX&+{6U2_v$f`|MS2BUSX$FM>Py-Wy;;%B&7h{Da@@reFJ*Q-MF1u^7d?Yl;HkEDAy+DstT)OpN8X)9<8qvR+QmgY)p}aNXw<WkZ9kg}Ra&q1sZ2t6%F0R{=_FU50Ep->ylcGT4yM1huQl55RZI=@R+Ks~ez1Bom0Wx7Qz5>LgCxa0<#w%ob=YFmh)EeFJYe6+!dUY)>>B66DQTCKuDXD>e&}wt_wd`JzZUIf{z7uzQq1*Q`SGQV*s+QC``%HA8-HvZTd$gCYI?q<PEAbXpdt=j{m{!M*>u{%iS@OEI>a%hcOTEfBCg%i+9dq=nZk=T3cG{7n1>a6<8iLXZai9S>=+&bHzosz1<kxrG(7wWtV{fQ6SayPS;a)iHO;C0%tz~7@<QFnB8e=yb>(qM*JyG5lri^luzL0hRGP~|^Bbp&hlhu!U2pWK&r>3(>-~GYQRjITa4}Bhs;wNh4U8<B~PwMLW1C5&xsT%v`*&kVig20u$O2!NE-bu&>F6Q}$$TblMR%WR`QnwE*N{=tpC@Ae6v)ym9xe&lYPU`GSL=krGleoaA-^nj;N$UO*jQj~0Vf0a)7s<58b89$^gIyOgbsWa+Z_S7%n$Ovoeu!!&B-^iMi^QY5Xh-~qk6_Y>djLj;yl$iQVlZc(h>nbjo<%Vgf)QSH9ODOi#7CWA=dOfz^QTaR`9X2FMBWHR38W&%+y=gJQzQ5V5sZ+$lfNXIsPq_tBo&0DfWQ2mr5o)x?)IcOs9cfv%BK;%oY^$uL?}Ghot!J;$OF5)N^?pEJceZ2!hWU8$R~H(Dg6emJLAPjz}B)Z#=A0hhK`JC<#gZ0Y1oaX1FLF>j8ihEyLP!9yfZUB4x6g=z?ryF0&VE&9v!2V;p*9Gh3d)H`kpvSthMClZc7zz*Nj0*+mXKobTyJcLE25b&GeL8y*Q%%s^zG{Lo0LUP&2y~7-m+h`V}Bqr_oLKRqIopxuJ9e$gB7af1Z8Up8R<z$OFn-FBM`4(`fkD#^4b1ub2LEgh(K#1M?a-76W_*n@T{#d&sd5LO^%0Ia2zG-&AZs@o$OUP%yS~W)V{`)~E=W?w~?7p_$kGp3~`_51OrY<-X@kvg1Lsi1g>)NjfRhALvP7Q~lbz*=7kk5+*&D(Odd4y%(?0cqeDOeycv|6xa}cYCd`n4DyfkAm9OpOoT`f^_c0=?#jL!zEvl1zr-a_VOJd<l2ina9-MF+0Z8cRAxMxFkOB}WUlNyU9M#!{<Ro%Xsv)){Hf~KTYq`o;Q-#El!(8uqndhR2N8XkF3{}3gqBf9kf!6}=s$SNAuCHcqzOwr<U7);h6wFa+G`^zB=Yxx$JJVnp#3|7aUq4ugZZtExl+;g&ALM}(g%>dDDiK#}0^aU=!`Ol|7$Ix~sppV<pdAvE=(Q801-W+&H9lx4P5(z79Dj7@XN>p$(f&4BQ?8*~0ihq~{uOY0$}J;}Mi6J;GR$%Gf#66f%8ivjh3>7@X-eFzspqueMz=?28xH;#nM~J{',
 'pubmed_resume_snapshot.py': 'c-qBU>u=n+5&yn_g`E2!>7r<J0a_Frs6cw>pe~ZsaGb*xTSJf~u9s)MKvIsKApd)3h7Xau>vgU`(FKxSQA2Vbzj={<_UzfG@kUe)i-x{^h}4!v_0Wx#;@2&UPTw+BZzCnDUXFuMSu_eJnYiQqO~mfGRQTh!_wS;rZ=$Mgqoz_-RBu^*I}W^8GJp2$*@AaCIjW?xKX1P9(f;06a@+E2`>4NQAfFU(?db1v=ocF?bP@1|<c2Ofhet~!xBe8xJEo^hqQ&*bN=Rns!I&Idt`>{)Hy0mIFUq%{KKy)n_RkL=qasS;`>rh&yH{}*#qcGnT9ty;gJ^JsRw~&BwG*d{#iC&w*u6?n{B#uwrY6x_$#Qt}>Q8^xFY_(CZ}<(9DoJUYU{#Z#Q1d(#HgAWjkqL^Y-{we2JSD8JhXyoX#FN^*_)DDXl`1uKye8@_x*nQ`bhU`!g99Mcyp$V7m9Spr-EG4~q90^&DJGVYb<5$Fp7adGx)Ii)iJ!UJmU7x`_&u$e>)+4gTy<kS)90TBSFF5#P)sJo)BH-Xv8`%mctQz?*a&b}wXpFjkSZa~G*dUqgx+VYmy=*+C2P)$w^b`y#`^}uC|;${<A3*Ynv=Fzl3F{F@kCh<J()G}G^GS%ARz!%gN5GZ4Wk8SWh3ck>TAV^UIP8sKJTlJt!N^0m-N6z-A)Z~rj+2<6Ns<bOQX#oqCVn1?Uv&X5#K~y@?NT{2YvJzuJhJx@C)0d-)Kvq>Ia2xmf&`{#LNY)YDRYvjmWAvVE<Rt*{KkNNaFWgO7vdi<LJa%Gm+IyZtQ4K`ITgJJFfBZ>gc}TjK&elWi@gGb6Ql6tOnJE-k#%Vi9TCyc+1Lms4Ltoi6QJvopZTthH*DEQ_J>O(&Kt9Ts6V}Qnxj^7ZW+VhYdAtqQ3%}^^g$*5XZrXS&5FE3=@Py5X#5{aqa>f^aX{{J0|#sYu>X+LjiSNR}dPTNCy%L(Y69}3PxrHZxBPxqy&Oy5=gFvrVxr-z>e09;inP5>iGtO3A~^{0xy=a9n8^Pj1dWE5ZQ=8R^8Y#dfmn*vS=4gEz&TWk!PF~cDc2gSL%2cUmb$g1@RndN{WC{$7NT6*({#)J9M127wKBw8|T1Z2&Y5Nms8MA$g`16q|GI`*qEeOTF{4*H|Ugd9rFg5)V?8cMYFSWr<U~z(Swxu+q_RB<idLnbii6xE7p9E_-DV!Wj##2+S9?<@rFVNASlNNja0}=3GfYH@m0FVH?nN=)=ZXf2dU5niy|JU>y9;fUtjaQZ@awb+x%v@i?zg}`dktO3Am+&A#m|6gMUPWKxcOxLMTKcO<@r3r4mAsez}fuRUAiF0I@@R#|(_TphY=?OW3mouef8S8WN2~m>zo*(<531Qq2KOl)z#nrrs*l>_x9}L(rNNcjg~In7XI{$%heaOo=5H*MO%=MgS#Nb;!f#LjNZ3NYG60-~3iyoPIoq$aY%(?O&ItAD{n}21^@A!Q}L>hi4p-4kIwTuNecANdAQZwmYtT`S3%v70?&xpt#X41D4>V@D21GE{obd=&$zU^tnczT_FzaW*NcQtBR1dj6j~QUtZ1YxCv1+)UYem8yLF6l^6eWaZGr3JUHmP7~fUn2r+oy1+*^~6gKi3rVIuB#DJ3qcr<~4%jm-ZQ4JlMMvU`Xsq}0dI)?wbu7=36{B-f|plq;fOr<ON04-1DIRVD#kltU-rBTQSzO^b&S34MZgg2b?q`tN6x_$yxkh21lIuPjCWup?rgqwD_PU0U;7F&6XAw@A4&bt8s5&{FjfdB3r=ZJdWoS*qB#tH<(>hkQDgEAQ)JcgJ#W`Y+K2g1CMgFpnIXV%|9hKs>Kv8jC!Lq5e;sj-C0QB4xbUcG#|OJNj?+am>jf2UDVBL+c$U@AVTJ!}~yKwCLtmC^=r%DGjsn@N0<!jTc1$kwB3Fd|^qvZ6w^>vRdLC0p$VLRj8%ZGCSS<VU^Dtp4KsJpjN8au5luowu`F!J~DvedmhPnS`sB?a|QS_qrZ3xWw#1K6r>hFWawe2*BFt%Ir#nb=40&6s>LX@(~aH|GB2)?T<L<lb-pAuNBVMb_@Wc0zpuC`t(wO*6ChCcGG-#RQ-dmfLn^dhR%$T0d7hA8Z;Tjip)=dSs1H#5OWkYPNmD8xOYgEj`(@;)A8_qjZ@h%j1a{m5TSS48qh;Yjx$dvc>a2N@ez8;C+a7fADOAxc)7DIUz&|e!_;Qr9NXJSs3~?0IH!6Wv&qhYVtPEnl4j>fu6FT}D8qJp?le|@FOtPB0>>+7mn?=_7JJN)_{B^RNM)_%ErruOT*E$i28LME-K?c?wOVT1`$?Z5>W;W)pG$1qGCW4u@S809F^j-W!32#yOLG14KL(H!T^eeTrKZLbydtNBO>qu@jEmQ>Dm?4sA=^DKfsv&_+fu?(7@SR)CjFkd2s>s~2{2{hV^({6Df?=a+X2pg9%9`g;z1Hu57gf18L;&@$x?Hxq3xf|7(rLNg%220r>#}oa7i?AaHtb(mDAxHG~+AZH+EyNCDLcF5KoQ==_j+v&sC3{7;AzVT6R;_57=Z;&xFSh`{KWjCdsVD9)hH!5AL+SGmDG#h-#1qb~Bp!UBkmuzA=~Kurvgo70ieZF8+KvEAR(q?J+o%v(4{Jq+^g*YXDz8xtDJeL%b%=ZBKGSt(lYeP<S}&6%%3_v5K<#)_Rgr!#9)%l&!DOJ%ak)K&5cI!=d~gAHXj5ReTlZIM7A_u++B5Za51>nb&y|HHSpe@R$Uhe?e-g`Wt-Pv-^<(S{vYGXdBX0LHKoi;4q5roh|H~Qh6s;T6qD<EMu^wwNiG_cK?;Q#!fbT92FSRw%Rx40Tb=(=YWvAda$&Roa|~r0c|6l?d=Xo__4a1YkMct!Ojkkt<5fYE~fA{2oh+p<K$hG*m4}xfEOb7Sh%$JO-{vhsw$@u2jzV8WIK?lmzF1c9-ZB?2eke=iEGFdEu8rK@jaVIjV`jp26uGzJ#sr}srL#k`heJi4}y#+hHJL7h#50$T&;G^M;?IOhiLb#xdyQgo_c)hE$o0(y!v{?cB?3UPCG1IRUTf4co5osHOI6g#`7-{G)Fo`R9G6gM1fpprlWM%i~8`~*xVZQhv*%g57W&gE`TAyrK1fOl1|Ry9trV)M3V69*36?bKn&G^y^cn%(A~0j1TF!TP<lfb3rF+0?SvZY9CX^TiY&JPyFom}%Z+IeN=I3K4TAnRS3{UA=m>Wb;SY0{Vk#gy4fojeAr#r;(?|^@tiGSn3dUz!s!`H5^1-;k34<0+kD6-c;4X(5H+O}PPRjJcA#fL_`z+iQkXui`(?U~)zoUpmf0-)-h1S@=fWiA0BJi_T(Qp8(%WO6<g~NWp%7W-vn2obTF*D5V!EMQ*98`q#RH_|_^+n<HzgImH<D`O2S0qJTPXetGXAbD4>3T?)mltn7z_IhiWoh~*%_;gel~bENP%n*`UxPHVhhv0|T_tm5M4>NlAN81^bky5;x4E;mP!tGq`Wo81z5S-^yv$OjLB;1(gvznCS~e&AA`mN^bAD)`ET}pPC+U8)->bP?h|U5m`VSez{~HK2(zR3tS=HB{U!>X^C0#sKPwTk>r~)<uWvy*pdsVnzGX5?W9>=2z+RYyTq;4KfwkoYR>MpWjk_*=OcdM$xD{~j%p{;#!K3%`ZSO-&V1NnOXJC?s+UY_IOByTyLzSte;#?|uc8bz|V>lB}C^ujjHDMqtxZaINX@0i06IIdCwpl9@l4EmN2X5?UjZnyOVvXdgL;bMv<rpPh-(vbO0g~CZei$7J=!lBd)*KAM{vb98T5sjq-tol5DETPYTFn}k*#wSszod9cD+>z+Yro_vwSG^cZ!I4>u%6xDMvsw0o=-}ZC)8zqp*EmS8cs2UYJ*M{dEQrCqVIS^#=Hfr{KxCf'}

EMBEDDED_SHA256 = {'ask120_snapshot.py': '4a6c6800bc8dc949ee49fa9de0b9e41d30062d676f5f51f6826db87a179913f3',
 'audit_snapshot.py': 'cb0ed8dcdf9f7209ea732219a14905f9cd034de29dec0696a71fee5290631002',
 'checkpoint_snapshot.py': '8ecff6c935274743509f8ca1b1a0a85dd612b58f02240c79e6ae9cab03e18ff6',
 'cnkang_snapshot.py': '79724e92d9758b1c89c70338d3e8157acd81167722d6871c9cc52bab8cb01b4d',
 'cnkang_transport_snapshot.py': '3366d872a9d999bc88b106c82def1b39239acbd90056359b7329e541bd371bd8',
 'corpus_runner_snapshot.py': '6e68c959e5bac7eadd2b852f56d07f27a4b4d10de4606f5d52b41d5c2367b60a',
 'crawler_snapshot.py': 'aa06bb6d678b563d475fc9a81f161bdeb26cc49744cedc031ea5c5cf57bfcd71',
 'diagnostics_snapshot.py': '0d45ea2439d7d11f42d7203b3628d18b05a54f8f86b746af235ed3e354af316c',
 'export_snapshot.py': 'e924cfa0a104928aa7d90826118955bbe49bbdca833994bb42746bf7e9c8ba30',
 'longchau_snapshot.py': '287b8cd009ff5e558f2a0e7c5592dd1b430a5e144da23619087c745f57287a3f',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '626e493ad05d4f6c96b257167c2cb81a815614122ce35272317ced8d5240f37d',
 'pubmed_efetch_snapshot.py': '983c7c3c1350c4e3a157a5b1b6b004a85f81074454ced5ffb7476f531f4cc3c7',
 'pubmed_resume_snapshot.py': '2c43d37a923335a61cc03d41632c340668ac8cc45afca15f24ebdcddc36220db'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'pubmed_resume_snapshot.py', 'checkpoint_snapshot.py', 'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}

HAS_CNKANG_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_CNKANG_SAMPLE = any(
            (urlsplit(json.loads(line)['url']).hostname or '').lower() == 'cnkang.com'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.cnkang.com')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_CNKANG_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_cnkang = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('cnkang_snapshot.py','cnkang_sha256'),
                      ('cnkang_transport_snapshot.py','cnkang_transport_sha256'),
                      ('crawler_snapshot.py','crawler_sha256'), ('corpus_runner_snapshot.py','runner_sha256')):
        if saved_cnkang.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run cnkang cũ cần RUN_TAG mới + RECOVERY_RUN_PATH; giữ nguyên raw/checkpoint cũ.')

HAS_ASK120_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            ask120_url = urlsplit(json.loads(line)['url'])
            ask120_id = ask120_url.path[len('/question/'):-len('.htm')]
            if ((ask120_url.hostname or '').lower() in ('120ask.com', 'www.120ask.com')
                    and ask120_url.path.startswith('/question/') and ask120_url.path.endswith('.htm')
                    and ask120_id.isdigit()):
                HAS_ASK120_SAMPLE = True
                break
if not POSTPROCESS_ONLY and HAS_ASK120_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_ask120_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('ask120_snapshot.py', 'ask120_sha256'),
                      ('crawler_snapshot.py', 'crawler_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256')):
        if saved_ask120_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run 120ask cũ cần RUN_TAG mới + RECOVERY_RUN_PATH; '
                             'không tự thay text/checkpoint đã trích bằng parser cũ.')

HAS_VOV_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_VOV_SAMPLE = any((urlsplit(json.loads(line)['url']).hostname or '').lower() == 'vov.vn'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.vov.vn')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_VOV_SAMPLE and VOV_USE_CURL and (RUN_DIR / 'corpus_config.json').exists():
    saved_vov_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name,key in (('priority_probe_snapshot.py','probe_sha256'),('corpus_runner_snapshot.py','runner_sha256')):
        if saved_vov_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run VOV cũ cần RUN_TAG mới + RECOVERY_RUN_PATH để chuyển sang curl/Chrome; giữ raw/records/cooldown.')

HAS_LONGCHAU_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_LONGCHAU_SAMPLE = any(urlsplit(json.loads(line)['url']).hostname in
            ('nhathuoclongchau.com.vn', 'www.nhathuoclongchau.com.vn') for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('longchau_snapshot.py', 'longchau_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256'), ('priority_probe_snapshot.py', 'probe_sha256')):
        if saved_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run Long Châu cũ cần recovery: dùng ZIP đầy đủ, RECOVERY_RUN_PATH + RUN_TAG mới; '
                             'không ép RESUME vì parser/runner đã đổi. Raw và records cũ được giữ nguyên.')

for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (name == 'ask120_snapshot.py' and POSTPROCESS_ONLY and HAS_ASK120_SAMPLE) or (name == 'longchau_snapshot.py' and POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE) or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
resume_spec = importlib.util.spec_from_file_location('pubmed_resume_upgrade', RUN_DIR / 'pubmed_resume_snapshot.py')
PUBMED_RESUME = importlib.util.module_from_spec(resume_spec)
resume_spec.loader.exec_module(PUBMED_RESUME)
if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and PUBMED_UPGRADE_EFETCH_SNAPSHOT:
    upgrade_status = PUBMED_RESUME.upgrade(RUN_DIR,
        zlib.decompress(base64.b85decode(EMBEDDED_FILES['pubmed_efetch_snapshot.py'])),
        {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}})
    print('EFetch snapshot:', json.dumps(upgrade_status, ensure_ascii=False), flush=True)

CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các website ngoài PubMed, Long Châu và VOV dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới đã lên lịch, gồm cả PubMed; retry không tính hai lần.
HTTP 400 chỉ retry khi body báo Timeout/Empty Response; HTTP 400 invalid UID/parameter không retry mù quáng.
5 attempts, read timeout 90 giây, budget 300 giây/request batch, backoff + jitter và Retry-After.
Batch còn lỗi chia 100 → 50 → 25 → … để cô lập, giới hạn tổng request/time; sau đó tiếp tục batch khác.
Cuối lượt thử pending thêm một vòng. Cooldown server dài hoặc 3 batch liên tiếp không lấy được bản ghi
thì hoãn để resume; không ném RuntimeError vì riêng một batch lỗi, không đánh dấu complete khi còn pending.
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_pending.csv`, `pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.
Resume EFetch cũ tự nâng cấp transport tương thích, giữ nguyên XML/record/schema và các snapshot crawl khác.
Checkpoint full ZIP/HF chạy định kỳ trong lúc crawl, khác với upload corpus cuối cùng sau audit/export.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

Long Châu mặc định dùng `curl_cffi`/Chrome, 1 worker, delay 5 giây; không chạy Selenium cho domain này. Parser giữ sapo/nội dung chính; `/benh/` lấy đầy đủ các trường HTML y khoa từ JSON đã xác minh slug/canonical. Xem `longchau_results.csv`, `longchau_summary.json`; parser v2 loại nhóm liên kết “Xem thêm”. Run mới có hậu tố `_longchau_v2`; dùng recovery từ ZIP đầy đủ để tái trích HTML 200 cũ, giữ cooldown. Checkpoint thuộc runner chung: lưu theo cadence cho mọi engine và khi dừng/ngắt; log có tiến độ chung theo ID.

VOV (`vov.vn`, `vov2.vov.vn` và subdomain) mặc định dùng `curl_cffi` với `impersonate="chrome"`, một request VOV tại một thời điểm. Giữ parser hiện có, timeout/budget/robots, raw và ID/URL. Không chạy Selenium cho URL VOV khi `VOV_USE_CURL=True`; 429 dừng nhóm VOV và lưu `vov_request_gate.json` qua resume/recovery. Run VOV cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH.

120ask V2 chỉ định tuyến `/question/<id>.htm`; `text` giữ câu hỏi và từng lượt theo thread. `ask120_threads`, `ask120_text_raw` và `ask120_qc` nằm trong record checkpoint; Parquet corpus không thêm cột. Run parser cũ phải dùng RUN_TAG mới + RECOVERY_RUN_PATH để tái trích raw, không resume ghi đè.

Cnkang dùng chung parser và Requests transport với check_cnkang.com.ipynb: một worker, gap tối thiểu 1,5 giây, HTTPS trước, robots từng origin, retry/cooldown hữu hạn; không chạy Selenium. Giữ encoding/body candidates, bảng, text trước/sau và parser_flags/source_flags riêng. Mã v6 gốc đã lưu trong docs/references/cnkang; chưa có raw 120 trang để chứng minh tương đương baseline cũ. Run cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH để tái trích raw, không ép resume.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'requests_trust_env': REQUESTS_TRUST_ENV, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'longchau_use_curl': LONGCHAU_USE_CURL,
            'cnkang': {'gap': CNKANG_REQUEST_GAP, 'attempts': CNKANG_ATTEMPTS,
                       'timeout': [CNKANG_CONNECT_TIMEOUT, CNKANG_READ_TIMEOUT],
                       'user_agent': CNKANG_USER_AGENT, 'trust_env': REQUESTS_TRUST_ENV},
            'vov_use_curl': VOV_USE_CURL,
            'longchau': {'workers': LONGCHAU_WORKERS, 'delay': LONGCHAU_DELAY, 'attempts': LONGCHAU_ATTEMPTS,
                          'timeout': [LONGCHAU_CONNECT_TIMEOUT, LONGCHAU_READ_TIMEOUT],
                          'budget': LONGCHAU_REQUEST_BUDGET, 'impersonate': 'chrome',
                          'retry_failed_on_resume': LONGCHAU_RETRY_FAILED_ON_RESUME},
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        PUBMED_CONFIG_CURRENT = {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}}
        corpus_settings['pubmed_efetch'] = PUBMED_CONFIG_CURRENT
        if PUBMED_UPGRADE_EFETCH_SNAPSHOT:
            corpus_settings = PUBMED_RESUME.runtime_settings(RUN_DIR, corpus_settings)
        if CHECKPOINT_ENABLED and CHECKPOINT_UPLOAD_ENABLED and not os.environ.get('HF_TOKEN'):
            try:
                from kaggle_secrets import UserSecretsClient
                os.environ['HF_TOKEN'] = UserSecretsClient().get_secret(CHECKPOINT_HF_SECRET_NAME)
            except Exception:
                print('Chưa có HF_TOKEN: checkpoint local vẫn lưu; backup HF chưa khả dụng.')
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
